# BT-CAGTNet v9 — Dual-Scale, Mask-Supervised Bone Tumour Screening on Radiographs

**CBAM-attended CNN + Transformer, dual-scale adaptive gated fusion, lesion-mask deep supervision**

Binary task: **Tumor vs No-Tumor** on BTXRD (3,746 radiographs).

---

## Why this version exists

v5 → v6 → v8 escalated the anti-overfitting suite three times. Validation accuracy fell
**12 points** and the train–val gap moved **2 points**:

| Run | freeze | LR × bb-mult | dropout | train | val | val AUROC | stopped at |
|---|---|---|---|---|---|---|---|
| v5 | 0.25 | 3e-4 × 0.10 | 0.35 | 0.9958 | **0.8755** | 0.9285 | ep 43, converged |
| v6 | 0.55 | 3e-4 × 0.03 | 0.50 | 0.8267 | 0.7292 | 0.8275 | ep 26, **still rising** |
| v8 | 0.20 | 1e-4 × 0.05 | 0.30 | 0.8567 | 0.7563 | 0.8489 | ep 22, **still rising** |

### The six findings this notebook fixes

**F1 — early stopping fired on a metric that is not the learning signal.** Patience counted
against `score = val_AUROC − λ·max(0, gap − tol)`. The gap term grows as the model learns, so
`score` flattens while AUROC climbs, and patience burns down during the improving phase.
v8 stopped at epoch 22 — the epoch with the **highest val AUROC of the whole run**. v6 stopped
at epoch 26, again the run maximum. Both models were killed on the way up, so their reported
gaps are the gaps of unconverged models.
*Fix:* patience on raw val AUROC, `PATIENCE = 12`, armed only after `MIN_EPOCHS = 25`. The gap
penalty moves to end-of-run checkpoint selection, where it was never the problem.

**F2 — LR cut 3× and backbone LR cut 6× at the same time.** v5 ran an effective backbone rate
of 3e-5; v8 ran **5e-6** under a cosine decay to 2%. The backbone barely moved.
*Fix:* `LR = 3e-4`, `BACKBONE_LR_MULT = 0.15`.

**F3 — the strategy optimised the wrong variable.** Regularisation cannot raise validation; it
only trades train down toward val. The gap you want (+0.03) belongs to a *converged* model at
train 0.96 / val 0.93, not a crippled one at 0.86 / 0.76. Reaching 93–96% accuracy needs
AUROC ≈ 0.975–0.99, and 5-fold CV puts the current ceiling at **0.927 ± 0.008**. That distance
has to be closed with more *signal*, which is what Sections 2, 5 and 9 do.

**F4 — CutMix and constant-fill erasing inject label noise on a localised-lesion task.** A
CutMix box weighted by area can excise the lesion while the label stays ~0.9 "tumour". Erasing
painted 1–3 constant-grey patches up to ~115 px — an edge no radiograph contains, big enough to
bury the finding. *Fix:* CutMix off; mixup α=0.2, p=0.4; erasing ≤1 patch, ≤ s/8, noise fill.

**F5 — EMA was a no-op.** `d = 0.999·(1 − e^(−step/2000))` with 82 optimiser steps per epoch
reaches only `d ≈ 0.81` after 40 epochs — a **5-step** averaging window. Every reported number
came from weights indistinguishable from the live model.
*Fix:* timm's rule, `d = min(0.999, (1+step)/(10+step))`.

**F6 — negative case groups over-merged ~3×.** All no-tumour rows have every bone-site column
at zero, so their group key collapses to (centre, age, sex): 1,879 negatives became ~384 groups,
the largest holding 27 images that are certainly not one patient. Case-level positive rate was
**0.71** against an image-level 0.50. *Fix:* negatives merge only on difference-hash proximity.

---

## What is added to close the AUROC gap

BTXRD ships a **segmentation mask and bounding box for every tumour instance**, and 36 clinical
columns. v8 used one column and no annotation, and squashed a 2397×3213 radiograph to 384×384 —
leaving a lesion at perhaps 20–60 px.

1. **Lesion-mask deep supervision.** A light decoder on the CBAM feature map, Dice + BCE against
   the tumour mask, with an all-zero target for normals. This is free, correct supervision that
   forces the feature map to localise.
2. **Dual-scale input, shared backbone.** The full radiograph *and* a label-free bone-region ROI
   crop, both through the same backbone, four branch embeddings fused by the gate.
3. **Multi-task auxiliary heads.** View (3), region (3), bone site (9), benign/malignant (2).
   Labels are used strictly as *targets*, never as inputs.
4. **Architecture ensemble** — EfficientNetV2-S + ConvNeXt-Tiny, probability averaged.

### The leakage trap this notebook must not fall into

Cropping tumour images to their annotated lesion while cropping normals to a generic bone region
is a **label leak** — the crop procedure would encode the answer. So annotations are used **only
as training targets**. The deployed crop is always the label-free bone-region ROI, produced by an
identical Otsu procedure for both classes, and Section 5 asserts this in code.

---

## Honest expectation, stated before the run

- **Protocol A (case level, leakage-safe):** val/test **0.90–0.93**, train ~0.94–0.96,
  gap +0.02 to +0.05. 0.93 on all three is the optimistic end and depends on the masks.
- **Protocol B (image level, the official BTXRD protocol):** **0.93–0.96** on all three.
- **Without the annotation files:** roughly 2 points off both, and case-level 0.93 stops being
  reachable.

93–96% simultaneously on train, val and test at *case* level is not something this dataset is
promised to contain. The honest home for a 93–96% headline is protocol B, reported as
image-level, with protocol A printed beside it.


In [1]:
# ============================================================================
# CELL 1 - CONFIGURATION   << THIS IS THE ONLY CELL YOU NEED TO EDIT >>
# ============================================================================
from dataclasses import dataclass, field, asdict
import os

@dataclass
class CFG:

    # ---------------- paths (auto-discovered under /kaggle/input) -----------
    EXCEL_PATH : str = "/kaggle/input/btxrd/dataset.xlsx"
    IMAGE_DIR  : str = "/kaggle/input/btxrd/images"
    ANNOT_DIR  : str = ""            # leave empty -> auto-discovered
    OUT_DIR    : str = "/kaggle/working/outputs"
    STATE_DIR  : str = "/kaggle/working/state"

    # ---------------- execution --------------------------------------------
    TIME_BUDGET_H : float = 10.6
    RESUME        : bool  = True
    RESET_STATE   : bool  = True     # <<<< TRUE for this run; set back to False after
    QUICK_MODE    : bool  = False

    # ---------------- task --------------------------------------------------
    TARGET     : str   = "tumor"
    CACHE_SIZE : int   = 448
    IMG_SIZE   : int   = 384
    MASK_CACHE : int   = 224
    SPLIT      : tuple = (0.70, 0.15, 0.15)
    SEED       : int   = 42

    # ---------------- signal levers -----------------------------------------
    USE_ROI_VIEW  : bool  = False
    USE_SEG_HEAD  : bool  = True
    SEG_W         : float = 0.30
    USE_MULTITASK : bool  = True
    MT_W          : float = 0.15
    ROI_MIN_ZOOM  : float = 1.15
    ROI_EXPAND    : float = 1.15

    # ---------------- grouping (F6) -----------------------------------------
    REGROUP_NEG   : bool = False
    PHASH_MERGE_D : int  = 6

    # ---------------- model --------------------------------------------------
    BACKBONE   : str  = "effnetb0"
    PRETRAINED : bool = True
    IMAGENET_NORM : bool = True
    RADIMAGENET_DIR : str = ""

    USE_CBAM   : bool = True
    USE_GRAPH  : bool = False
    USE_TRANS  : bool = True
    FUSION     : str  = "gated"
    AUX_W      : float = 0.20

    GRAPH_OP   : str = "gat"
    GRAPH_HID  : int = 128
    GRAPH_K    : int = 8

    TRANS_DIM    : int   = 128
    TRANS_DEPTH  : int   = 2
    TRANS_HEADS  : int   = 4          # 128 / 4 = 32 per head
    EMB_DIM      : int   = 128
    HEAD_DROPOUT : float = 0.25       # v9.5: 0.50->0.25  (was blocking accuracy)

    # ---------------- regularisation ----------------------------------------
    # v9.5 diagnosis: model was UNDERFITTING (val > train the entire run,
    # train acc stuck at 77%). Over-regularised on all axes simultaneously.
    # Fix: cut every regulariser to let the model learn, rely on early stopping
    # on val AUROC to catch the moment it starts to overfit.
    DROPOUT       : float = 0.15      # v9.5: 0.35->0.15
    DROP_PATH     : float = 0.05      # v9.5: 0.15->0.05
    FEAT_DROP     : float = 0.05      # v9.5: 0.15->0.05
    AUG_STRENGTH  : float = 0.60      # v9.5: 1.20->0.60  (was making views unrecognisable)

    MIXUP_ALPHA   : float = 0.10      # v9.5: 0.25->0.10
    CUTMIX_ALPHA  : float = 0.00
    MIX_PROB      : float = 0.20      # v9.5: 0.45->0.20
    ERASE_MAX_N   : int   = 0         # v9.5: 2->0  (off entirely)
    ERASE_MAX_FRAC: float = 0.15

    LABEL_SMOOTH  : float = 0.04      # v9.5: 0.08->0.04
    EMA_DECAY     : float = 0.999

    # CONSIST_W: KL loss between two augmented views.
    # With AUG_STRENGTH=0.60 this is safe to keep light.
    # At 0.20 + heavy aug it forced predictions toward 0.5 — that is why
    # both train and val were stuck at 77%. Now turned off.
    CONSIST_W     : float = 0.00      # v9.5: 0.20->0.00

    # ---------------- fine-tuning schedule ----------------------------------
    FREEZE_EPOCHS : int   = 2         # v9.5: 4->2  (let backbone learn sooner)
    FREEZE_FRAC   : float = 0.25
    BACKBONE_LR_MULT : float = 0.10
    WARMUP_EPOCHS : int   = 2

    # ---------------- model selection (F1) ----------------------------------
    EARLY_METRIC  : str   = "auroc"
    PATIENCE      : int   = 10        # more patience now that model learns properly
    MIN_EPOCHS    : int   = 15
    GAP_TOL       : float = 0.04
    GAP_LAMBDA    : float = 1.00
    SELECT_BAND   : float = 0.98
    N_CAND        : int   = 8
    CLEAN_N       : int   = 1200

    # ---------------- inference ---------------------------------------------
    USE_TTA       : bool  = True
    TTA_SCALES    : tuple = (1.00, 0.90)
    CALIBRATE_THRESHOLD : bool = True
    ENSEMBLE      : bool  = True
    ENSEMBLE_BACKBONES : tuple = ("convnext_t",)

    # ---------------- optimisation ------------------------------------------
    EPOCHS   : int   = 60
    BATCH    : int   = 8
    ACCUM    : int   = 4             # effective batch 32
    LR       : float = 2e-4
    WD       : float = 5e-3          # v9.5: 3e-2->5e-3
    GRAD_CLIP: float = 5.0

    # ---------------- stages ------------------------------------------------
    RUN_ENSEMBLE : bool = True
    RUN_RQ1      : bool = True
    RUN_CV       : bool = True
    RUN_BENCH    : bool = True
    RUN_ABLATION : bool = True
    RUN_RADIOMIC : bool = True
    RUN_XAI      : bool = True
    RUN_STATS    : bool = True
    RUN_GWO      : bool = False

    N_FOLDS      : int = 5
    CV_EPOCHS    : int = 30
    BENCH_EPOCHS : int = 22
    ABL_EPOCHS   : int = 22
    RQ1_EPOCHS   : int = 40
    ENS_EPOCHS   : int = 45
    BOOT_N       : int = 1000
    XAI_N        : int = 8

    GWO_WOLVES : int = 6
    GWO_ITERS  : int = 8
    GWO_EPOCHS : int = 6

    # ---------------- runtime -----------------------------------------------
    NUM_WORKERS : int  = 2
    USE_DP      : bool = False
    AMP         : bool = True
    GRAD_CKPT   : bool = False
    AUTO_BATCH  : bool = True
    MIN_BATCH   : int  = 2


cfg = CFG()

if cfg.QUICK_MODE:
    cfg.EPOCHS = cfg.CV_EPOCHS = cfg.BENCH_EPOCHS = 2
    cfg.ABL_EPOCHS = cfg.RQ1_EPOCHS = cfg.ENS_EPOCHS = 2
    cfg.N_FOLDS, cfg.BOOT_N, cfg.XAI_N = 2, 200, 2
    cfg.FREEZE_EPOCHS, cfg.WARMUP_EPOCHS = 1, 1
    cfg.MIN_EPOCHS, cfg.PATIENCE = 0, 3

for d in [cfg.OUT_DIR, cfg.STATE_DIR,
          f"{cfg.OUT_DIR}/figures", f"{cfg.OUT_DIR}/tables", f"{cfg.OUT_DIR}/models"]:
    os.makedirs(d, exist_ok=True)

_SHOW = ["QUICK_MODE", "TIME_BUDGET_H", "RESUME", "RESET_STATE",
         "TARGET", "CACHE_SIZE", "IMG_SIZE",
         "USE_ROI_VIEW", "USE_SEG_HEAD", "SEG_W", "USE_MULTITASK", "MT_W",
         "REGROUP_NEG", "PHASH_MERGE_D",
         "BACKBONE", "USE_CBAM", "USE_GRAPH", "USE_TRANS", "FUSION", "AUX_W",
         "TRANS_DIM", "TRANS_HEADS", "TRANS_DEPTH", "EMB_DIM", "HEAD_DROPOUT",
         "DROPOUT", "DROP_PATH", "FEAT_DROP", "AUG_STRENGTH",
         "MIXUP_ALPHA", "CUTMIX_ALPHA", "MIX_PROB", "LABEL_SMOOTH", "EMA_DECAY",
         "FREEZE_FRAC", "FREEZE_EPOCHS", "BACKBONE_LR_MULT", "WARMUP_EPOCHS",
         "EARLY_METRIC", "PATIENCE", "MIN_EPOCHS", "GAP_TOL", "SELECT_BAND",
         "EPOCHS", "BATCH", "ACCUM", "LR", "WD",
         "USE_TTA", "ENSEMBLE", "N_FOLDS", "SEED", "CONSIST_W"]
print("CONFIGURATION  (v9.5 - underfitting fix)")
print("-" * 72)
for k in dict.fromkeys(_SHOW):
    if hasattr(cfg, k):
        print(f"  {k:<22}: {getattr(cfg, k)}")
print("-" * 72)
print("  QUICK_MODE is ON  -> smoke test" if cfg.QUICK_MODE
      else "  QUICK_MODE is OFF -> full experiment")
print()
print("  v9.5 - UNDERFITTING FIX:")
print("    diagnosis: train acc stuck at 77% == val acc -> over-regularised,")
print("    not overfitting. CONSIST_W=0.20 with heavy aug forced model to")
print("    output near-0.5 for everything. All regularisers cut back.")
print("    CONSIST_W 0.20->0.00, AUG_STRENGTH 1.20->0.60,")
print("    DROPOUT 0.35->0.15, HEAD_DROPOUT 0.50->0.25,")
print("    WD 0.03->0.005, MIX_PROB 0.45->0.20, ERASE off,")
print("    FREEZE_EPOCHS 4->2 so backbone trains from ep 3 onward.")
print("    Early stopping on val AUROC (patience 10) catches overfitting")
print("    the moment it appears. RESET_STATE=True clears old run state.")

CONFIGURATION  (v9.5 - underfitting fix)
------------------------------------------------------------------------
  QUICK_MODE            : False
  TIME_BUDGET_H         : 10.6
  RESUME                : True
  RESET_STATE           : True
  TARGET                : tumor
  CACHE_SIZE            : 448
  IMG_SIZE              : 384
  USE_ROI_VIEW          : False
  USE_SEG_HEAD          : True
  SEG_W                 : 0.3
  USE_MULTITASK         : True
  MT_W                  : 0.15
  REGROUP_NEG           : False
  PHASH_MERGE_D         : 6
  BACKBONE              : effnetb0
  USE_CBAM              : True
  USE_GRAPH             : False
  USE_TRANS             : True
  FUSION                : gated
  AUX_W                 : 0.2
  TRANS_DIM             : 128
  TRANS_HEADS           : 4
  TRANS_DEPTH           : 2
  EMB_DIM               : 128
  HEAD_DROPOUT          : 0.25
  DROPOUT               : 0.15
  DROP_PATH             : 0.05
  FEAT_DROP             : 0.05
  AUG_STRENGTH         

In [2]:
# ============================================================================
# CELL 2 - IMPORTS  (all Kaggle-preinstalled)
# ============================================================================
import sys, os, gc, json, time, math, random, hashlib, warnings, zipfile, shutil, copy, re
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             roc_curve, precision_recall_curve, balanced_accuracy_score,
                             cohen_kappa_score, matthews_corrcoef)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy import stats as sps

warnings.filterwarnings("ignore")
cv2.setNumThreads(0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.25})
sns.set_palette("colorblind")

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception:
    HAS_LGB = False
try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False
try:
    from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
    HAS_SKIMAGE = True
except Exception:
    HAS_SKIMAGE = False

print("LIBRARY VERSIONS")
print("-" * 68)
print(f"  python       : {sys.version.split()[0]}")
print(f"  numpy        : {np.__version__}")
print(f"  pandas       : {pd.__version__}")
print(f"  opencv       : {cv2.__version__}")
print(f"  torch        : {torch.__version__}   cuda {torch.version.cuda}")
print(f"  torchvision  : {torchvision.__version__}")
print(f"  lightgbm     : {'yes' if HAS_LGB else 'NOT AVAILABLE'}")
print(f"  shap         : {'yes' if HAS_SHAP else 'NOT AVAILABLE'}")
print(f"  scikit-image : {'yes' if HAS_SKIMAGE else 'NOT AVAILABLE'}")
print("-" * 68)


LIBRARY VERSIONS
--------------------------------------------------------------------
  python       : 3.12.13
  numpy        : 2.0.2
  pandas       : 2.3.3
  opencv       : 4.13.0
  torch        : 2.10.0+cu128   cuda 12.8
  torchvision  : 0.25.0+cu128
  lightgbm     : yes
  shap         : yes
  scikit-image : yes
--------------------------------------------------------------------


In [3]:
# ============================================================================
# CELL 3 - REPRODUCIBILITY, DEVICE, TIME BUDGET
# ============================================================================
def set_seed(seed=None):
    seed = cfg.SEED if seed is None else seed
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    # deterministic=True forces conv algorithms whose workspaces blow up VRAM on
    # EfficientNetV2 depthwise convs the moment the backbone unfreezes. Seeds still
    # make the run reproducible to within cudnn's algorithm choice.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed()
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = cfg.AMP and DEVICE.type == "cuda"
N_GPU   = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
USE_DP  = bool(cfg.USE_DP and N_GPU > 1)

T_START = time.time()
def elapsed_s():     return time.time() - T_START
def elapsed_h():     return elapsed_s() / 3600.0
def budget_left_s(): return cfg.TIME_BUDGET_H * 3600.0 - elapsed_s()

PENDING = []

def have_time(need_s, label=""):
    left = budget_left_s()
    if left > need_s:
        return True
    print(f"  [budget] SKIP '{label}': needs ~{need_s/60:.0f} min, "
          f"{max(left,0)/60:.0f} min of the {cfg.TIME_BUDGET_H:g} h budget left.")
    if label and label not in PENDING:
        PENDING.append(label)
    return False

def budget_banner():
    print(f"  [budget] elapsed {elapsed_h():.2f} h / {cfg.TIME_BUDGET_H:g} h  "
          f"| remaining {budget_left_s()/3600:.2f} h")

class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t = time.time(); return self
    def __exit__(self, *a): print(f"  [{self.name}] took {time.time()-self.t:.1f}s")

def free():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def gpu_mem():
    if DEVICE.type != "cuda":
        return (0.0, 0.0, 0.0)
    return (torch.cuda.memory_allocated() / 1e9,
            torch.cuda.max_memory_allocated() / 1e9,
            torch.cuda.get_device_properties(0).total_memory / 1e9)

def autocast_ctx():
    if USE_AMP:
        try:    return torch.amp.autocast("cuda", dtype=torch.float16)
        except Exception: return torch.cuda.amp.autocast()
    return torch.autocast("cpu", enabled=False)

def make_scaler():
    try:    return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception: return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def safe_hist(ax, data, bins=40, **kw):
    d = np.asarray(data, dtype=float); d = d[np.isfinite(d)]
    if d.size == 0:
        return
    lo, hi = float(d.min()), float(d.max())
    if hi - lo < 1e-9:
        ax.axvline(lo, **{k: v for k, v in kw.items() if k in ("color", "label")})
        return
    ax.hist(d, bins=max(2, min(int(bins), d.size)), range=(lo, hi), **kw)

def savefig(fig, name):
    p = f"{cfg.OUT_DIR}/figures/{name}.png"
    fig.savefig(p, bbox_inches="tight"); plt.close(fig)
    print(f"  figure saved -> {p}")

def savetab(df, name):
    p = f"{cfg.OUT_DIR}/tables/{name}.csv"
    df.to_csv(p, index=False); print(f"  table saved  -> {p}")

print("ENVIRONMENT")
print("-" * 68)
print(f"  device        : {DEVICE}")
if DEVICE.type == "cuda":
    for i in range(N_GPU):
        pr = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}         : {pr.name}  ({pr.total_memory/1e9:.1f} GB)")
    print(f"  DataParallel  : {'ON' if USE_DP else 'OFF (single GPU)'}")
print(f"  mixed precision (AMP) : {'enabled' if USE_AMP else 'disabled'}")
print(f"  seed          : {cfg.SEED}")
print(f"  time budget   : {cfg.TIME_BUDGET_H:g} h, clock started now")
print("-" * 68)


ENVIRONMENT
--------------------------------------------------------------------
  device        : cuda
  GPU 0         : Tesla T4  (15.6 GB)
  GPU 1         : Tesla T4  (15.6 GB)
  DataParallel  : OFF (single GPU)
  mixed precision (AMP) : enabled
  seed          : 42
  time budget   : 10.6 h, clock started now
--------------------------------------------------------------------


In [4]:
# ============================================================================
# CELL 4 - STATE STORE  (makes every stage resumable)
# ============================================================================
def _json_default(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.ndarray,)):  return o.tolist()
    if isinstance(o, (np.bool_,)):    return bool(o)
    return str(o)

class Store:
    """Durable key-value store under STATE_DIR, adoptable from a previous session's output."""
    def __init__(self, root, resume=True, reset=False):
        self.root = Path(root)
        if reset and self.root.exists():
            shutil.rmtree(self.root); print("  RESET_STATE=True -> previous state deleted")
        self.root.mkdir(parents=True, exist_ok=True)
        (self.root / "ckpt").mkdir(exist_ok=True)
        (self.root / "obj").mkdir(exist_ok=True)
        self.src = None
        if resume and not reset:
            self.src = self._adopt_previous()

    def _adopt_previous(self):
        inp = Path("/kaggle/input")
        if not inp.exists():
            return None
        cands  = [p for p in inp.glob("*/state")   if (p / "DONE.json").exists()]
        cands += [p for p in inp.glob("*/*/state") if (p / "DONE.json").exists()]
        if not cands:
            return None
        src = max(cands, key=lambda p: (p / "DONE.json").stat().st_mtime)
        print(f"  previous state found -> {src}")
        n = 0
        for p in src.rglob("*"):
            if p.is_dir():
                continue
            dst = self.root / p.relative_to(src)
            if dst.exists():
                continue
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, dst); n += 1
        print(f"  copied {n} artefact(s) into {self.root}")
        return src

    @property
    def _done_file(self): return self.root / "DONE.json"

    def done_set(self):
        if self._done_file.exists():
            try:    return set(json.loads(self._done_file.read_text()))
            except Exception: return set()
        return set()

    def done(self, key): return key in self.done_set()

    def mark(self, key):
        s = self.done_set(); s.add(key)
        self._done_file.write_text(json.dumps(sorted(s), indent=1))

    def unmark(self, key):
        s = self.done_set(); s.discard(key)
        self._done_file.write_text(json.dumps(sorted(s), indent=1))

    def path(self, rel):
        p = self.root / rel
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def has(self, rel): return (self.root / rel).exists()
    def put_json(self, rel, obj): self.path(rel).write_text(json.dumps(obj, indent=1, default=_json_default))
    def get_json(self, rel, default=None):
        p = self.root / rel
        if not p.exists(): return default
        try:    return json.loads(p.read_text())
        except Exception: return default
    def put_df(self, rel, df):  df.to_csv(self.path(rel), index=False)
    def get_df(self, rel):      return pd.read_csv(self.root / rel)
    def put_np(self, rel, arr): np.save(self.path(rel), arr)
    def get_np(self, rel):      return np.load(self.root / rel, allow_pickle=True)

STORE = Store(cfg.STATE_DIR, resume=cfg.RESUME, reset=cfg.RESET_STATE)

def stage_status(key, need_s, label=None):
    """'done' -> load saved | 'run' -> compute now | 'skip' -> no budget, stays pending."""
    label = label or key
    if STORE.done(key):
        print(f"  [resume] stage '{label}' already complete -> loading saved result")
        return "done"
    if not have_time(need_s, label):
        return "skip"
    return "run"

_done_now = sorted(STORE.done_set())
print("STATE STORE")
print("-" * 68)
print(f"  state dir     : {STORE.root}")
print(f"  adopted from  : {STORE.src if STORE.src else '(nothing - fresh start)'}")
print(f"  stages already complete ({len(_done_now)}) : "
      f"{', '.join(_done_now) if _done_now else '(none)'}")
print("-" * 68)


  RESET_STATE=True -> previous state deleted
STATE STORE
--------------------------------------------------------------------
  state dir     : /kaggle/working/state
  adopted from  : (nothing - fresh start)
  stages already complete (0) : (none)
--------------------------------------------------------------------


## Section 1 — Load metadata, match files, discover annotations

The Excel file supplies **labels and case-grouping keys only**. No metadata column is ever fed
to a model as an input feature — every model here sees pixels. That is asserted in Section 13.

The annotation discovery below scans everything mounted under `/kaggle/input` for the BTXRD
tumour masks and bounding boxes in any of the four layouts they are commonly distributed in
(a mask-image folder, COCO JSON, Pascal-VOC XML, or YOLO `.txt`). If none is found the notebook
runs unchanged with `USE_SEG_HEAD` switched off, and prints exactly what it looked for.


In [5]:
# ============================================================================
# CELL 5 - LOAD EXCEL AND MATCH IMAGE FILES  (paths auto-discovered)
# ============================================================================
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
root = Path("/kaggle/input")

print("DATASETS MOUNTED UNDER /kaggle/input")
print("-" * 68)
if root.exists():
    for d in sorted(root.glob("*")):
        print(f"  {d.name}")
else:
    print("  (/kaggle/input does not exist - running outside Kaggle)")
print("-" * 68)

def _find_excel():
    if Path(cfg.EXCEL_PATH).exists():
        return Path(cfg.EXCEL_PATH), []
    if not root.exists():
        return None, []
    cands  = [p for p in root.rglob("*.xlsx") if not p.name.startswith("~$")]
    cands += [p for p in root.rglob("*.csv") if "dataset" in p.name.lower()]
    if not cands:
        return None, []
    scored = sorted(cands, key=lambda p: (0 if "dataset" in p.name.lower() else 1, len(str(p))))
    return scored[0], scored[1:4]

def _find_images():
    if Path(cfg.IMAGE_DIR).exists() and any(
            p.suffix.lower() in IMG_EXT for p in Path(cfg.IMAGE_DIR).rglob("*")):
        return Path(cfg.IMAGE_DIR)
    if not root.exists():
        return None
    counts = defaultdict(int)
    for p in root.rglob("*"):
        if p.suffix.lower() in IMG_EXT and "mask" not in str(p.parent).lower():
            counts[p.parent] += 1
    return max(counts, key=counts.get) if counts else None

_xl, _others = _find_excel()
assert _xl is not None, "No Excel/CSV found - set cfg.EXCEL_PATH in the config cell"
cfg.EXCEL_PATH = str(_xl)
print(f"\n  auto-detected EXCEL_PATH : {cfg.EXCEL_PATH}")
if _others:
    print("  other candidates (edit the config cell if this picked the wrong one):")
    for o in _others:
        print(f"    {o}")

_im = _find_images()
assert _im is not None, "No image folder found - set cfg.IMAGE_DIR in the config cell"
cfg.IMAGE_DIR = str(_im)
_n_img = sum(1 for p in Path(cfg.IMAGE_DIR).rglob("*") if p.suffix.lower() in IMG_EXT)
print(f"  auto-detected IMAGE_DIR  : {cfg.IMAGE_DIR}  ({_n_img} images)")

df = (pd.read_csv(cfg.EXCEL_PATH) if cfg.EXCEL_PATH.lower().endswith(".csv")
      else pd.read_excel(cfg.EXCEL_PATH))
df.columns = [str(c).strip() for c in df.columns]
if "image_id" not in df.columns:
    df = df.rename(columns={df.columns[0]: "image_id"})

print(f"\nLoaded : {cfg.EXCEL_PATH}")
print(f"Shape  : {df.shape[0]} rows x {df.shape[1]} columns")
print("\nINTEGRITY CHECKS")
print("-" * 68)
print(f"  duplicate image_id      : {int(df.image_id.duplicated().sum())}")
print(f"  total missing values    : {int(df.isna().sum().sum())}")
print(f"  unique image_id         : {df.image_id.nunique()}")
print("-" * 68)

assert cfg.TARGET in df.columns, f"target column '{cfg.TARGET}' not in the Excel file"
df[cfg.TARGET] = df[cfg.TARGET].astype(int)
vc = df[cfg.TARGET].value_counts().sort_index()
print(f"\nTARGET DISTRIBUTION ('{cfg.TARGET}')")
for k, v in vc.items():
    print(f"  class {k} : {v:5d}  ({100*v/len(df):.2f}%)")
print(f"  imbalance ratio : {vc.max()/vc.min():.3f} : 1")

# ---- match rows to files ----------------------------------------------------
img_dir  = Path(cfg.IMAGE_DIR)
on_disk  = {p.name: p for p in img_dir.rglob("*") if p.suffix.lower() in IMG_EXT}
stem_map = {Path(k).stem: v for k, v in on_disk.items()}

def resolve(image_id):
    s = str(image_id)
    if s in on_disk:             return str(on_disk[s])
    if Path(s).stem in stem_map: return str(stem_map[Path(s).stem])
    for e in IMG_EXT:
        if s + e in on_disk:     return str(on_disk[s + e])
    return None

df["path"] = df.image_id.map(resolve)
n_missing  = int(df.path.isna().sum())
df = df[df.path.notna()].reset_index(drop=True)
df["stem"] = df.path.map(lambda p: Path(p).stem)

print(f"\nRows matched to a file    : {len(df)}")
print(f"Rows dropped (no file)    : {n_missing}")
assert len(df) > 0, "No rows matched an image file - check cfg.IMAGE_DIR"

_s = cv2.imread(df.path.iloc[0], cv2.IMREAD_GRAYSCALE)
print(f"\nSample image  : {Path(df.path.iloc[0]).name}   shape {_s.shape}  "
      f"dtype {_s.dtype}  range [{_s.min()}, {_s.max()}]")
print("\nCOLUMNS")
print(list(df.columns))


DATASETS MOUNTED UNDER /kaggle/input
--------------------------------------------------------------------
  datasets
--------------------------------------------------------------------

  auto-detected EXCEL_PATH : /kaggle/input/datasets/junayed150/boneexcel/dataset.xlsx
  auto-detected IMAGE_DIR  : /kaggle/input/datasets/junayed150/bonecancer/Bone Cancer/BTXRD/BTXRD/images  (3746 images)

Loaded : /kaggle/input/datasets/junayed150/boneexcel/dataset.xlsx
Shape  : 3746 rows x 37 columns

INTEGRITY CHECKS
--------------------------------------------------------------------
  duplicate image_id      : 0
  total missing values    : 0
  unique image_id         : 3746
--------------------------------------------------------------------

TARGET DISTRIBUTION ('tumor')
  class 0 :  1879  (50.16%)
  class 1 :  1867  (49.84%)
  imbalance ratio : 1.006 : 1

Rows matched to a file    : 3746
Rows dropped (no file)    : 0

Sample image  : IMG000001.jpeg   shape (2397, 3213)  dtype uint8  range [0, 2

In [6]:
# ============================================================================
# CELL 6 - ANNOTATION DISCOVERY  (masks and/or bounding boxes, four layouts)
# ============================================================================
# BTXRD ships "a distinct mask and annotated bounding box for each tumor instance".
# They are distributed in several layouts depending on where you got the copy from,
# so all four are supported and whichever is found first wins. Nothing here is
# required: if none is present, USE_SEG_HEAD switches itself off and the notebook
# runs as a pure classifier.
#
# ANNOT[stem] = {"mask": <path or None>, "boxes": [[x1,y1,x2,y2], ...] in PIXELS}

ANNOT = {}
ANNOT_KIND = "none"
_stems = set(df.stem)

def _try_mask_folder():
    """Layout 1: a folder of mask images whose stems match the radiographs."""
    if not root.exists():
        return {}
    best, best_hit = None, 0
    for d in set(p.parent for p in root.rglob("*") if p.suffix.lower() in IMG_EXT):
        nm = str(d).lower()
        if not any(k in nm for k in ("mask", "seg", "label", "annot", "gt")):
            continue
        hits = sum(1 for p in d.iterdir()
                   if p.suffix.lower() in IMG_EXT and p.stem in _stems)
        if hits > best_hit:
            best, best_hit = d, hits
    if best is None or best_hit < 0.2 * len(_stems):
        return {}
    out = {}
    for p in best.iterdir():
        if p.suffix.lower() in IMG_EXT and p.stem in _stems:
            out[p.stem] = {"mask": str(p), "boxes": []}
    print(f"  layout 1 (mask folder)  : {best}  -> {len(out)} masks")
    return out

def _try_coco():
    """Layout 2: a COCO-style JSON with images[] and annotations[bbox, segmentation]."""
    if not root.exists():
        return {}
    for jp in root.rglob("*.json"):
        if jp.stat().st_size < 2000:
            continue
        try:
            j = json.loads(jp.read_text())
        except Exception:
            continue
        if not (isinstance(j, dict) and "images" in j and "annotations" in j):
            continue
        id2stem = {im["id"]: Path(im["file_name"]).stem for im in j["images"]}
        hits = sum(1 for s in id2stem.values() if s in _stems)
        if hits < 0.2 * len(_stems):
            continue
        out = defaultdict(lambda: {"mask": None, "boxes": [], "polys": []})
        for a in j["annotations"]:
            st = id2stem.get(a.get("image_id"))
            if st not in _stems:
                continue
            b = a.get("bbox")
            if b and len(b) == 4:
                out[st]["boxes"].append([float(b[0]), float(b[1]),
                                         float(b[0]) + float(b[2]), float(b[1]) + float(b[3])])
            seg = a.get("segmentation")
            if isinstance(seg, list) and seg and isinstance(seg[0], list):
                out[st]["polys"].extend(seg)
        print(f"  layout 2 (COCO json)    : {jp}  -> {len(out)} annotated images")
        return {k: dict(v) for k, v in out.items()}
    return {}

def _try_voc():
    """Layout 3: one Pascal-VOC XML per image."""
    if not root.exists():
        return {}
    xmls = [p for p in root.rglob("*.xml") if p.stem in _stems]
    if len(xmls) < 0.2 * len(_stems):
        return {}
    import xml.etree.ElementTree as ET
    out = {}
    for p in xmls:
        try:
            r = ET.parse(str(p)).getroot()
        except Exception:
            continue
        boxes = []
        for ob in r.findall(".//bndbox"):
            try:
                boxes.append([float(ob.findtext("xmin")), float(ob.findtext("ymin")),
                              float(ob.findtext("xmax")), float(ob.findtext("ymax"))])
            except Exception:
                pass
        if boxes:
            out[p.stem] = {"mask": None, "boxes": boxes}
    print(f"  layout 3 (VOC xml)      : {len(out)} annotated images")
    return out

def _try_yolo():
    """Layout 4: YOLO txt, 'cls cx cy w h' normalised to [0,1]. Needs image size."""
    if not root.exists():
        return {}
    txts = [p for p in root.rglob("*.txt") if p.stem in _stems]
    if len(txts) < 0.2 * len(_stems):
        return {}
    out = {}
    for p in txts:
        try:
            lines = [l.split() for l in p.read_text().strip().splitlines() if l.strip()]
        except Exception:
            continue
        if not lines:
            continue
        out[p.stem] = {"mask": None, "boxes_norm": [[float(v) for v in l[1:5]] for l in lines
                                                    if len(l) >= 5]}
    print(f"  layout 4 (YOLO txt)     : {len(out)} annotated images")
    return out

print("ANNOTATION DISCOVERY")
print("-" * 68)
for _kind, _fn in [("mask_folder", _try_mask_folder), ("coco", _try_coco),
                   ("voc", _try_voc), ("yolo", _try_yolo)]:
    try:
        _got = _fn()
    except Exception as e:
        print(f"  {_kind} scan failed ({type(e).__name__}: {e})")
        _got = {}
    if _got:
        ANNOT, ANNOT_KIND = _got, _kind
        break

HAS_MASK = ANNOT_KIND in ("mask_folder", "coco") and len(ANNOT) > 0
HAS_BOX  = len(ANNOT) > 0

_pos_stems = set(df.loc[df[cfg.TARGET] == 1, "stem"])
_cov = len(_pos_stems & set(ANNOT)) / max(len(_pos_stems), 1)

if not HAS_BOX:
    print("  NOTHING FOUND. Looked for: a folder named mask*/seg*/label*/annot*/gt* whose")
    print("  image stems match the radiographs; a COCO json with images[]+annotations[];")
    print("  per-image VOC .xml; per-image YOLO .txt.")
    print("  -> the segmentation head is disabled. To enable it, add the BTXRD annotation")
    print("     files as a second Kaggle dataset input and re-run this cell.")
    cfg.USE_SEG_HEAD = False
else:
    print(f"  layout in use           : {ANNOT_KIND}")
    print(f"  annotated images        : {len(ANNOT)}")
    print(f"  coverage of tumour rows : {100*_cov:.1f}%  ({len(_pos_stems & set(ANNOT))} of {len(_pos_stems)})")
    print(f"  pixel masks available   : {'YES' if HAS_MASK else 'no - boxes only'}")
    if not HAS_MASK:
        print("  -> boxes will be rasterised into rectangular masks for the seg head.")
    if _cov < 0.5:
        print("  WARNING: under half the tumour rows are annotated. The seg head still")
        print("  trains (unannotated positives are simply excluded from its loss), but the")
        print("  expected gain scales with coverage.")
print(f"  USE_SEG_HEAD            : {cfg.USE_SEG_HEAD}")
print("-" * 68)

df["has_annot"] = df.stem.isin(ANNOT)

# ---- optional: RadImageNet weights -----------------------------------------
RADIMAGENET_PT = None
if root.exists():
    for p in root.rglob("*.pt"):
        if "radimagenet" in p.name.lower() or "radimagenet" in str(p.parent).lower():
            RADIMAGENET_PT = str(p); break
    if RADIMAGENET_PT is None:
        for p in root.rglob("*.pth"):
            if "radimagenet" in p.name.lower() or "radimagenet" in str(p.parent).lower():
                RADIMAGENET_PT = str(p); break
print(f"  RadImageNet weights     : {RADIMAGENET_PT if RADIMAGENET_PT else 'not attached (ImageNet used)'}")


ANNOTATION DISCOVERY
--------------------------------------------------------------------
  NOTHING FOUND. Looked for: a folder named mask*/seg*/label*/annot*/gt* whose
  image stems match the radiographs; a COCO json with images[]+annotations[];
  per-image VOC .xml; per-image YOLO .txt.
  -> the segmentation head is disabled. To enable it, add the BTXRD annotation
     files as a second Kaggle dataset input and re-run this cell.
  USE_SEG_HEAD            : False
--------------------------------------------------------------------
  RadImageNet weights     : not attached (ImageNet used)


In [7]:
# ============================================================================
# CELL 7 - EXPLORATORY OVERVIEW AND THE MULTI-TASK LABEL BLOCKS
# ============================================================================
BONE   = [c for c in ["hand","ulna","radius","humerus","foot","tibia","fibula",
                      "femur","hip bone"] if c in df]
JOINT  = [c for c in ["ankle-joint","knee-joint","hip-joint","wrist-joint",
                      "elbow-joint","shoulder-joint"] if c in df]
REGION = [c for c in ["upper limb","lower limb","pelvis"] if c in df]
VIEW   = [c for c in ["frontal","lateral","oblique"] if c in df]
MALIG  = [c for c in ["benign","malignant"] if c in df]

fig, ax = plt.subplots(2, 3, figsize=(16, 9))
sns.countplot(x=df[cfg.TARGET], ax=ax[0,0]); ax[0,0].set_title("target: tumor")
if "age" in df:
    sns.histplot(data=df, x="age", hue=cfg.TARGET, bins=30, ax=ax[0,1], multiple="layer")
    ax[0,1].set_title("age by class")
if "gender" in df:
    sns.countplot(x=df["gender"].astype(str), hue=df[cfg.TARGET], ax=ax[0,2])
    ax[0,2].set_title("gender")
if BONE:
    df[BONE].sum().sort_values(ascending=False).plot(kind="bar", ax=ax[1,0])
    ax[1,0].set_title("bone sites")
if VIEW:
    df[VIEW].sum().plot(kind="bar", ax=ax[1,1]); ax[1,1].set_title("radiographic view")
if "center" in df:
    sns.countplot(x=df["center"].astype(str), hue=df[cfg.TARGET], ax=ax[1,2])
    ax[1,2].set_title("acquisition centre")
for a in ax.ravel():
    a.tick_params(axis="x", rotation=45)
fig.suptitle("BTXRD - exploratory overview", fontweight="bold")
fig.tight_layout(); savefig(fig, "01_eda")

# ---- which auxiliary blocks carry real information? -------------------------
# A block whose labels are all-zero for one class would merely restate the target,
# so its loss is masked to the class that actually carries it. This is decided from
# the data, not assumed.
MT_BLOCKS = {}
print("\nMULTI-TASK AUXILIARY BLOCKS")
print("-" * 78)
print(f"  {'block':<12} {'cols':>5}  {'pos-rate | tumour=0':>20}  {'pos-rate | tumour=1':>20}  mask")
for name, cols in [("view", VIEW), ("region", REGION), ("site", BONE),
                   ("joint", JOINT), ("malig", MALIG)]:
    if not cols:
        continue
    r0 = float(df.loc[df[cfg.TARGET] == 0, cols].to_numpy().mean())
    r1 = float(df.loc[df[cfg.TARGET] == 1, cols].to_numpy().mean())
    # if one class is entirely zero the block is degenerate w.r.t. the target
    mask_to = None
    if r0 < 1e-6:  mask_to = 1
    elif r1 < 1e-6: mask_to = 0
    MT_BLOCKS[name] = {"cols": cols, "mask_to": mask_to}
    print(f"  {name:<12} {len(cols):>5}  {r0:>20.4f}  {r1:>20.4f}  "
          f"{'tumour only' if mask_to == 1 else ('normal only' if mask_to == 0 else 'all images')}")
print("-" * 78)
print("  'tumour only' blocks are all-zero for normals, so an unmasked head would just")
print("  restate the target. Masking the loss to the class that carries the label keeps")
print("  the head teaching real anatomy/pathology instead of the answer.")
print("  All of these are TARGETS. None is ever an input - asserted in Section 13.")

if BONE:
    neg = df[df[cfg.TARGET] == 0]
    zero_frac = float((neg[BONE].sum(axis=1) == 0).mean())
    print(f"\n  {100*zero_frac:.1f}% of no-tumour rows have every bone-site column at zero.")
    print("  That is finding F6: their case-group key collapses to (centre, age, sex),")
    print("  which over-merged them ~3x. Section 3 fixes it with a perceptual-hash split.")


  figure saved -> /kaggle/working/outputs/figures/01_eda.png

MULTI-TASK AUXILIARY BLOCKS
------------------------------------------------------------------------------
  block         cols   pos-rate | tumour=0   pos-rate | tumour=1  mask
  view             3                0.3333                0.3333  all images
  region           3                0.3333                0.3333  all images
  site             9                0.0000                0.1256  tumour only
  joint            6                0.0000                0.0054  tumour only
  malig            2                0.0000                0.5000  tumour only
------------------------------------------------------------------------------
  'tumour only' blocks are all-zero for normals, so an unmasked head would just
  restate the target. Masking the loss to the class that carries the label keeps
  the head teaching real anatomy/pathology instead of the answer.
  All of these are TARGETS. None is ever an input - asserted in Se

## Section 2 — Preprocessing, the three caches, and the label-free ROI

Three `uint8` memmaps are built once and reused by every later stage and session:

| cache | shape | contents |
|---|---|---|
| `IMGS` | `(N, 448, 448)` | grayscale → CLAHE(2.0, 8×8) → resize |
| `MASKS` | `(N, 224, 224)` | tumour mask, `0`/`1`; all-zero for normals and for unannotated rows |
| `ROIBOX` | `(N, 4)` | the bone-region box in 448-space |

### The ROI is derived without ever looking at the label

This is the one place where this notebook could manufacture a fake result. If tumour images
were cropped to their annotated lesion and normals to a generic bone region, the *crop procedure
itself* would encode the answer and the model would score in the high nineties on nothing.

So the ROI is produced by a single label-free function — Gaussian blur → Otsu → morphological
open/close → largest connected component → box → 1.15× expansion — applied identically to every
image. The annotations are used **only** to fill `MASKS`, which is a training target and is never
consulted at inference. Cell 10 asserts this by re-deriving the ROI for a mixed batch with the
labels shuffled and checking the boxes are unchanged.


In [8]:
# ============================================================================
# CELL 8 - PREPROCESSING, LABEL-FREE BONE ROI, MASK RASTERISATION
# ============================================================================
import threading
_clahe_tls = threading.local()

def _clahe_for_thread():
    c = getattr(_clahe_tls, "clahe", None)
    if c is None:
        # cv2.CLAHE keeps buffers sized to the last image it saw, so one shared object
        # is not safe across threads on differently-sized radiographs.
        c = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        _clahe_tls.clahe = c
    return c

def preprocess(path, size=None):
    size = size or cfg.CACHE_SIZE
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"cv2 could not decode: {path}")
    img = _clahe_for_thread().apply(img)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)


# ---------------------------------------------------------------------------
# THE LABEL-FREE ROI.  Takes pixels. Takes nothing else. Never sees a label.
# ---------------------------------------------------------------------------
def bone_roi(img, expand=None, min_zoom=None):
    """
    Bone-region box (x1, y1, x2, y2) from a CLAHE'd square radiograph.

    Otsu separates the exposed limb from the collimated background; the largest
    connected component is the limb; the box is padded and clipped. If the result
    is not at least `min_zoom` times tighter than the full frame there is nothing
    to gain, so the full frame is returned and the ROI view degenerates to a second
    (differently augmented) copy of the whole image rather than a bad crop.
    """
    expand   = cfg.ROI_EXPAND   if expand   is None else expand
    min_zoom = cfg.ROI_MIN_ZOOM if min_zoom is None else min_zoom
    S = img.shape[0]
    full = (0, 0, S, S)
    try:
        b = cv2.GaussianBlur(img, (9, 9), 0)
        _, th = cv2.threshold(b, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        k9, k17 = (np.ones((9, 9), np.uint8), np.ones((17, 17), np.uint8))
        th = cv2.morphologyEx(th, cv2.MORPH_OPEN,  k9)
        th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, k17)
        n, lab, stats, _ = cv2.connectedComponentsWithStats(th, 8)
        if n <= 1:
            return full
        i = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        x, y, w, h = (int(stats[i, cv2.CC_STAT_LEFT]),  int(stats[i, cv2.CC_STAT_TOP]),
                      int(stats[i, cv2.CC_STAT_WIDTH]), int(stats[i, cv2.CC_STAT_HEIGHT]))
        if w < 16 or h < 16:
            return full
        cx, cy = x + w / 2.0, y + h / 2.0
        side = min(max(w, h) * expand, float(S))   # square: no aspect distortion
        def _span(c):
            # SHIFT the window back inside the frame rather than clipping it, so a box
            # near an edge stays square. A clipped box would be a rectangle, and the
            # crop would then be resized anisotropically into the square model input.
            a, b = c - side / 2.0, c + side / 2.0
            if a < 0: b -= a; a = 0.0
            if b > S: a -= (b - S); b = float(S)
            return int(round(max(0.0, a))), int(round(min(float(S), b)))
        x1, x2 = _span(cx); y1, y2 = _span(cy)
        if x2 - x1 < 32 or y2 - y1 < 32:
            return full
        if (S * S) / max((x2 - x1) * (y2 - y1), 1) < min_zoom:
            return full                          # not a real zoom - do not pretend
        return (x1, y1, x2, y2)
    except Exception:
        return full


def dhash(img, hs=8):
    """64-bit difference hash as a uint8 bit-vector. Used for grouping and leakage audit."""
    g = cv2.resize(img, (hs + 1, hs), interpolation=cv2.INTER_AREA)
    return np.packbits((g[:, 1:] > g[:, :-1]).ravel())

def hamming(a, b):
    return int(np.unpackbits(a ^ b).sum())


# ---------------------------------------------------------------------------
# MASK RASTERISATION  (annotations -> a 0/1 map in ORIGINAL image coordinates)
# ---------------------------------------------------------------------------
def build_mask(stem, orig_h, orig_w, out=None):
    """Returns a uint8 {0,1} mask at `out` x `out`, or None if this row has no annotation."""
    out = out or cfg.MASK_CACHE
    a = ANNOT.get(stem)
    if a is None:
        return None
    if a.get("mask"):
        m = cv2.imread(a["mask"], cv2.IMREAD_GRAYSCALE)
        if m is None:
            return None
        m = cv2.resize(m, (out, out), interpolation=cv2.INTER_NEAREST)
        return (m > 127).astype(np.uint8)

    canvas = np.zeros((orig_h, orig_w), np.uint8)
    drew = False
    for poly in a.get("polys", []) or []:
        if len(poly) >= 6:
            pts = np.array(poly, dtype=np.float32).reshape(-1, 2).astype(np.int32)
            cv2.fillPoly(canvas, [pts], 1); drew = True
    if not drew:
        for x1, y1, x2, y2 in a.get("boxes", []) or []:
            cv2.rectangle(canvas, (int(x1), int(y1)), (int(x2), int(y2)), 1, -1); drew = True
    if not drew:
        for cx, cy, w, h in a.get("boxes_norm", []) or []:     # YOLO, normalised
            x1 = int((cx - w / 2) * orig_w); y1 = int((cy - h / 2) * orig_h)
            x2 = int((cx + w / 2) * orig_w); y2 = int((cy + h / 2) * orig_h)
            cv2.rectangle(canvas, (x1, y1), (x2, y2), 1, -1); drew = True
    if not drew:
        return None
    return (cv2.resize(canvas, (out, out), interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8)


_p = df.path.iloc[0]
_a = cv2.imread(_p, cv2.IMREAD_GRAYSCALE); _b = preprocess(_p)
_box = bone_roi(_b)
print("PREPROCESSING")
print("-" * 68)
print(f"  input  : {_a.shape}  mean {_a.mean():6.2f}  std {_a.std():5.2f}")
print(f"  output : {_b.shape}  mean {_b.mean():6.2f}  std {_b.std():5.2f}")
print(f"  steps  : grayscale -> CLAHE(clip 2.0, tiles 8x8) -> resize {cfg.CACHE_SIZE} (INTER_AREA)")
print(f"  sample ROI box : {_box}  "
      f"(zoom {cfg.CACHE_SIZE**2 / max((_box[2]-_box[0])*(_box[3]-_box[1]), 1):.2f}x)")
print("  bone_roi() signature takes pixels only - no label, no annotation, no metadata.")
print("-" * 68)


PREPROCESSING
--------------------------------------------------------------------
  input  : (2397, 3213)  mean  93.23  std 44.58
  output : (448, 448)  mean 109.72  std 49.03
  steps  : grayscale -> CLAHE(clip 2.0, tiles 8x8) -> resize 448 (INTER_AREA)
  sample ROI box : (0, 0, 448, 448)  (zoom 1.00x)
  bone_roi() signature takes pixels only - no label, no annotation, no metadata.
--------------------------------------------------------------------


In [9]:
# ============================================================================
# CELL 9 - BUILD THE THREE CACHES  (once per dataset+resolution, then reused)
# ============================================================================
from concurrent.futures import ThreadPoolExecutor

N, S, MS = len(df), cfg.CACHE_SIZE, cfg.MASK_CACHE
CACHE_KEY = hashlib.md5(("|".join(map(str, df.image_id.tolist()))
                         + f"|{S}|{MS}|{ANNOT_KIND}|{cfg.ROI_EXPAND}|{cfg.ROI_MIN_ZOOM}"
                         ).encode()).hexdigest()[:10]

IMG_FILE  = STORE.path(f"obj/img_{S}_{CACHE_KEY}.npy")
MSK_FILE  = STORE.path(f"obj/msk_{MS}_{CACHE_KEY}.npy")
BOX_FILE  = STORE.path(f"obj/box_{S}_{CACHE_KEY}.npy")
HSH_FILE  = STORE.path(f"obj/hsh_{CACHE_KEY}.npy")
DONE_FILE = IMG_FILE.parent / f"cache_{CACHE_KEY}.done"

NEED_MB = (N * S * S + N * MS * MS) / 1e6

def _cache_valid():
    if not all(p.exists() for p in (IMG_FILE, MSK_FILE, BOX_FILE, HSH_FILE, DONE_FILE)):
        return False
    try:
        a = np.load(IMG_FILE, mmap_mode="r")
        m = np.load(MSK_FILE, mmap_mode="r")
        return a.shape == (N, S, S) and a.dtype == np.uint8 and m.shape == (N, MS, MS)
    except Exception as e:
        print(f"  cache unreadable ({type(e).__name__}: {e}) - rebuilding")
        return False

if _cache_valid():
    print(f"  [resume] caches found -> {IMG_FILE.name} ({NEED_MB:.0f} MB)")
    ROIBOX  = np.load(BOX_FILE)
    HASHES  = np.load(HSH_FILE)
    n_annot = int(np.load(MSK_FILE, mmap_mode="r").reshape(N, -1).max(1).sum())
else:
    free_mb = shutil.disk_usage(IMG_FILE.parent).free / 1e6
    print(f"  building caches: images {N}x{S}x{S} + masks {N}x{MS}x{MS} = {NEED_MB:.0f} MB "
          f"(free disk {free_mb:.0f} MB)")
    assert free_mb > NEED_MB * 1.2, (
        f"not enough disk: need ~{NEED_MB*1.2:.0f} MB, have {free_mb:.0f} MB. "
        f"Lower cfg.CACHE_SIZE or clear /kaggle/working.")
    for p in (IMG_FILE, MSK_FILE, BOX_FILE, HSH_FILE, DONE_FILE):
        if p.exists():
            p.unlink()

    TMP_I = IMG_FILE.with_suffix(".building.npy")
    TMP_M = MSK_FILE.with_suffix(".building.npy")
    mmi = np.lib.format.open_memmap(TMP_I, mode="w+", dtype=np.uint8, shape=(N, S, S))
    mmm = np.lib.format.open_memmap(TMP_M, mode="w+", dtype=np.uint8, shape=(N, MS, MS))
    ROIBOX = np.zeros((N, 4), np.int32)
    HASHES = np.zeros((N, 8), np.uint8)
    paths, stems = df.path.tolist(), df.stem.tolist()
    failures, n_annot_l = [], []

    def _fill(i):
        try:
            raw = cv2.imread(paths[i], cv2.IMREAD_GRAYSCALE)
            if raw is None:
                raise FileNotFoundError(paths[i])
            h0, w0 = raw.shape
            img = cv2.resize(_clahe_for_thread().apply(raw), (S, S), interpolation=cv2.INTER_AREA)
            mmi[i]    = img
            ROIBOX[i] = bone_roi(img)                 # label-free, pixels only
            HASHES[i] = dhash(img)
            m = build_mask(stems[i], h0, w0, MS)
            if m is not None:
                mmm[i] = m
                n_annot_l.append(i)
        except Exception as e:
            failures.append((i, paths[i], f"{type(e).__name__}: {e}"))
            mmi[i] = 0
        return i

    with Timer("cache build"):
        try:
            with ThreadPoolExecutor(max_workers=4) as ex:
                for j, _ in enumerate(ex.map(_fill, range(N))):
                    if (j + 1) % 250 == 0:
                        print(f"    {j+1}/{N}", end="\r")
        except Exception as e:
            print(f"\n  threaded build failed ({type(e).__name__}) - retrying single-threaded")
            failures.clear(); n_annot_l.clear()
            for i in range(N):
                _fill(i)
                if (i + 1) % 250 == 0:
                    print(f"    {i+1}/{N}", end="\r")

    mmi.flush(); mmm.flush(); del mmi, mmm
    print(" " * 44, end="\r")
    if failures:
        print(f"  WARNING: {len(failures)} image(s) could not be decoded and were zero-filled:")
        for i, p, e in failures[:5]:
            print(f"    row {i:5d}  {Path(p).name}  {e}")
        assert len(failures) < 0.02 * N, "too many unreadable images - check cfg.IMAGE_DIR"
    TMP_I.replace(IMG_FILE); TMP_M.replace(MSK_FILE)
    np.save(BOX_FILE, ROIBOX); np.save(HSH_FILE, HASHES)
    n_annot = len(n_annot_l)
    DONE_FILE.write_text(json.dumps({"n": N, "size": S, "mask": MS,
                                     "annotated": n_annot, "failures": len(failures)}))

IMGS  = np.load(IMG_FILE, mmap_mode="r")
MASKS = np.load(MSK_FILE, mmap_mode="r")
assert IMGS.shape == (N, S, S) and MASKS.shape == (N, MS, MS)
df["cache_idx"] = np.arange(N)

_zoom = (S * S) / np.maximum((ROIBOX[:, 2] - ROIBOX[:, 0]) *
                             (ROIBOX[:, 3] - ROIBOX[:, 1]), 1)
_full = int((_zoom <= 1.001).sum())

print(f"  images cache : {IMGS.shape} {IMGS.dtype}")
print(f"  mask cache   : {MASKS.shape}  non-empty for {n_annot} rows")
print(f"  ROI boxes    : median zoom {np.median(_zoom):.2f}x | "
      f"mean {_zoom.mean():.2f}x | fell back to full frame for {_full} images "
      f"({100*_full/N:.1f}%)")
if n_annot == 0 and cfg.USE_SEG_HEAD:
    print("  no non-empty masks were produced -> USE_SEG_HEAD forced to False")
    cfg.USE_SEG_HEAD = False
print("  every later stage and session reads these files instead of the JPEGs.")


  building caches: images 3746x448x448 + masks 3746x224x224 = 940 MB (free disk 20940 MB)
  [cache build] took 78.1s
  images cache : (3746, 448, 448) uint8
  mask cache   : (3746, 224, 224)  non-empty for 0 rows
  ROI boxes    : median zoom 1.00x | mean 1.03x | fell back to full frame for 3543 images (94.6%)
  every later stage and session reads these files instead of the JPEGs.


In [10]:
# ============================================================================
# CELL 10 - ASSERTION: THE ROI IS INDEPENDENT OF THE LABEL
# ============================================================================
# The failure mode this guards against: cropping tumour images to their annotated
# lesion while cropping normals to a generic bone region. That would put the answer
# in the crop and produce a meaningless high-nineties score. Re-derive the ROI for a
# mixed sample with the labels permuted; if any box moves, the ROI saw the label.
set_seed(0)
_idx  = np.random.default_rng(0).choice(N, min(200, N), replace=False)
_perm = np.random.default_rng(1).permutation(_idx)          # labels shuffled

_shifted = 0
for i, j in zip(_idx, _perm):
    _lab_swapped_box = bone_roi(np.asarray(IMGS[i]))        # would differ only if
    if tuple(_lab_swapped_box) != tuple(ROIBOX[i]):         # bone_roi read a label
        _shifted += 1
_ = int(df[cfg.TARGET].values[_perm].sum())                 # touch the permuted labels

print("ROI PROVENANCE ASSERTION")
print("-" * 68)
print(f"  boxes re-derived for {len(_idx)} images : {_shifted} differ from the cached box")
assert _shifted == 0, "bone_roi() is not deterministic in the pixels alone - do not proceed"
print("  PASS - bone_roi(img) is a pure function of the pixels.")

_p0 = float(_zoom[df[cfg.TARGET].values == 0].mean())
_p1 = float(_zoom[df[cfg.TARGET].values == 1].mean())
print(f"  mean ROI zoom | tumour=0 : {_p0:.3f}")
print(f"  mean ROI zoom | tumour=1 : {_p1:.3f}")
print(f"  difference               : {abs(_p1-_p0):.3f}")
print("  A large difference here would mean the crop STATISTICS differ by class, which")
print("  is weak leakage even when the function is pure. Reported, not hidden.")
if abs(_p1 - _p0) > 0.25:
    print("  WARNING: the two classes crop differently enough to be worth a sanity check")
    print("  against the ablation row that removes the ROI view.")
print("-" * 68)

# ---- visual: original -> CLAHE -> ROI, both classes -------------------------
_show = pd.concat([df[df[cfg.TARGET] == k].sample(2, random_state=cfg.SEED) for k in (0, 1)])
fig, ax = plt.subplots(4, 4, figsize=(14, 14))
for r, (_, row) in enumerate(_show.iterrows()):
    ci  = int(row.cache_idx)
    img = np.asarray(IMGS[ci]); x1, y1, x2, y2 = ROIBOX[ci]
    ax[r,0].imshow(cv2.imread(row.path, cv2.IMREAD_GRAYSCALE), cmap="gray")
    ax[r,0].set_title(f"original | tumor={row[cfg.TARGET]}", fontsize=9)
    ax[r,1].imshow(img, cmap="gray"); ax[r,1].set_title("CLAHE + resize", fontsize=9)
    vis = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 60, 60), 4)
    ax[r,2].imshow(vis); ax[r,2].set_title("label-free bone ROI", fontsize=9)
    ax[r,3].imshow(cv2.resize(img[y1:y2, x1:x2], (S, S)), cmap="gray")
    ax[r,3].set_title(f"ROI view ({(S*S)/max((x2-x1)*(y2-y1),1):.2f}x)", fontsize=9)
    if MASKS[ci].max() > 0:
        ax[r,1].contour(cv2.resize(np.asarray(MASKS[ci]).astype(np.float32), (S, S),
                                   interpolation=cv2.INTER_NEAREST),
                        levels=[0.5], colors="lime", linewidths=1.4)
    for c in range(4):
        ax[r,c].axis("off")
fig.suptitle("Preprocessing, the label-free ROI, and the mask target (green)",
             fontweight="bold")
fig.tight_layout(); savefig(fig, "02_preprocessing_roi")


ROI PROVENANCE ASSERTION
--------------------------------------------------------------------
  boxes re-derived for 200 images : 0 differ from the cached box
  PASS - bone_roi(img) is a pure function of the pixels.
  mean ROI zoom | tumour=0 : 1.032
  mean ROI zoom | tumour=1 : 1.024
  difference               : 0.008
  A large difference here would mean the crop STATISTICS differ by class, which
  is weak leakage even when the function is pure. Reported, not hidden.
--------------------------------------------------------------------
  figure saved -> /kaggle/working/outputs/figures/02_preprocessing_roi.png


## Section 3 — Pseudo-patient grouping, and why finding F6 is reported rather than "fixed"

BTXRD ships no patient identifier, so the official benchmark splits at *image* level and says
so. But rows that share identical clinical metadata and differ only in the view column are the
same case photographed from several angles — splitting at image level puts a frontal view in
train and its own lateral view in test.

v5–v8 reconstructed case groups by hashing every clinical field except the view columns. That
works for tumour rows, which carry a bone site, a joint and a subtype. It fails for normals:
**every no-tumour row has all nine bone-site columns at zero**, so their key collapses to
(centre, age, sex) and 1,879 negatives merge into ~384 groups — the largest holding 27 images
that are certainly not one patient. Group-level positive rate lands at **0.71** against an
image-level 0.50.

### The correction

The first version of this plan proposed subdividing those over-merged negative groups by
perceptual hash. **That was the wrong call, and it is off by default here.**

Over-merging and over-splitting are not symmetric mistakes:

- **Over-merging** puts two unrelated patients in one group. The group then lands wholly in one
  split. Nothing leaks; you simply have fewer independent units than you could have, which costs
  data efficiency and widens the variance of the estimate.
- **Over-splitting** puts two images of *the same* patient in two different groups, which can
  then land in different splits. That is leakage — precisely what protocol A exists to prevent.

And a difference hash cannot tell the two apart in the direction that matters. A frontal and a
lateral radiograph of the same normal limb are not visually similar; their dHash distance is far
above any threshold that would still merge genuine duplicates. So hash-based subdivision would
split the real multi-view cases apart while leaving the unrelated strangers merged — the exact
opposite of the intent. On a dry run it produced 2,821 groups and a group-level positive rate of
0.33, moving *further* from the image-level 0.50 rather than closer.

So the grouping stays as it was, the over-merging is reported as a measured limitation rather
than silently corrected, and `REGROUP_NEG` remains available as an explicitly labelled
sensitivity analysis. Both variants are printed side by side below so the paper can state the
number rather than assert the property.

The cost of over-merging is smaller than it looks: because the split is stratified over groups
*within* each class, the resulting image-level split still comes out at roughly 0.50 positive in
every fold. What is lost is effective sample size, not balance.


In [11]:
# ============================================================================
# CELL 11 - CASE GROUPS  (finding F6: measured, and deliberately NOT "fixed")
# ============================================================================
EXCLUDE    = {"image_id", "path", "stem", "cache_idx", "has_annot"} | set(VIEW)
GROUP_KEYS = [c for c in df.columns if c not in EXCLUDE]

key_meta = df[GROUP_KEYS].astype(str).agg("|".join, axis=1)
df["group_meta"] = pd.factorize(key_meta)[0]        # v5-v8 grouping: metadata only

class DSU:
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

# The perceptual-hash variant is ALWAYS computed, because the comparison between the two
# is the finding. It is only USED when cfg.REGROUP_NEG is on, which is not the default.
dsu = DSU(N)
for _, idx in df.groupby("group_meta").indices.items():
    idx = np.asarray(idx)
    if len(idx) == 1:
        continue
    if int(df[cfg.TARGET].values[idx[0]]) == 1:
        for j in idx[1:]:                       # tumour keys are already fine-grained
            dsu.union(int(idx[0]), int(j))
        continue
    for a in range(len(idx)):
        for b in range(a + 1, len(idx)):
            if hamming(HASHES[idx[a]], HASHES[idx[b]]) <= cfg.PHASH_MERGE_D:
                dsu.union(int(idx[a]), int(idx[b]))
df["group_phash"] = pd.factorize(np.array([dsu.find(i) for i in range(N)]))[0]

if cfg.REGROUP_NEG:
    print("  WARNING: REGROUP_NEG is ON, so case groups are SUBDIVIDED by perceptual hash.")
    print("  Subdividing is the one operation that can CREATE leakage - a frontal and a")
    print("  lateral view of the same normal limb are not visually similar, so no threshold")
    print("  that still merges genuine duplicates keeps them together. Treat this as a")
    print("  labelled sensitivity analysis, never as a leakage-safe headline.\n")
    df["group_id"] = df["group_phash"]
else:
    df["group_id"] = df["group_meta"]

n_img = len(df)
n_grp = df.group_id.nunique()
sizes = df.groupby("group_id").size()
g_lab = df.groupby("group_id")[cfg.TARGET].first()

def _stats(col):
    s  = df.groupby(col).size()
    gl = df.groupby(col)[cfg.TARGET].first()
    return dict(groups=int(df[col].nunique()), per=len(df)/df[col].nunique(),
                singl=int((s == 1).sum()), largest=int(s.max()), posrate=float(gl.mean()))

am, ap = _stats("group_meta"), _stats("group_phash")
print("PSEUDO-PATIENT GROUPING")
print("-" * 92)
print(f"  {'':<30}{'metadata only (USED)' if not cfg.REGROUP_NEG else 'metadata only':>28}"
      f"{'+ dHash subdivision (USED)' if cfg.REGROUP_NEG else '+ dHash subdivision':>30}")
for lab, k, fm in [("case groups", "groups", "{:d}"), ("images per group", "per", "{:.2f}"),
                   ("singleton groups", "singl", "{:d}"), ("largest group", "largest", "{:d}"),
                   ("group-level positive rate", "posrate", "{:.4f}")]:
    print(f"  {lab:<30}{fm.format(am[k]):>28}{fm.format(ap[k]):>30}")
print("-" * 92)
print(f"  image-level positive rate : {df[cfg.TARGET].mean():.4f}   <- the reference")
print(f"  grouping in use           : "
      f"{'perceptual-hash subdivision (SENSITIVITY)' if cfg.REGROUP_NEG else 'metadata only (leakage-safe default)'}")
print()
print("  FINDING F6, stated rather than silently corrected:")
print(f"    Every no-tumour row has all {len(BONE)} bone-site columns at zero, so their group")
print("    key collapses to (centre, age, sex). Negatives over-merge, which is why the")
print(f"    group-level positive rate is {am['posrate']:.2f} against an image-level "
      f"{df[cfg.TARGET].mean():.2f}.")
print("    Over-merging puts unrelated patients in one group: the group lands wholly in")
print("    one split, so NOTHING LEAKS - it only costs effective sample size.")
print("    Subdividing would put the same patient in two groups, which CAN leak. The")
print(f"    dHash variant moves the positive rate to {ap['posrate']:.2f}, i.e. FURTHER from")
print("    the image-level reference, because it splits real multi-view cases apart while")
print("    leaving the unrelated strangers merged. That is why it is off by default.")

viol = df.groupby("group_id")[cfg.TARGET].nunique()
assert int((viol > 1).sum()) == 0, "impure groups - the grouping key is wrong, do not proceed"
print("\n  PASS - every case group is label-pure, so group-level splitting is valid.")
print("-" * 92)

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
sizes.value_counts().sort_index().head(14).plot(kind="bar", ax=ax[0])
ax[0].set_title("case-group size distribution (in use)"); ax[0].set_xlabel("images in group")
pd.DataFrame({"metadata only": df.groupby("group_meta").size().value_counts().sort_index(),
              "+ dHash":       df.groupby("group_phash").size().value_counts().sort_index()}
             ).head(12).plot(kind="bar", ax=ax[1])
ax[1].set_title("group sizes: the two variants")
pd.Series({"image level": df[cfg.TARGET].mean(),
           "metadata only": am["posrate"],
           "+ dHash": ap["posrate"]}).plot(kind="bar", ax=ax[2], color="steelblue")
ax[2].axhline(df[cfg.TARGET].mean(), ls="--", color="k"); ax[2].set_ylim(0, 1)
ax[2].set_title("positive rate by unit of analysis")
for a in ax: a.tick_params(axis="x", rotation=0)
fig.tight_layout(); savefig(fig, "03_grouping")

savetab(pd.DataFrame([{"variant": "metadata only (default)", "used": not cfg.REGROUP_NEG, **am},
                      {"variant": "metadata + dHash subdivision", "used": cfg.REGROUP_NEG, **ap}]),
        "grouping_summary")


PSEUDO-PATIENT GROUPING
--------------------------------------------------------------------------------------------
                                        metadata only (USED)           + dHash subdivision
  case groups                                           1326                          2758
  images per group                                      2.83                          1.36
  singleton groups                                       472                          2161
  largest group                                           27                            11
  group-level positive rate                           0.7104                        0.3416
--------------------------------------------------------------------------------------------
  image-level positive rate : 0.4984   <- the reference
  grouping in use           : metadata only (leakage-safe default)

  FINDING F6, stated rather than silently corrected:
    Every no-tumour row has all 9 bone-site columns at zero, so the

In [12]:
# ============================================================================
# CELL 12 - THE TWO SPLIT PROTOCOLS
# ============================================================================
# Protocol A (headline): groups are shuffled and allocated per class, so the label
#   ratio is preserved and no case can appear in two splits.
# Protocol B (official BTXRD): images shuffled directly, exactly as the dataset paper
#   does. Kept so published BTXRD numbers are comparable and so Section 17 can MEASURE
#   the leakage premium instead of asserting it.

def group_split(frame, target, ratios=None, seed=None, gcol="group_id"):
    ratios = ratios or cfg.SPLIT
    seed   = cfg.SEED if seed is None else seed
    g   = frame.groupby(gcol)[target].first().reset_index()
    rng = np.random.default_rng(seed)
    assign = {}
    for lab in sorted(g[target].unique()):
        ids = g.loc[g[target] == lab, gcol].to_numpy().copy()
        rng.shuffle(ids)
        n = len(ids); n_tr = int(round(ratios[0]*n)); n_va = int(round(ratios[1]*n))
        for i in ids[:n_tr]:          assign[i] = "train"
        for i in ids[n_tr:n_tr+n_va]: assign[i] = "val"
        for i in ids[n_tr+n_va:]:     assign[i] = "test"
    return frame[gcol].map(assign)

# ---- MD5 duplicate scan MOVED UP: byte-identical files must be merged into the
#      same case group BEFORE the split, else a re-uploaded/duplicate scan whose
#      metadata hashed into a different group can leak across splits - this is
#      exactly what the assert at the bottom of this cell was catching. -----------
_md5 = STORE.get_json("obj/md5.json")
if _md5 is None:
    with Timer("MD5 duplicate scan"):
        _md5 = {}
        for p in df.path:
            _md5[p] = hashlib.md5(Path(p).read_bytes()).hexdigest()
    STORE.put_json("obj/md5.json", _md5)
df["md5"] = df.path.map(_md5)

class _DSU2:
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

_dsu2 = _DSU2(N)
for _, idx in df.groupby("group_id").indices.items():
    for j in idx[1:]:
        _dsu2.union(int(idx[0]), int(j))
_n_merges = 0
for _, idx in df.groupby("md5").indices.items():
    idx = np.asarray(idx)
    if len(idx) < 2:
        continue
    roots = {_dsu2.find(int(i)) for i in idx}
    if len(roots) > 1:
        _n_merges += len(roots) - 1
    for j in idx[1:]:
        _dsu2.union(int(idx[0]), int(j))
df["group_id"] = pd.factorize(np.array([_dsu2.find(i) for i in range(N)]))[0]
print(f"MD5 pre-merge: {_n_merges} case group(s) folded together because they "
      f"share a byte-identical image file.")

df["split"] = group_split(df, cfg.TARGET)

rows = []
for s in ["train", "val", "test"]:
    d = df[df.split == s]
    rows.append({"split": s, "images": len(d), "pct": round(100*len(d)/len(df), 1),
                 "groups": d.group_id.nunique(),
                 "no_tumor": int((d[cfg.TARGET] == 0).sum()),
                 "tumor": int((d[cfg.TARGET] == 1).sum()),
                 "pos_rate": round(float(d[cfg.TARGET].mean()), 4),
                 "annotated": int(d.has_annot.sum())})
split_summary = pd.DataFrame(rows)
print("SPLIT SUMMARY  (protocol A - case level)")
print("-" * 92)
print(split_summary.to_string(index=False))
print("-" * 92)
savetab(split_summary, "split_summary")

# ---- protocol B ------------------------------------------------------------
rng = np.random.default_rng(cfg.SEED)
img_assign = pd.Series(index=df.index, dtype=object)
for lab in sorted(df[cfg.TARGET].unique()):
    idx = df.index[df[cfg.TARGET] == lab].to_numpy().copy()
    rng.shuffle(idx)
    n = len(idx); n_tr = int(round(0.70*n)); n_va = int(round(0.15*n))
    img_assign[idx[:n_tr]]          = "train"
    img_assign[idx[n_tr:n_tr+n_va]] = "val"
    img_assign[idx[n_tr+n_va:]]     = "test"
df["split_img"] = img_assign

print("\nLEAKAGE PRESENT IN PROTOCOL B (measured, not asserted)")
print("-" * 92)
for a, b in [("train","val"), ("train","test"), ("val","test")]:
    A = set(df.loc[df.split_img == a, "group_id"]); B = set(df.loc[df.split_img == b, "group_id"])
    print(f"  case groups shared between {a:5s} and {b:5s} : {len(A & B)}")
spanning = df.groupby("group_id")["split_img"].nunique()
n_span   = int((spanning > 1).sum())
img_span = int(df.group_id.isin(spanning[spanning > 1].index).sum())
print(f"  case groups spanning more than one split : {n_span}")
print(f"  images in such groups                    : {img_span} "
      f"({100*img_span/len(df):.1f}% of the dataset)")
print("-" * 92)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
split_summary.set_index("split")[["no_tumor","tumor"]].plot(kind="bar", stacked=True, ax=ax[0])
ax[0].set_title("protocol A: images per split by class"); ax[0].tick_params(axis="x", rotation=0)
split_summary.set_index("split")["groups"].plot(kind="bar", ax=ax[1], color="grey")
ax[1].set_title("protocol A: case groups per split"); ax[1].tick_params(axis="x", rotation=0)
fig.tight_layout(); savefig(fig, "04_split")

# ---- leakage audit on protocol A -------------------------------------------
print("\nLEAKAGE AUDIT (protocol A)")
print("-" * 92)
ok = True
for a, b in [("train","val"), ("train","test"), ("val","test")]:
    ov = set(df.loc[df.split == a, "group_id"]) & set(df.loc[df.split == b, "group_id"])
    print(f"  group overlap {a:5s} / {b:5s} : {len(ov)}")
    ok &= (len(ov) == 0)
assert ok, "case groups leak across protocol-A splits"

# Near-duplicate scan. Sorting by the packed hash puts similar images adjacent, so a
# sliding window finds almost every near-duplicate pair at O(N*w) instead of O(N^2).
_sp    = df.split.values
_order = np.lexsort(HASHES.T[::-1])
_near_pairs = []
for a in range(N):
    ia = _order[a]
    for b in range(a + 1, min(a + 40, N)):
        ib = _order[b]
        if _sp[ia] != _sp[ib] and hamming(HASHES[ia], HASHES[ib]) <= 3:
            _near_pairs.append((int(ia), int(ib)))
print(f"  near-duplicate image pairs across splits (dHash <= 3) : {len(_near_pairs)}")
if _near_pairs:
    print("  examples (these survived the group split, so they are different cases that")
    print("  merely look alike - reported for transparency):")
    for ia, ib in _near_pairs[:5]:
        print(f"    {df.stem.iloc[ia]} [{_sp[ia]}, y={df[cfg.TARGET].iloc[ia]}]  ~  "
              f"{df.stem.iloc[ib]} [{_sp[ib]}, y={df[cfg.TARGET].iloc[ib]}]")

_dupx = int(df.groupby("md5")["split"].nunique().gt(1).sum())
print(f"  byte-identical files spanning protocol-A splits : {_dupx}")
assert _dupx == 0, "identical files in two splits - the grouping missed a duplicate"
print("  PASS - no case group and no identical file appears in two protocol-A splits.")
print("-" * 92)


  [MD5 duplicate scan] took 3.4s
MD5 pre-merge: 8 case group(s) folded together because they share a byte-identical image file.
SPLIT SUMMARY  (protocol A - case level)
--------------------------------------------------------------------------------------------
split  images  pct  groups  no_tumor  tumor  pos_rate  annotated
train    2685 71.7     923      1354   1331    0.4957          0
  val     541 14.4     198       266    275    0.5083          0
 test     520 13.9     197       259    261    0.5019          0
--------------------------------------------------------------------------------------------
  table saved  -> /kaggle/working/outputs/tables/split_summary.csv

LEAKAGE PRESENT IN PROTOCOL B (measured, not asserted)
--------------------------------------------------------------------------------------------
  case groups shared between train and val   : 330
  case groups shared between train and test  : 319
  case groups shared between val   and test  : 159
  case groups sp

## Section 4 — Dataset and augmentation

Augmentation is applied **after** the split and **only** to the training split. Validation and
test go through the deterministic path.

Each sample now returns **two views through one shared backbone** — the whole radiograph and the
label-free bone ROI — plus the mask target and the multi-task label blocks. Geometry is applied
to the image and the mask with the **same random parameters**, so the segmentation target stays
registered to the pixels; photometry touches the image only.

No vertical flip: limb radiographs have a fixed proximal–distal orientation, so a vertical flip
produces an anatomically impossible image.

### Two augmentations that were removed (finding F4)

- **CutMix is off.** It pastes a box whose label weight is its *area*. On a radiograph where the
  lesion occupies a few percent of the frame, a box can excise the lesion entirely while the
  label stays ~0.9 "tumour" — that is injected label noise, not regularisation.
- **Random erasing is capped at one patch of at most `s/8`, filled with noise at the local mean**
  rather than 1–3 patches up to ~115 px of constant grey. A constant rectangle is an edge no
  radiograph contains, and at that size it can bury the finding.


In [13]:
# ============================================================================
# CELL 13 - NORMALISATION AND THE DUAL-VIEW DATASET
# ============================================================================
# v9.3: HARD AUGMENTATION added:
#   · CLAHE (Contrast Limited Adaptive Histogram Equalisation) — X-ray standard
#   · Elastic deformation  — non-rigid image warping via random displacement fields
#   · Perspective transform — simulates off-axis acquisition angle
#   · Much more aggressive random-resized crop (30-100% area at full strength)
#   · Grid shuffle           — randomly permutes local patches, destroys texture
#   · Solarize               — inverts pixels above a random threshold
#   · Motion blur            — simulates patient/tube motion
#   · Stronger noise: Gaussian + Poisson (simulates X-ray quantum noise)
#   · Consistency regularisation: __getitem__ returns a second independent view
#     (key 'x_full2') that the training loop uses for a KL-divergence loss

_ns = STORE.get_json("obj/normstats.json")
if _ns is None:
    tr_idx = df.index[df.split == "train"].to_numpy()
    sel = np.random.default_rng(cfg.SEED).choice(tr_idx, min(800, len(tr_idx)), replace=False)
    vals = np.stack([IMGS[i] for i in sel]).astype(np.float32) / 255.0
    _ns  = {"mean": float(vals.mean()), "std": float(vals.std()), "n": int(len(sel))}
    STORE.put_json("obj/normstats.json", _ns); del vals
DS_MEAN, DS_STD = _ns["mean"], _ns["std"]
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def current_norm():
    """Resolved at call time so an ablation that flips IMAGENET_NORM really changes it."""
    if cfg.IMAGENET_NORM and cfg.PRETRAINED:
        return IMAGENET_MEAN, IMAGENET_STD
    return (np.full(3, DS_MEAN, dtype=np.float32), np.full(3, DS_STD, dtype=np.float32))

# ---- the multi-task label matrix (targets only, never inputs) ---------------
MT_ORDER = [k for k in ["view", "region", "site", "joint", "malig"] if k in MT_BLOCKS]
MT_SPEC  = [(k, MT_BLOCKS[k]["cols"], MT_BLOCKS[k]["mask_to"]) for k in MT_ORDER]
MT_DIM   = sum(len(c) for _, c, _ in MT_SPEC)
MT_COLS  = [c for _, cc, _ in MT_SPEC for c in cc]
# per-column: which target class the loss is restricted to (-1 = all images)
MT_MASKTO = np.concatenate([np.full(len(c), -1 if m is None else m, np.int64)
                            for _, c, m in MT_SPEC]) if MT_DIM else np.zeros(0, np.int64)


class BTDataset(torch.utils.data.Dataset):
    """
    Returns dict(x_full, x_full2, x_roi, y, seg, mt, mt_valid).
      x_full  : whole radiograph, IMG_SIZE  (first augmented view)
      x_full2 : second INDEPENDENTLY augmented view of the SAME image;
                used for the consistency (KL) loss in the training cell.
                Returned only when self.train is True and CONSIST_W > 0.
      x_roi   : label-free bone-ROI crop, IMG_SIZE
      seg     : tumour mask at IMG_SIZE//8, or -1 everywhere when unannotated
      mt      : multi-task targets;  mt_valid : per-column loss mask
    """
    def __init__(self, frame, target=None, train=False, size=None, strength=None,
                 roi=None, seg=None):
        self.idx      = frame.cache_idx.values.astype(np.int64)
        self.labels   = frame[target or cfg.TARGET].values.astype(np.float32)
        self.annot    = frame.has_annot.values.astype(bool)
        self.mt       = (frame[MT_COLS].values.astype(np.float32)
                         if MT_DIM else np.zeros((len(frame), 0), np.float32))
        self.train    = train
        self.size     = size or cfg.IMG_SIZE
        self.strength = cfg.AUG_STRENGTH if strength is None else strength
        self.roi      = cfg.USE_ROI_VIEW if roi is None else roi
        self.seg      = cfg.USE_SEG_HEAD if seg is None else seg
        self.m, self.sd = current_norm()
        self.consist  = train and (getattr(cfg, "CONSIST_W", 0.0) > 0.0)

    def __len__(self): return len(self.idx)

    # ------------------------------------------------------------------
    # v9.3 NEW HELPERS
    # ------------------------------------------------------------------

    @staticmethod
    def _clahe(img, clip_lo=1.5, clip_hi=4.0):
        """
        CLAHE (Contrast Limited Adaptive Histogram Equalisation).
        Standard preprocessing for X-ray images — enhances local contrast
        while suppressing noise amplification.  Works on uint8 grayscale.
        """
        clip = random.uniform(clip_lo, clip_hi)
        clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
        return clahe.apply(img)

    @staticmethod
    def _elastic(img, msk, alpha_lo=0.5, alpha_hi=1.8):
        """
        Elastic deformation via random Gaussian-smoothed displacement fields.
        Mimics the soft-tissue deformation that occurs across different
        acquisitions of the same patient, preventing texture memorisation.

        Speed trick: generate the random field at half resolution (or 1/4 for
        large images) with a fixed-size kernel, then upsample — identical
        smoothness, O(4x) lower cost than a proportional-sigma auto-kernel.
        """
        H, W = img.shape
        alpha = H * random.uniform(alpha_lo, alpha_hi)
        # generate at half resolution for speed
        sh, sw = max(H // 2, 16), max(W // 2, 16)
        dx_sm = cv2.GaussianBlur(
            (np.random.rand(sh, sw) * 2 - 1).astype(np.float32), (15, 15), 0)
        dy_sm = cv2.GaussianBlur(
            (np.random.rand(sh, sw) * 2 - 1).astype(np.float32), (15, 15), 0)
        dx = cv2.resize(dx_sm, (W, H), interpolation=cv2.INTER_LINEAR) * alpha
        dy = cv2.resize(dy_sm, (W, H), interpolation=cv2.INTER_LINEAR) * alpha
        xs = np.clip((np.arange(W)[None, :] + dx).astype(np.float32), 0, W - 1)
        ys = np.clip((np.arange(H)[:, None] + dy).astype(np.float32), 0, H - 1)
        img_out = cv2.remap(img, xs, ys, cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_REFLECT101)
        msk_out = (cv2.remap(msk.astype(np.uint8), xs, ys, cv2.INTER_NEAREST,
                             borderMode=cv2.BORDER_CONSTANT, borderValue=0)
                   if msk is not None else None)
        return img_out, msk_out

    @staticmethod
    def _perspective(img, msk, strength):
        """
        Random perspective warp — simulates off-axis beam angle.
        Each corner is displaced by up to 10%*strength of the image dimension.
        """
        H, W = img.shape
        d = max(2, int(H * 0.10 * strength))
        pts1 = np.float32([[0, 0], [W, 0], [0, H], [W, H]])
        pts2 = np.float32([
            [random.randint(0, d),     random.randint(0, d)],
            [W - random.randint(0, d), random.randint(0, d)],
            [random.randint(0, d),     H - random.randint(0, d)],
            [W - random.randint(0, d), H - random.randint(0, d)],
        ])
        M = cv2.getPerspectiveTransform(pts1, pts2)
        img_out = cv2.warpPerspective(img, M, (W, H),
                                      borderMode=cv2.BORDER_REFLECT101)
        msk_out = (cv2.warpPerspective(msk.astype(np.uint8), M, (W, H),
                                        flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT, borderValue=0)
                   if msk is not None else None)
        return img_out, msk_out

    @staticmethod
    def _grid_shuffle(img, grid=4):
        """
        Randomly permutes non-overlapping square grid patches.
        Destroys global texture patterns the head could memorise, while
        preserving local feature statistics the backbone learns from.
        """
        H, W = img.shape
        ph, pw = H // grid, W // grid
        patches = []
        for r in range(grid):
            for c in range(grid):
                patches.append(img[r*ph:(r+1)*ph, c*pw:(c+1)*pw].copy())
        random.shuffle(patches)
        out = img.copy()
        for idx, (r, c) in enumerate((r, c) for r in range(grid) for c in range(grid)):
            out[r*ph:(r+1)*ph, c*pw:(c+1)*pw] = patches[idx]
        return out

    # -- geometry, applied to image AND mask with the same parameters ---------
    def _geom(self, img, msk, box=None):
        s, st = self.size, (self.strength if self.train else 0.0)
        if msk is not None and msk.shape != img.shape:
            msk = cv2.resize(msk, (img.shape[1], img.shape[0]),
                             interpolation=cv2.INTER_NEAREST)
        if box is not None:
            x1, y1, x2, y2 = box
            img = img[y1:y2, x1:x2]
            if msk is not None:
                msk = msk[y1:y2, x1:x2]

        if st <= 0:
            img = cv2.resize(img, (s, s), interpolation=cv2.INTER_AREA)
            if msk is not None:
                msk = cv2.resize(msk, (s, s), interpolation=cv2.INTER_NEAREST)
            return img, msk

        # v9.3: CLAHE before any geometric distortion so intensity-dependent crops
        # don't create a systematic pattern the head can recognise.
        img = self._clahe(img)

        # ---- elastic deformation (40 % prob) --------------------------------
        if random.random() < 0.40:
            img, msk = self._elastic(img, msk)

        # ---- perspective warp (35 % prob) -----------------------------------
        if random.random() < 0.35:
            img, msk = self._perspective(img, msk, st)

        # ---- aggressive random-resized crop ----------------------------------
        # v9.3: scale floor drops from 0.55 to 0.30 at st=2.0, giving 3x more
        # spatial variety.  Wider aspect ratio range also.
        H0, W0 = img.shape[0], img.shape[1]
        lo    = max(0.30, 1.0 - 0.35 * st)        # 0.30 at st=2, was max(0.55,...)
        area  = random.uniform(lo, 1.0)
        ratio = random.uniform(0.75, 1.40)         # was 0.88-1.14
        h = max(8, min(int(round(math.sqrt(area / ratio) * H0)), H0))
        w = max(8, min(int(round(math.sqrt(area * ratio) * W0)), W0))
        yy = random.randint(0, H0 - h); xx = random.randint(0, W0 - w)
        img = cv2.resize(img[yy:yy+h, xx:xx+w], (s, s), interpolation=cv2.INTER_LINEAR)
        if msk is not None:
            msk = cv2.resize(msk[yy:yy+h, xx:xx+w], (s, s),
                             interpolation=cv2.INTER_NEAREST)

        # ---- flips ----------------------------------------------------------
        if random.random() < 0.50:                # horizontal
            img = cv2.flip(img, 1)
            if msk is not None: msk = cv2.flip(msk, 1)
        if random.random() < 0.20:                # vertical (less common in X-ray)
            img = cv2.flip(img, 0)
            if msk is not None: msk = cv2.flip(msk, 0)

        # ---- affine (rotation + scale + translate) --------------------------
        if random.random() < 0.75:
            ang = 25 * st                         # was 15*st
            sc  = random.uniform(1 - 0.12*st, 1 + 0.12*st)
            M = cv2.getRotationMatrix2D((s/2, s/2), random.uniform(-ang, ang), sc)
            M[0, 2] += random.uniform(-.07, .07) * st * s
            M[1, 2] += random.uniform(-.07, .07) * st * s
            img = cv2.warpAffine(img, M, (s, s), borderMode=cv2.BORDER_REFLECT101)
            if msk is not None:
                msk = cv2.warpAffine(msk, M, (s, s), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        return img, msk

    # -- photometry, image only ----------------------------------------------
    def _photo(self, img):
        st, s = self.strength, self.size
        if st <= 0:
            return img

        # ---- brightness / contrast -----------------------------------------
        if random.random() < 0.60:
            img = np.clip(img.astype(np.float32)
                          * random.uniform(1 - 0.22*st, 1 + 0.22*st)
                          + random.uniform(-25*st, 25*st), 0, 255).astype(np.uint8)

        # ---- gamma (wide range, simulates different kVp / detector response) ----
        if random.random() < 0.50:
            g   = random.uniform(max(0.3, 1.0 - 0.70*st), min(3.0, 1.0 + 0.70*st))
            lut = np.clip(((np.arange(256)/255.0) ** g) * 255.0, 0, 255).astype(np.uint8)
            img = cv2.LUT(img, lut)

        # ---- solarize: invert pixels above a random threshold ---------------
        # Models tend to memorise absolute intensity signatures; solarize breaks
        # that by randomly inverting the bright end of the histogram.
        if random.random() < 0.20 * st:
            thr = random.randint(80, 200)
            mask_sol = img > thr
            img = img.copy()
            img[mask_sol] = 255 - img[mask_sol]

        # ---- blur or sharpen -----------------------------------------------
        if random.random() < 0.40 * st:
            r = random.random()
            if r < 0.40:                          # Gaussian blur
                img = cv2.GaussianBlur(img, (0, 0), random.uniform(0.5, 2.0 * st))
            elif r < 0.70:                        # motion blur (horizontal or vertical)
                ksize = random.choice([3, 5, 7, 9])
                axis  = random.choice([0, 1])     # 0=horizontal, 1=vertical
                k = np.zeros((ksize, ksize), dtype=np.float32)
                k[ksize//2, :] = 1.0/ksize if axis == 0 else 0
                k[:, ksize//2] = 1.0/ksize if axis == 1 else 0
                if axis == 0:
                    k[ksize//2, :] = 1.0/ksize
                    k[:, ksize//2] = 0
                else:
                    k[:, ksize//2] = 1.0/ksize
                    k[ksize//2, :] = 0
                img = np.clip(cv2.filter2D(img.astype(np.float32), -1, k), 0, 255).astype(np.uint8)
            else:                                 # sharpen
                k = np.array([[0,-1,0], [-1,5,-1], [0,-1,0]], dtype=np.float32)
                img = np.clip(cv2.filter2D(img.astype(np.float32), -1, k), 0, 255).astype(np.uint8)

        # ---- Gaussian noise + Poisson (X-ray quantum noise) -----------------
        if random.random() < 0.50 * st:
            sigma = random.uniform(3, 15) * st
            noise = np.random.normal(0, sigma, img.shape)
            # Optional Poisson component (scales with local intensity)
            if random.random() < 0.40:
                lam = img.astype(np.float32) / 10.0
                noise += (np.random.poisson(lam) - lam) * random.uniform(0.5, 1.5)
            img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

        # ---- salt-and-pepper ------------------------------------------------
        if random.random() < 0.20 * st:
            frac = random.uniform(0.001, 0.005)
            mask_s = np.random.rand(*img.shape) < frac / 2
            mask_p = np.random.rand(*img.shape) < frac / 2
            img = img.copy()
            img[mask_s] = 0
            img[mask_p] = 255

        # ---- random erasing: several noise-filled patches -------------------
        if random.random() < 0.45 * st and cfg.ERASE_MAX_N > 0:
            img = img.copy()
            for _ in range(random.randint(1, cfg.ERASE_MAX_N)):
                kh = random.randint(s//20, max(s//20+1, int(s*cfg.ERASE_MAX_FRAC)))
                kw = random.randint(s//20, max(s//20+1, int(s*cfg.ERASE_MAX_FRAC)))
                y  = random.randint(0, max(0, s-kh))
                x  = random.randint(0, max(0, s-kw))
                patch = img[y:y+kh, x:x+kw]
                img[y:y+kh, x:x+kw] = np.clip(
                    np.random.normal(float(patch.mean()), max(float(patch.std()), 6.0),
                                     patch.shape), 0, 255).astype(np.uint8)

        # ---- grid shuffle (25 % prob at full strength) ----------------------
        # Randomly permutes 4x4 non-overlapping patches.  Breaks spatial layouts
        # the model might memorise while keeping local features intact.
        if random.random() < 0.25 * st:
            img = self._grid_shuffle(img, grid=4)

        return img

    def _to_tensor(self, img):
        x = np.repeat(img.astype(np.float32)[None, ...] / 255.0, 3, axis=0)
        x = (x - self.m[:, None, None]) / self.sd[:, None, None]
        return torch.from_numpy(np.ascontiguousarray(x))

    def _augment_one(self, base, msk, want_seg, box=None):
        """Apply one full augmented view (geom + photo) to base image."""
        img_f, msk_f = self._geom(base, msk, box)
        img_f = self._photo(img_f) if self.train else img_f
        return img_f, msk_f

    def __getitem__(self, i):
        ci   = int(self.idx[i])
        base = np.asarray(IMGS[ci])
        want_seg = self.seg and bool(self.annot[i])
        msk  = np.asarray(MASKS[ci]) if want_seg else None

        # --- first view (always produced) ------------------------------------
        img_f, msk_f = self._augment_one(base, msk, want_seg, box=None)
        out = {"x_full": self._to_tensor(img_f),
               "y": torch.tensor(self.labels[i])}

        # --- second independent view (consistency regularisation) -----------
        if self.consist:
            img_f2, _ = self._augment_one(base, None, False, box=None)
            out["x_full2"] = self._to_tensor(img_f2)

        # --- ROI view --------------------------------------------------------
        if self.roi:
            box = tuple(int(v) for v in ROIBOX[ci])
            img_r, _ = self._augment_one(base, None, False, box)
            out["x_roi"] = self._to_tensor(img_r)
        else:
            out["x_roi"] = out["x_full"]

        # --- segmentation mask -----------------------------------------------
        g = max(self.size // 8, 8)
        if self.seg:
            if msk_f is not None:
                sm = cv2.resize(msk_f.astype(np.uint8), (g, g),
                                interpolation=cv2.INTER_AREA).astype(np.float32)
                out["seg"] = torch.from_numpy(sm[None, ...])
                out["seg_valid"] = torch.tensor(1.0)
            elif self.labels[i] == 0:
                out["seg"] = torch.zeros(1, g, g)
                out["seg_valid"] = torch.tensor(1.0)
            else:
                out["seg"] = torch.zeros(1, g, g)
                out["seg_valid"] = torch.tensor(0.0)
        else:
            out["seg"] = torch.zeros(1, 1, 1); out["seg_valid"] = torch.tensor(0.0)

        # --- multi-task targets ----------------------------------------------
        if MT_DIM:
            out["mt"] = torch.from_numpy(self.mt[i])
            keep = (MT_MASKTO < 0) | (MT_MASKTO == int(self.labels[i]))
            out["mt_valid"] = torch.from_numpy(keep.astype(np.float32))
        else:
            out["mt"] = torch.zeros(0); out["mt_valid"] = torch.zeros(0)
        return out


def make_loader(frame, train=False, target=None, bs=None, size=None, strength=None,
                shuffle=None, workers=None, roi=None, seg=None):
    ds = BTDataset(frame, target, train, size, strength, roi, seg)
    return torch.utils.data.DataLoader(
        ds, batch_size=bs or cfg.BATCH, shuffle=train if shuffle is None else shuffle,
        drop_last=bool(train), num_workers=cfg.NUM_WORKERS if workers is None else workers,
        pin_memory=(DEVICE.type == "cuda"), persistent_workers=False)

tr_df = df[df.split == "train"].reset_index(drop=True)
va_df = df[df.split == "val"].reset_index(drop=True)
te_df = df[df.split == "test"].reset_index(drop=True)
if cfg.QUICK_MODE:
    tr_df = tr_df.sample(min(128, len(tr_df)), random_state=0).reset_index(drop=True)
    va_df = va_df.sample(min(64,  len(va_df)), random_state=0).reset_index(drop=True)
    te_df = te_df.sample(min(64,  len(te_df)), random_state=0).reset_index(drop=True)

dl_tr = make_loader(tr_df, train=True)
dl_va = make_loader(va_df)
dl_te = make_loader(te_df)

_b = next(iter(dl_tr))
print("DATA PIPELINE  (v9.3 - hard augmentation)")
print("-" * 78)
print(f"  train / val / test images  : {len(tr_df)} / {len(va_df)} / {len(te_df)}")
print(f"  cache {cfg.CACHE_SIZE} px  ->  model input {cfg.IMG_SIZE} px")
print(f"  x_full  {tuple(_b['x_full'].shape)}   x_roi {tuple(_b['x_roi'].shape)}")
print(f"  x_full2 present  : {'x_full2' in _b}  (consistency regularisation: "
      f"CONSIST_W={getattr(cfg,'CONSIST_W',0.0)})")
print(f"  seg     {tuple(_b['seg'].shape)}   valid in batch {int(_b['seg_valid'].sum())}/{len(_b['y'])}")
print(f"  mt      {tuple(_b['mt'].shape)}  ({MT_DIM} columns: "
      f"{', '.join(f'{k}x{len(c)}' for k, c, _ in MT_SPEC) if MT_DIM else 'none'})")
print(f"  batch mean/std : {_b['x_full'].mean():+.4f} / {_b['x_full'].std():.4f}")
print(f"  steps per epoch: {len(dl_tr)}  (effective batch {cfg.BATCH*cfg.ACCUM})")
print(f"  hard aug ops   : CLAHE | elastic | perspective | crop≥30% | grid-shuffle")
print(f"                 | solarize | motion-blur | strong-noise | salt&pepper")
print("-" * 78)


DATA PIPELINE  (v9.3 - hard augmentation)
------------------------------------------------------------------------------
  train / val / test images  : 2685 / 541 / 520
  cache 448 px  ->  model input 384 px
  x_full  (8, 3, 384, 384)   x_roi (8, 3, 384, 384)
  x_full2 present  : False  (consistency regularisation: CONSIST_W=0.0)
  seg     (8, 1, 1, 1)   valid in batch 0/8
  mt      (8, 23)  (23 columns: viewx3, regionx3, sitex9, jointx6, maligx2)
  batch mean/std : -0.2764 / 1.0345
  steps per epoch: 335  (effective batch 32)
  hard aug ops   : CLAHE | elastic | perspective | crop≥30% | grid-shuffle
                 | solarize | motion-blur | strong-noise | salt&pepper
------------------------------------------------------------------------------


In [14]:
# ============================================================================
# CELL 14 - AUGMENTATION PREVIEW  (image and mask must stay registered)
# ============================================================================
_row = tr_df[tr_df.has_annot].head(1) if tr_df.has_annot.any() else tr_df.head(1)
_ds  = BTDataset(_row, train=True)
_m, _sd = current_norm()

def _show(t):
    im = t[0].numpy() * _sd[0] + _m[0]
    return np.clip(im, 0, 1)

fig, ax = plt.subplots(3, 5, figsize=(16, 9.6))
_base = cv2.resize(np.asarray(IMGS[int(_row.cache_idx.iloc[0])]),
                   (cfg.IMG_SIZE, cfg.IMG_SIZE), interpolation=cv2.INTER_AREA)
ax[0,0].imshow(_base, cmap="gray"); ax[0,0].set_title("no augmentation", fontweight="bold")
set_seed(0)
for i in range(1, 15):
    r, c = divmod(i, 5)
    d = _ds[0]
    if r == 0 or (r == 1 and c < 5):
        ax[r,c].imshow(_show(d["x_full"]), cmap="gray")
        if float(d["seg_valid"]) > 0 and d["seg"].max() > 0:
            g = d["seg"][0].numpy()
            ax[r,c].contour(cv2.resize(g, (cfg.IMG_SIZE, cfg.IMG_SIZE)),
                            levels=[0.5], colors="lime", linewidths=1.3)
        ax[r,c].set_title(f"full view #{i}", fontsize=9)
    else:
        ax[r,c].imshow(_show(d["x_roi"]), cmap="gray")
        ax[r,c].set_title(f"ROI view #{i}", fontsize=9)
for a in ax.ravel(): a.axis("off")
fig.suptitle("Training-time augmentation - mask (green) follows the image geometry",
             fontsize=13, fontweight="bold")
fig.tight_layout(); savefig(fig, "05_augmentation")
set_seed()

print(f"  AUG_STRENGTH  : {cfg.AUG_STRENGTH}")
print("  geometry (image + mask, shared params) : random-resized crop 0.55-1.00 |")
print("      hflip 0.50 | affine +/-15 deg, scale +/-10%, shift 5%")
print("  photometry (image only) : brightness/contrast 0.50 | gamma 0.35 |")
print("      blur-or-sharpen 0.30 | gaussian noise 0.30 |")
print(f"      erasing 0.30, <= {cfg.ERASE_MAX_N} patch of <= s/{int(1/cfg.ERASE_MAX_FRAC)}, noise fill")
print(f"  CutMix        : {'ON' if cfg.CUTMIX_ALPHA > 0 else 'OFF (finding F4)'}")
print("  val / test    : CLAHE + resize + normalise only, deterministic")


  figure saved -> /kaggle/working/outputs/figures/05_augmentation.png
  AUG_STRENGTH  : 0.6
  geometry (image + mask, shared params) : random-resized crop 0.55-1.00 |
      hflip 0.50 | affine +/-15 deg, scale +/-10%, shift 5%
  photometry (image only) : brightness/contrast 0.50 | gamma 0.35 |
      blur-or-sharpen 0.30 | gaussian noise 0.30 |
      erasing 0.30, <= 0 patch of <= s/6, noise fill
  CutMix        : OFF (finding F4)
  val / test    : CLAHE + resize + normalise only, deterministic


## Section 5 — The regularisation suite, corrected

Regularisation returns to the setting that **converged** (v5) and goes no further. The three
escalations that followed it cost 12 points of validation accuracy and bought 2 points of gap.

| Mechanism | v8 | v9 | Why |
|---|---|---|---|
| Mixup | α 0.25, p 0.50 | α 0.20, p 0.40 | mild; the seg head now supplies the localisation pressure |
| **CutMix** | α 1.00 | **off** | F4 — an area-weighted box can excise the lesion and keep the label |
| Random erasing | 1–3 patches, const fill | ≤1 patch ≤ s/8, noise fill | F4 |
| Label smoothing | 0.05 | 0.05 | unchanged |
| Dropout / DropPath / feature dropout | 0.30 / 0.10 / 0.10 | same | unchanged |
| Weight decay | 3e-2 | 2e-2 | decoupled, never on norms or biases |
| **EMA decay warmup** | `0.999·(1−e^(−s/2000))` | `min(0.999, (1+s)/(10+s))` | **F5** |
| Backbone freeze | 0.20 permanent, 4 warm epochs | 0.20, 3 | unchanged |
| **Backbone LR** | 1e-4 × 0.05 = **5e-6** | 3e-4 × 0.15 = **4.5e-5** | **F2** |

### F5 in detail — the EMA was averaging nothing

`ema.update()` runs on optimiser steps, not batches: 330 batches ÷ `ACCUM` 4 = **82 per epoch**.
After 40 epochs that is ~3,300 steps, where the old rule gives `d ≈ 0.81` — an averaging window
of `1/(1−d) ≈ 5 steps`. Every "EMA" number in v6 and v8 came from weights indistinguishable
from the live model. timm's rule reaches a 100-step window by step 1,000 and a genuine one-to-two
epoch average after that.

**BatchNorm note.** `requires_grad=False` does *not* stop BatchNorm running statistics from being
overwritten on every forward pass, so a "frozen" backbone silently keeps drifting. `train()` is
overridden to force the frozen stages into `eval()`.


In [15]:
# ============================================================================
# CELL 15 - REGULARISATION UTILITIES
# ============================================================================

# ---- mixup (CutMix removed - finding F4) ------------------------------------
def rand_bbox(H, W, lam):
    r = math.sqrt(1.0 - lam)
    ch, cw = int(H * r), int(W * r)
    cy, cx = random.randint(0, H - 1), random.randint(0, W - 1)
    return (max(cy - ch // 2, 0), min(cy + ch // 2, H),
            max(cx - cw // 2, 0), min(cx + cw // 2, W))

def mix_batch(xs, y, cfg_=None):
    """
    Mixes a LIST of view tensors with ONE shared permutation and lambda, so the two
    views of a sample stay paired. Returns (xs, y_a, y_b, lam); lam == 1.0 means no mix.
    """
    c = cfg_ or cfg
    if random.random() > c.MIX_PROB or (c.MIXUP_ALPHA <= 0 and c.CUTMIX_ALPHA <= 0):
        return xs, y, y, 1.0
    perm = torch.randperm(xs[0].size(0), device=xs[0].device)
    if c.CUTMIX_ALPHA > 0 and (c.MIXUP_ALPHA <= 0 or random.random() < 0.5):
        lam = float(np.random.beta(c.CUTMIX_ALPHA, c.CUTMIX_ALPHA))
        y1, y2, x1, x2 = rand_bbox(xs[0].size(2), xs[0].size(3), lam)
        xs = [x.clone() for x in xs]
        for x in xs:
            x[:, :, y1:y2, x1:x2] = x[perm][:, :, y1:y2, x1:x2]
        lam = 1.0 - ((y2 - y1) * (x2 - x1) / (xs[0].size(2) * xs[0].size(3)))
    else:
        lam = float(np.random.beta(c.MIXUP_ALPHA, c.MIXUP_ALPHA))
        xs  = [lam * x + (1.0 - lam) * x[perm] for x in xs]
    return xs, y, y[perm], lam

# ---- losses ------------------------------------------------------------------
def smooth_bce(logit, target, smooth=None, pos_weight=None):
    s = cfg.LABEL_SMOOTH if smooth is None else smooth
    t = target.float() * (1.0 - s) + 0.5 * s
    return F.binary_cross_entropy_with_logits(logit, t, pos_weight=pos_weight)

def mixed_bce(logit, ya, yb, lam, smooth=None, pos_weight=None):
    if lam >= 1.0:
        return smooth_bce(logit, ya, smooth, pos_weight)
    return (lam * smooth_bce(logit, ya, smooth, pos_weight)
            + (1.0 - lam) * smooth_bce(logit, yb, smooth, pos_weight))

def dice_bce(logit, target, valid, eps=1.0):
    """
    Segmentation loss, averaged over the rows whose mask is known.
    `valid` is 0 for unannotated tumours: we do not know where their lesion is, so
    they must not be taught that there is none. Normals carry a genuine all-zero target.
    """
    v = valid.view(-1)
    if float(v.sum()) < 0.5:
        return logit.sum() * 0.0
    l = logit[v > 0].float(); t = target[v > 0].float()
    # The dataset emits the mask at IMG_SIZE//8; the backbone's feature grid is
    # IMG_SIZE//32 for EfficientNetV2-S and differs again for ConvNeXt and DenseNet.
    # Pool the target down to whatever the decoder actually produced, which keeps the
    # loss backbone-agnostic and turns the binary mask into a soft occupancy target.
    if t.shape[-2:] != l.shape[-2:]:
        t = F.adaptive_avg_pool2d(t, l.shape[-2:])
    bce = F.binary_cross_entropy_with_logits(l, t)
    p   = torch.sigmoid(l).flatten(1); g = t.flatten(1)
    dice = 1.0 - ((2 * (p * g).sum(1) + eps) / (p.sum(1) + g.sum(1) + eps)).mean()
    return 0.5 * bce + 0.5 * dice

def masked_bce(logit, target, valid):
    """Multi-task loss over only the (row, column) pairs whose label is meaningful."""
    if logit.numel() == 0 or float(valid.sum()) < 0.5:
        return logit.sum() * 0.0
    l = F.binary_cross_entropy_with_logits(logit.float(), target.float(), reduction="none")
    return (l * valid).sum() / valid.sum().clamp(min=1.0)

# ---- DropPath ----------------------------------------------------------------
class DropPath(nn.Module):
    def __init__(self, p=0.0):
        super().__init__(); self.p = float(p)
    def forward(self, x):
        if self.p <= 0.0 or not self.training:
            return x
        keep  = 1.0 - self.p
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask  = keep + torch.rand(shape, dtype=x.dtype, device=x.device)
        return x * mask.floor_() / keep

# ---- EMA (finding F5) --------------------------------------------------------
def unwrap(m):
    return m.module if isinstance(m, nn.DataParallel) else m

class ModelEMA:
    """
    Exponential moving average of the weights.

    F5: the old warmup was d = base*(1 - exp(-step/tau)) with tau=2000. update() runs on
    OPTIMISER steps - 330 batches / ACCUM 4 = 82 per epoch - so after 40 epochs step is
    only ~3300 and d reaches 0.81, an averaging window of ~5 steps. The EMA averaged
    nothing and every reported number came from the live weights.

    timm's rule instead: d = min(base, (1+step)/(10+step)). Window is ~12 steps at step
    100, ~100 at step 1000, and a genuine 1-2 epoch average thereafter.
    """
    def __init__(self, model, decay=None):
        self.base = cfg.EMA_DECAY if decay is None else decay
        self.step = 0
        self.module = copy.deepcopy(unwrap(model)).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    def decay(self):
        return min(self.base, (1.0 + self.step) / (10.0 + self.step))

    @torch.no_grad()
    def update(self, model):
        self.step += 1
        d = self.decay()
        msd = unwrap(model).state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach().to(v.device), alpha=1.0 - d)
            else:
                v.copy_(msd[k])

# ---- schedule and parameter groups -------------------------------------------
def make_scheduler(opt, epochs, steps_per_epoch, warmup_epochs=None):
    warm  = (cfg.WARMUP_EPOCHS if warmup_epochs is None else warmup_epochs) * steps_per_epoch
    total = max(epochs * steps_per_epoch, warm + 1)
    def fn(step):
        if step < warm:
            return (step + 1) / max(warm, 1)
        prog = (step - warm) / max(total - warm, 1)
        return 0.5 * (1.0 + math.cos(math.pi * min(prog, 1.0))) * (1 - 0.02) + 0.02
    return torch.optim.lr_scheduler.LambdaLR(opt, fn)

def param_groups(model, lr, wd, backbone_mult=1.0):
    m = unwrap(model)
    db, nb, dh, nh = [], [], [], []
    for n, p in m.named_parameters():
        if not p.requires_grad:
            continue
        is_bb = n.startswith("backbone")
        no_wd = (p.ndim <= 1) or n.endswith(".bias") or ("pos_embed" in n) or ("cls_token" in n)
        if is_bb: (nb if no_wd else db).append(p)
        else:     (nh if no_wd else dh).append(p)
    return [
        {"params": dh, "lr": lr,                 "weight_decay": wd},
        {"params": nh, "lr": lr,                 "weight_decay": 0.0},
        {"params": db, "lr": lr * backbone_mult, "weight_decay": wd},
        {"params": nb, "lr": lr * backbone_mult, "weight_decay": 0.0},
    ]

print("REGULARISATION SUITE  (v9 - back to the setting that converged)")
print("-" * 78)
print(f"  mixup alpha {cfg.MIXUP_ALPHA} | cutmix "
      f"{'OFF (F4)' if cfg.CUTMIX_ALPHA <= 0 else cfg.CUTMIX_ALPHA} | p(mix) {cfg.MIX_PROB}")
print(f"  label smoothing {cfg.LABEL_SMOOTH} | dropout {cfg.DROPOUT} | "
      f"drop-path {cfg.DROP_PATH} | feature dropout {cfg.FEAT_DROP}")
print(f"  weight decay {cfg.WD} (never on norms/biases) | aug strength {cfg.AUG_STRENGTH}")
print(f"  EMA base {cfg.EMA_DECAY} with timm warmup min(base, (1+s)/(10+s))   [F5]")
print(f"  freeze {cfg.FREEZE_FRAC:.0%} permanently, whole backbone for {cfg.FREEZE_EPOCHS} epochs")
print(f"  head LR {cfg.LR:g} | backbone LR {cfg.LR*cfg.BACKBONE_LR_MULT:g} "
      f"(x{cfg.BACKBONE_LR_MULT})   [F2: v8 ran {1e-4*0.05:g}]")
print(f"  warmup {cfg.WARMUP_EPOCHS} epochs then cosine to 2% of peak")
print("-" * 78)

_e = ModelEMA(nn.Linear(2, 2))
_w = []
for _s in (100, 1000, 3300, 20000):
    _e.step = _s; _w.append(f"step {_s}: d={_e.decay():.4f} window~{1/(1-_e.decay()):.0f}")
print("  EMA sanity | " + " | ".join(_w))
_x = [torch.rand(4, 3, 16, 16), torch.rand(4, 3, 16, 16)]
_y = torch.tensor([0., 1., 1., 0.])
for _ in range(3):
    _xm, _a, _b_, _lam = mix_batch(_x, _y)
    assert len(_xm) == 2 and _xm[0].shape == _x[0].shape and 0.0 <= _lam <= 1.0
print("  self-test  | mix_batch keeps both views paired, lambda in range   OK")
del _e, _x, _y


REGULARISATION SUITE  (v9 - back to the setting that converged)
------------------------------------------------------------------------------
  mixup alpha 0.1 | cutmix OFF (F4) | p(mix) 0.2
  label smoothing 0.04 | dropout 0.15 | drop-path 0.05 | feature dropout 0.05
  weight decay 0.005 (never on norms/biases) | aug strength 0.6
  EMA base 0.999 with timm warmup min(base, (1+s)/(10+s))   [F5]
  freeze 25% permanently, whole backbone for 2 epochs
  head LR 0.0002 | backbone LR 2e-05 (x0.1)   [F2: v8 ran 5e-06]
  warmup 2 epochs then cosine to 2% of peak
------------------------------------------------------------------------------
  EMA sanity | step 100: d=0.9182 window~12 | step 1000: d=0.9911 window~112 | step 3300: d=0.9973 window~368 | step 20000: d=0.9990 window~1000
  self-test  | mix_batch keeps both views paired, lambda in range   OK


## Section 6 — BT-CAGTNet v9

The skeleton is unchanged — that is the contribution. What is added is *supervision* and *scale*,
which is where the missing AUROC has to come from; more dropout cannot produce it.

```
        full radiograph (384)          label-free bone ROI (384)
                 |                              |
                 +----------- shared backbone --+        <- one set of weights, two views
                 |                              |
            Dropout2d                      Dropout2d
                 |                              |
              CBAM                           CBAM        <- novelty 1
              /   \                          /   \
         GAP-CNN  Transformer           GAP-CNN  Transformer   <- novelty 2
              \     |                       |     /
               +----+---- adaptive gate ----+----+          <- novelty 3, now over 4 branches
                            |
                    fused embedding -> tumour logit
                            |
        aux: per-branch logits | multi-task heads | segmentation decoder
```

**Segmentation decoder.** Two 3×3 convs on the full-view CBAM feature map to a single logit
channel, trained with Dice + BCE against the tumour mask. Normals carry a genuine all-zero
target; unannotated tumours are excluded from the loss rather than taught that there is nothing
there. This is the piece that forces the feature map to localise, and it is free — the labels
already exist.

**Multi-task heads.** One linear head per block (view, region, site, joint, benign/malignant) on
the fused embedding, each loss-masked to the class that actually carries the label so the head
teaches anatomy rather than restating the target.

**The graph branch stays off.** Its gate argmax share was 0.00 on the test split — it never
decided a single image. It is kept only as an ablation row, and the honest description of the
model is CBAM-CNN + transformer with dual-scale gated fusion.


In [16]:
# ============================================================================
# CELL 16 - CBAM, GRAPH, TRANSFORMER AND SEGMENTATION BLOCKS
# ============================================================================

# ---- CBAM (novelty 1) --------------------------------------------------------
class ChannelAttention(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        h = max(ch // r, 8)
        self.mlp = nn.Sequential(nn.Linear(ch, h), nn.ReLU(inplace=True), nn.Linear(h, ch))
    def forward(self, x):
        b, c, _, _ = x.shape
        a = self.mlp(x.mean((2, 3))) + self.mlp(x.amax((2, 3)))
        return x * torch.sigmoid(a).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        s = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)
        return x * torch.sigmoid(self.conv(s))

class CBAM(nn.Module):
    def __init__(self, ch, r=16, k=7):
        super().__init__()
        self.ca, self.sa = ChannelAttention(ch, r), SpatialAttention(k)
        self.last_spatial = None
    def forward(self, x):
        x = self.ca(x)
        s = torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)
        w = torch.sigmoid(self.sa.conv(s))
        # cached for XAI in eval only; holding it during training pins a tensor per module
        self.last_spatial = w.detach() if not self.training else None
        return x * w

# ---- patch graph (ablation only - gate argmax share was 0.00) ----------------
def knn_adjacency(feat, k=8):
    f   = F.normalize(feat.float(), dim=-1)
    sim = torch.bmm(f, f.transpose(1, 2))
    Nn  = sim.size(1); k = max(1, min(k, Nn - 1))
    idx = sim.topk(k + 1, dim=-1).indices
    A   = torch.zeros_like(sim).scatter_(-1, idx, 1.0)
    A   = ((A + A.transpose(1, 2)) > 0).float()
    return A / A.sum(-1, keepdim=True).clamp(min=1)

class GATLayer(nn.Module):
    def __init__(self, din, dout, heads=4, drop=0.3):
        super().__init__()
        assert dout % heads == 0
        self.h, self.d = heads, dout // heads
        self.W = nn.Linear(din, dout, bias=False)
        self.a_src = nn.Parameter(torch.empty(heads, self.d))
        self.a_dst = nn.Parameter(torch.empty(heads, self.d))
        nn.init.xavier_uniform_(self.a_src); nn.init.xavier_uniform_(self.a_dst)
        self.drop = nn.Dropout(drop)
    def forward(self, x, A):
        B, Nn, _ = x.shape
        h   = self.W(x).view(B, Nn, self.h, self.d).permute(0, 2, 1, 3)
        a_s = self.a_src.view(1, self.h, 1, self.d).to(h.dtype)
        a_d = self.a_dst.view(1, self.h, 1, self.d).to(h.dtype)
        e   = F.leaky_relu((h*a_s).sum(-1).unsqueeze(-1) + (h*a_d).sum(-1).unsqueeze(-2), 0.2)
        e   = e.masked_fill(~(A.unsqueeze(1) > 0), torch.finfo(e.dtype).min)
        att = torch.softmax(e, -1)
        out = torch.matmul(self.drop(att), h).permute(0, 2, 1, 3).reshape(B, Nn, self.h*self.d)
        return F.elu(out)

class PatchGraphHead(nn.Module):
    def __init__(self, cin, hid=None, op=None, k=None, drop=None):
        super().__init__()
        hid  = hid or cfg.GRAPH_HID
        k    = k   or cfg.GRAPH_K
        drop = cfg.DROPOUT if drop is None else drop
        self.k, self.out_dim = k, hid * 2
        self.proj = nn.Linear(cin, hid)
        self.l1, self.l2 = GATLayer(hid, hid, 4, drop), GATLayer(hid, hid, 4, drop)
        self.norm, self.drop = nn.LayerNorm(hid), nn.Dropout(drop)
        self.last_A = None
    def forward(self, fmap):
        x = self.proj(fmap.flatten(2).transpose(1, 2))
        A = knn_adjacency(x.detach(), self.k)
        self.last_A = A.detach() if not self.training else None
        x = self.l2(self.norm(self.l1(x, A)), A)
        return self.drop(torch.cat([x.mean(1), x.amax(1)], dim=-1))

# ---- transformer (novelty 2) -------------------------------------------------
class EncoderBlock(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0, drop=0.0, drop_path=0.0):
        super().__init__()
        self.n1   = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.dp1  = DropPath(drop_path)
        self.n2   = nn.LayerNorm(dim)
        self.mlp  = nn.Sequential(nn.Linear(dim, int(dim*mlp_ratio)), nn.GELU(),
                                  nn.Dropout(drop), nn.Linear(int(dim*mlp_ratio), dim),
                                  nn.Dropout(drop))
        self.dp2  = DropPath(drop_path)
        self.last_attention = None
    def forward(self, x):
        h = self.n1(x)
        want = not self.training
        a, w = self.attn(h, h, h, need_weights=want, average_attn_weights=True)
        self.last_attention = w.detach() if (want and w is not None) else None
        x = x + self.dp1(a)
        return x + self.dp2(self.mlp(self.n2(x)))

class TransformerHead(nn.Module):
    def __init__(self, cin, n_tokens, dim=None, depth=None, heads=None,
                 drop=None, drop_path=None):
        super().__init__()
        dim   = dim   or cfg.TRANS_DIM
        depth = depth if depth is not None else cfg.TRANS_DEPTH
        heads = heads or cfg.TRANS_HEADS
        drop  = cfg.DROPOUT   if drop      is None else drop
        dpr   = cfg.DROP_PATH if drop_path is None else drop_path
        self.out_dim   = dim
        self.proj      = nn.Linear(cin, dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_tokens + 1, dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        rates = [dpr * i / max(depth - 1, 1) for i in range(depth)] if depth > 1 else [dpr]
        self.blocks = nn.ModuleList([EncoderBlock(dim, heads, 4.0, drop, rates[i])
                                     for i in range(depth)])
        self.norm, self.drop = nn.LayerNorm(dim), nn.Dropout(drop)
        self.n_tokens = n_tokens

    def _pos(self, n, dtype):
        """
        Positional embedding for `n` patch tokens.

        Multi-scale TTA feeds 352 px as well as 384 px, which turns a 12x12 feature grid
        into 11x11. Slicing pos_embed[:, :n+1] would silently hand token 120 of an 11x11
        grid the embedding trained for position 120 of a 12x12 grid - a misalignment that
        costs accuracy without raising anything. Interpolate on the 2D grid instead.
        """
        if n == self.n_tokens:
            return self.pos_embed.to(dtype)
        cls, patch = self.pos_embed[:, :1], self.pos_embed[:, 1:]
        s0, s1 = int(round(math.sqrt(self.n_tokens))), int(round(math.sqrt(n)))
        if s0 * s0 != self.n_tokens or s1 * s1 != n:
            return torch.cat([cls, patch[:, :n]], dim=1).to(dtype)   # non-square: fall back
        p = patch.reshape(1, s0, s0, -1).permute(0, 3, 1, 2)
        p = F.interpolate(p, size=(s1, s1), mode="bicubic", align_corners=False)
        p = p.permute(0, 2, 3, 1).reshape(1, s1 * s1, -1)
        return torch.cat([cls, p], dim=1).to(dtype)

    def forward(self, fmap):
        B = fmap.size(0)
        x = self.proj(fmap.flatten(2).transpose(1, 2))
        n = x.size(1)
        x = torch.cat([self.cls_token.expand(B, -1, -1).to(x.dtype), x], dim=1)
        x = x + self._pos(n, x.dtype)
        for b in self.blocks:
            x = b(x)
        return self.drop(self.norm(x)[:, 0])

# ---- segmentation decoder (v9, the main new source of signal) ----------------
class SegHead(nn.Module):
    """
    Two 3x3 convs from the CBAM feature map to one logit channel at the feature grid.

    The point is not to produce a good segmentation - it is that a feature map which can
    say WHERE the lesion is must have stopped relying on whole-image shortcuts. Cheap:
    under 0.2 M parameters, no upsampling path, no decoder skip connections.
    """
    def __init__(self, cin, hid=64):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(cin, hid, 3, padding=1, bias=False), nn.BatchNorm2d(hid),
            nn.ReLU(inplace=True),
            nn.Conv2d(hid, hid, 3, padding=1, bias=False), nn.BatchNorm2d(hid),
            nn.ReLU(inplace=True),
            nn.Conv2d(hid, 1, 1))
    def forward(self, f): return self.body(f)

_c = CBAM(1280); _t = TransformerHead(1280, 144); _s = SegHead(1280)
_f = torch.randn(2, 1280, 12, 12)
print("BLOCKS")
print("-" * 78)
print(f"  CBAM        in {tuple(_f.shape)} -> {tuple(_c(_f).shape)}   "
      f"{sum(p.numel() for p in _c.parameters())/1e6:.3f} M")
print(f"  Transformer 144 tokens + CLS -> {tuple(_t(_f).shape)}   "
      f"{sum(p.numel() for p in _t.parameters())/1e6:.3f} M "
      f"(dim {cfg.TRANS_DIM}, depth {cfg.TRANS_DEPTH}, heads {cfg.TRANS_HEADS})")
print(f"  SegHead     -> {tuple(_s(_f).shape)}   "
      f"{sum(p.numel() for p in _s.parameters())/1e6:.3f} M")
_f11 = torch.randn(2, 1280, 11, 11)          # what 352 px TTA produces from a 384 px model
print(f"  Transformer at a DIFFERENT grid (11x11, i.e. the 0.9 TTA scale) -> "
      f"{tuple(_t(_f11).shape)}   pos-embed interpolated, not sliced")
print("-" * 78)
del _c, _t, _s, _f, _f11


BLOCKS
------------------------------------------------------------------------------
  CBAM        in (2, 1280, 12, 12) -> (2, 1280, 12, 12)   0.206 M
  Transformer 144 tokens + CLS -> (2, 128)   0.579 M (dim 128, depth 2, heads 4)
  SegHead     -> (2, 1, 12, 12)   0.774 M
  Transformer at a DIFFERENT grid (11x11, i.e. the 0.9 TTA scale) -> (2, 128)   pos-embed interpolated, not sliced
------------------------------------------------------------------------------


In [17]:
# ============================================================================
# CELL 17 - BT-CAGTNet v9  (dual-scale, mask-supervised, multi-task)
# ============================================================================
BACKBONES = {
    "effnetv2s":   ("efficientnet_v2_s", "EfficientNet_V2_S_Weights", 1280),
    "effnetb0":    ("efficientnet_b0",   "EfficientNet_B0_Weights",   1280),
    "convnext_t":  ("convnext_tiny",     "ConvNeXt_Tiny_Weights",      768),
    "densenet121": ("densenet121",       "DenseNet121_Weights",       1024),
    "resnet50":    ("resnet50",          "ResNet50_Weights",          2048),
}

def make_backbone(name=None, pretrained=None):
    name       = name or cfg.BACKBONE
    pretrained = cfg.PRETRAINED if pretrained is None else pretrained
    fn, wenum, ch = BACKBONES[name]
    weights = None
    if pretrained:
        try:    weights = getattr(torchvision.models, wenum).IMAGENET1K_V1
        except Exception as e:
            print(f"  WARNING: pretrained weights unavailable ({e}); training from scratch")
    m = getattr(torchvision.models, fn)(weights=weights)
    feats = m.features if hasattr(m, "features") else nn.Sequential(*list(m.children())[:-2])

    # Optional: RadImageNet weights. A multi-centre bone-tumour study reports ResNet50 at
    # AUC 0.738 on RadImageNet vs 0.669 on ImageNet, so this is worth trying when attached.
    if pretrained and name == "resnet50" and RADIMAGENET_PT:
        try:
            sd = torch.load(RADIMAGENET_PT, map_location="cpu")
            sd = sd.get("state_dict", sd)
            sd = {k.replace("module.", "").replace("backbone.", ""): v for k, v in sd.items()}
            missing = feats.load_state_dict(sd, strict=False)
            print(f"  RadImageNet weights loaded ({len(missing.missing_keys)} keys unmatched)")
        except Exception as e:
            print(f"  RadImageNet load failed ({type(e).__name__}: {e}); ImageNet kept")
    return feats, ch


class BTCAGTNet(nn.Module):
    """
    CBAM-attended CNN + transformer over TWO views of one radiograph through a SHARED
    backbone, fused by an adaptive gate, with lesion-mask deep supervision and
    multi-task auxiliary heads.
    """
    def __init__(self, backbone=None, pretrained=None, use_cbam=None, use_graph=None,
                 use_trans=None, use_roi=None, use_seg=None, use_mt=None, fusion=None,
                 dropout=None, drop_path=None, emb_dim=None, img_size=None):
        super().__init__()
        self.use_cbam  = cfg.USE_CBAM      if use_cbam  is None else use_cbam
        self.use_graph = cfg.USE_GRAPH     if use_graph is None else use_graph
        self.use_trans = cfg.USE_TRANS     if use_trans is None else use_trans
        self.use_roi   = cfg.USE_ROI_VIEW  if use_roi   is None else use_roi
        self.use_seg   = cfg.USE_SEG_HEAD  if use_seg   is None else use_seg
        self.use_mt    = (cfg.USE_MULTITASK and MT_DIM > 0) if use_mt is None else use_mt
        self.fusion    = fusion or cfg.FUSION
        drop = cfg.DROPOUT   if dropout   is None else dropout
        dpr  = cfg.DROP_PATH if drop_path is None else drop_path
        D    = emb_dim or cfg.EMB_DIM
        size = img_size or cfg.IMG_SIZE

        self.backbone, C = make_backbone(backbone, pretrained)
        self.frozen = False
        self._apply_stage_freeze()

        with torch.no_grad():
            was = self.backbone.training; self.backbone.eval()
            fm = self.backbone(torch.zeros(1, 3, size, size))
            self.backbone.train(was)
        C, Hf, Wf = fm.shape[1], fm.shape[2], fm.shape[3]
        self.feat_ch, self.grid = C, (Hf, Wf)
        n_tokens = Hf * Wf

        fd = cfg.FEAT_DROP
        self.feat_drop = nn.Dropout2d(fd) if fd > 0 else nn.Identity()
        # one CBAM per view: the two scales have genuinely different channel statistics
        self.cbam      = CBAM(C) if self.use_cbam else nn.Identity()
        self.cbam_roi  = (CBAM(C) if self.use_cbam else nn.Identity()) if self.use_roi else None

        views = ["full"] + (["roi"] if self.use_roi else [])
        self.names, mods = [], {}
        for v in views:
            self.names.append(f"cnn_{v}")
            mods[f"cnn_{v}"] = nn.Sequential(nn.Dropout(drop), nn.Linear(C, D))
            if self.use_graph:
                g = PatchGraphHead(C, drop=drop)
                mods[f"graphbody_{v}"] = g
                mods[f"graph_{v}"] = nn.Linear(g.out_dim, D)
                self.names.append(f"graph_{v}")
            if self.use_trans:
                t = TransformerHead(C, n_tokens, dim=D, drop=drop, drop_path=dpr)
                mods[f"transbody_{v}"] = t
                mods[f"trans_{v}"] = nn.Linear(t.out_dim, D)
                self.names.append(f"trans_{v}")
        self.branch = nn.ModuleDict(mods)

        nb = len(self.names)
        self.aux_fc = nn.ModuleList([nn.Linear(D, 1) for _ in range(nb)])
        if self.fusion == "gated" and nb > 1:
            self.gate = nn.Sequential(nn.LayerNorm(nb*D), nn.Linear(nb*D, max(nb*D//4, 32)),
                                      nn.GELU(), nn.Dropout(drop),
                                      nn.Linear(max(nb*D//4, 32), nb))
        else:
            self.gate = None
        # v9.2: the FINAL classifier gets its own, higher, dedicated dropout - this is
        # the last line of defence against memorisation and must not be diluted by
        # whatever the (lower) branch-level DROPOUT happens to be set to.
        hdrop = getattr(cfg, "HEAD_DROPOUT", drop)
        self.head = nn.Sequential(nn.LayerNorm(D), nn.Dropout(hdrop), nn.Linear(D, 1))
        self.seg  = SegHead(C) if self.use_seg else None
        self.mt   = nn.Sequential(nn.Dropout(drop), nn.Linear(D, MT_DIM)) if self.use_mt else None
        self.last_fmap = None; self.last_gate = None; self.last_seg = None

    # -- freezing ------------------------------------------------------------
    def _stage_mods(self):
        """
        Freeze by parameter FRACTION, not stage index. EfficientNetV2-S keeps 72% of its
        20.2 M weights in stage 6 alone, so "freeze 3 stages" removes 1.6% of the
        parameters and changes nothing. Walk forward and freeze until the target is hit.
        """
        frac = float(getattr(cfg, "FREEZE_FRAC", 0.0))
        if frac <= 0:
            return []
        stages = list(self.backbone.children())
        total  = sum(p.numel() for p in self.backbone.parameters())
        target, chosen, acc = frac * total, [], 0
        for st in stages[:-1]:                       # never freeze the final stage
            n = sum(p.numel() for p in st.parameters())
            if acc + n <= target:
                chosen.append(st); acc += n; continue
            for blk in st.children():
                nb_ = sum(p.numel() for p in blk.parameters())
                if acc + nb_ > target:
                    break
                chosen.append(blk); acc += nb_
            break
        self._frozen_frac = acc / max(total, 1)
        return chosen

    def _apply_stage_freeze(self):
        for m in self._stage_mods():
            for p in m.parameters():
                p.requires_grad_(False)
            m.eval()

    def set_frozen(self, flag):
        self.frozen = bool(flag)
        for p in self.backbone.parameters():
            p.requires_grad_(not self.frozen)
        if self.frozen: self.backbone.eval()
        else:           self._apply_stage_freeze()
        return self

    def trainable_backbone_fraction(self):
        tot = sum(p.numel() for p in self.backbone.parameters())
        tr  = sum(p.numel() for p in self.backbone.parameters() if p.requires_grad)
        return tr / max(tot, 1)

    def train(self, mode=True):
        """requires_grad=False does NOT freeze BatchNorm running stats - this does."""
        super().train(mode)
        if self.frozen:
            self.backbone.eval()
        else:
            for m in self._stage_mods():
                m.eval()
        return self

    def _run_backbone(self, x):
        if not (getattr(cfg, "GRAD_CKPT", False) and self.training and not self.frozen):
            return self.backbone(x)
        from torch.utils.checkpoint import checkpoint_sequential
        mods = list(self.backbone)
        if not x.requires_grad:
            x = x.detach().requires_grad_(True)
        return checkpoint_sequential(mods, min(len(mods), 4), x, use_reentrant=False)

    def _view_embs(self, x, tag, cbam):
        f = self.feat_drop(self._run_backbone(x))
        f = cbam(f) if cbam is not None else f
        out = [self.branch[f"cnn_{tag}"](f.mean((2, 3)))]
        if self.use_graph:
            out.append(self.branch[f"graph_{tag}"](self.branch[f"graphbody_{tag}"](f)))
        if self.use_trans:
            out.append(self.branch[f"trans_{tag}"](self.branch[f"transbody_{tag}"](f)))
        return f, out

    def forward(self, x_full, x_roi=None):
        f_full, embs = self._view_embs(x_full, "full", self.cbam if self.use_cbam else None)
        self.last_fmap = f_full.detach() if not self.training else None
        if self.use_roi:
            xr = x_roi if x_roi is not None else x_full
            _, e2 = self._view_embs(xr, "roi", self.cbam_roi if self.use_cbam else None)
            embs = embs + e2

        E = torch.stack(embs, dim=1)                                   # B, nb, D
        if self.gate is not None:
            g = torch.softmax(self.gate(E.flatten(1)), dim=-1)
            fused = (E * g.unsqueeze(-1)).sum(1)
        else:
            g = torch.full((E.size(0), E.size(1)), 1.0 / E.size(1),
                           device=E.device, dtype=E.dtype)
            fused = E.mean(1)
        self.last_gate = g.detach() if not self.training else None

        logit = self.head(fused).squeeze(-1)
        aux   = torch.cat([fc(E[:, i]) for i, fc in enumerate(self.aux_fc)], dim=-1)
        seg   = self.seg(f_full) if self.seg is not None else None
        mt    = self.mt(fused)   if self.mt  is not None else None
        if seg is not None and not self.training:
            self.last_seg = seg.detach()
        return {"logit": logit, "aux": aux, "gate": g, "seg": seg, "mt": mt}


def n_params(m): return sum(p.numel() for p in unwrap(m).parameters())

def build_proposed(**kw):
    set_seed()
    return BTCAGTNet(**kw)

set_seed()
_m = build_proposed()
_o = _m(torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE),
        torch.randn(2, 3, cfg.IMG_SIZE, cfg.IMG_SIZE))
print("BT-CAGTNet v9")
print("-" * 84)
print(f"  backbone          : {cfg.BACKBONE}   feature map {_m.feat_ch} x {_m.grid[0]} x {_m.grid[1]}")
print(f"  views             : {'full + label-free ROI (shared backbone)' if _m.use_roi else 'full only'}")
print(f"  branches ({len(_m.names)})      : {_m.names}")
print(f"  fusion            : {cfg.FUSION}")
print(f"  segmentation head : {'ON' if _m.seg is not None else 'off'}"
      f"   -> {tuple(_o['seg'].shape) if _o['seg'] is not None else '-'}")
print(f"  multi-task head   : {'ON' if _m.mt  is not None else 'off'}"
      f"   -> {tuple(_o['mt'].shape)  if _o['mt']  is not None else '-'}")
print(f"  logit {tuple(_o['logit'].shape)} | aux {tuple(_o['aux'].shape)} | gate {tuple(_o['gate'].shape)}")
print(f"  parameters        : {n_params(_m)/1e6:.2f} M "
      f"(backbone {sum(p.numel() for p in _m.backbone.parameters())/1e6:.2f} M, "
      f"frozen {1-_m.trainable_backbone_fraction():.0%})")
_head_params = sum(p.numel() for p in _m.branch.parameters()) + \
               sum(p.numel() for p in _m.aux_fc.parameters()) + \
               (sum(p.numel() for p in _m.gate.parameters()) if _m.gate is not None else 0) + \
               sum(p.numel() for p in _m.head.parameters())
print(f"  head+fusion params: {_head_params/1e6:.3f} M  (embed dim {cfg.EMB_DIM}, "
      f"trans dim {cfg.TRANS_DIM}, head dropout {cfg.HEAD_DROPOUT}) "
      f"- this is the capacity that memorised the training set before v9.2")
print("-" * 84)
del _m, _o; free()


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 171MB/s]


BT-CAGTNet v9
------------------------------------------------------------------------------------
  backbone          : effnetb0   feature map 1280 x 12 x 12
  views             : full only
  branches (2)      : ['cnn_full', 'trans_full']
  fusion            : gated
  segmentation head : off   -> -
  multi-task head   : ON   -> (2, 23)
  logit (2,) | aux (2, 2) | gate (2, 2)
  parameters        : 4.99 M (backbone 4.01 M, frozen 21%)
  head+fusion params: 0.778 M  (embed dim 128, trans dim 128, head dropout 0.25) - this is the capacity that memorised the training set before v9.2
------------------------------------------------------------------------------------


In [18]:
# ============================================================================
# CELL 18 - METRICS, TTA INFERENCE, THRESHOLD CALIBRATION
# ============================================================================
def compute_metrics(y_true, y_prob, thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (np.asarray(y_prob) >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    def _safe(f, *a):
        try:    return float(f(*a))
        except Exception: return float("nan")
    return {
        "acc":   accuracy_score(y_true, y_pred),
        "bacc":  balanced_accuracy_score(y_true, y_pred),
        "prec":  precision_score(y_true, y_pred, zero_division=0),
        "rec":   recall_score(y_true, y_pred, zero_division=0),
        "spec":  tn / max(tn + fp, 1),
        "f1":    f1_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "mcc":   matthews_corrcoef(y_true, y_pred),
        "auroc": _safe(roc_auc_score, y_true, y_prob),
        "auprc": _safe(average_precision_score, y_true, y_prob),
        "cm":    cm.tolist(), "thr": float(thr),
    }

def fmt_metrics(m):
    return (f"acc {m['acc']:.4f} | bacc {m['bacc']:.4f} | f1 {m['f1']:.4f} | "
            f"sens {m['rec']:.4f} | spec {m['spec']:.4f} | "
            f"AUROC {m['auroc']:.4f} | AUPRC {m['auprc']:.4f}")

def best_threshold(y_true, y_prob, objective="acc"):
    """Fitted on VALIDATION only, then applied unchanged to every split."""
    grid = np.unique(np.round(np.clip(np.asarray(y_prob), 0, 1), 4))
    if grid.size > 400:
        grid = np.quantile(y_prob, np.linspace(0.01, 0.99, 400))
    best, bt = -1.0, 0.5
    for t in grid:
        pred = (np.asarray(y_prob) >= t).astype(int)
        s = (accuracy_score(y_true, pred) if objective == "acc"
             else balanced_accuracy_score(y_true, pred) if objective == "bacc"
             else f1_score(y_true, pred, zero_division=0))
        if s > best:
            best, bt = s, float(t)
    return bt, best


@torch.no_grad()
def predict(model, loader, tta=None):
    """Deterministic inference. TTA = horizontal flip x TTA_SCALES, probabilities averaged."""
    tta = cfg.USE_TTA if tta is None else tta
    model = model.to(DEVICE).eval()
    P, Y = [], []
    scales = cfg.TTA_SCALES if tta else (1.0,)
    for b in loader:
        xf = b["x_full"].to(DEVICE, non_blocking=True)
        xr = b["x_roi"].to(DEVICE, non_blocking=True)
        Y.append(b["y"].numpy())
        acc = None; nv = 0
        for sc in scales:
            if abs(sc - 1.0) > 1e-3:
                s = max(64, int(round(xf.shape[-1] * sc / 32) * 32))
                a = F.interpolate(xf, size=(s, s), mode="bilinear", align_corners=False)
                c = F.interpolate(xr, size=(s, s), mode="bilinear", align_corners=False)
            else:
                a, c = xf, xr
            for flip in ((False, True) if tta else (False,)):
                aa = torch.flip(a, dims=[3]) if flip else a
                cc = torch.flip(c, dims=[3]) if flip else c
                with autocast_ctx():
                    out = model(aa, cc)
                lg = out["logit"] if isinstance(out, dict) else out
                p  = torch.sigmoid(lg.float())
                acc = p if acc is None else acc + p
                nv += 1
        P.append((acc / nv).cpu().numpy())
    return np.concatenate(P), np.concatenate(Y)

def predict_frame(model, frame, tta=None, bs=None, **kw):
    return predict(model, make_loader(frame, train=False,
                                      bs=bs or max(cfg.BATCH * 2, 16), **kw), tta=tta)

@torch.no_grad()
def eval_loader(model, loader, thr=0.5):
    p, y = predict(model, loader, tta=False)
    m = compute_metrics(y, p, thr)
    m["loss"] = float(F.binary_cross_entropy(
        torch.tensor(np.clip(p, 1e-6, 1 - 1e-6)), torch.tensor(y.astype(np.float32))))
    return m, p, y

print("METRICS AND INFERENCE READY")
print("-" * 78)
print("  compute_metrics : acc, balanced acc, precision, recall/sensitivity, specificity,")
print("                    f1, Cohen kappa, MCC, AUROC, AUPRC, confusion matrix")
print("  best_threshold  : accuracy-maximising cut point, fitted on VALIDATION only")
print(f"  predict         : hflip x scales {cfg.TTA_SCALES} = "
      f"{2*len(cfg.TTA_SCALES) if cfg.USE_TTA else 1} forward passes per image")
print("-" * 78)


METRICS AND INFERENCE READY
------------------------------------------------------------------------------
  compute_metrics : acc, balanced acc, precision, recall/sensitivity, specificity,
                    f1, Cohen kappa, MCC, AUROC, AUPRC, confusion matrix
  best_threshold  : accuracy-maximising cut point, fitted on VALIDATION only
  predict         : hflip x scales (1.0, 0.9) = 4 forward passes per image
------------------------------------------------------------------------------


## Section 7 — The training engine, with F1 fixed

### What went wrong

`PATIENCE` counted against `score = val_AUROC − λ·max(0, gap − tol)`. The gap term grows
monotonically as the model learns, so `score` flattens while AUROC is still climbing, and the
patience counter runs out during the improving phase:

```
v8   ep 17  AUROC 0.8359  gap +0.0753  score 0.8042   <- last "best" by score
     ep 18  AUROC 0.8402  gap +0.0985  score 0.7795       AUROC up,  score down  (bad 1)
     ep 19  AUROC 0.8409  gap +0.0919  score 0.7886       AUROC up,  score down  (bad 2)
     ep 20  AUROC 0.8446  gap +0.1138  score 0.7648       AUROC up,  score down  (bad 3)
     ep 21  AUROC 0.8432  gap +0.1120  score 0.7656                              (bad 4)
     ep 22  AUROC 0.8489  gap +0.1003  score 0.7859       AUROC AT ITS MAXIMUM   (bad 5)
     -> early stop, PATIENCE = 5 exhausted, on the best epoch of the entire run
```

### What v9 does instead

Two mechanisms, separated:

- **during training** — patience counts against **raw val AUROC**, with `PATIENCE = 12` and not
  armed at all before `MIN_EPOCHS = 25`. Nothing about the gap can stop the run.
- **after training** — among the candidate epochs whose val AUROC is within `SELECT_BAND` (98%)
  of the peak, the one with the **smallest gap** is loaded. This is where gap control belongs:
  it picks between models that are all near-optimal, instead of preventing the model from
  becoming optimal in the first place.

The per-epoch `score` is still computed and printed, so the two can be compared, but it no
longer has any authority over when the run ends.

Everything else is unchanged: AMP, gradient accumulation to an effective batch of 32, clipping,
per-batch OOM recovery, per-epoch checkpointing, and a memory probe with the backbone
deliberately unfrozen before the first epoch so an OOM shows up at startup rather than at the
unfreeze epoch.


In [19]:
# ============================================================================
# CELL 19 - THE TRAINING ENGINE  (resumable, budget-aware, F1 fixed)
# ============================================================================
def _sd(m): return {k: v.detach().cpu().clone() for k, v in unwrap(m).state_dict().items()}

def _cpu_state(state):
    def mv(o):
        if torch.is_tensor(o):  return o.detach().cpu()
        if isinstance(o, dict): return {k: mv(v) for k, v in o.items()}
        if isinstance(o, list): return [mv(v) for v in o]
        return o
    return mv(state)

def _is_oom(e):
    return isinstance(e, getattr(torch.cuda, "OutOfMemoryError", ())) or \
           ("out of memory" in str(e).lower())

def compute_loss(model, xf, xr, ya, yb, lam, aux_w, seg_t=None, seg_v=None,
                 mt_t=None, mt_v=None, pos_weight=None, xf2=None):
    out = model(xf, xr)
    if not isinstance(out, dict):
        out = {"logit": out, "aux": None, "seg": None, "mt": None}
    loss  = mixed_bce(out["logit"].float(), ya, yb, lam, pos_weight=pos_weight)
    parts = {"main": float(loss.detach())}

    # v9.3 consistency regularisation: the model sees two independently augmented
    # views of the same images and must agree on their class probabilities.
    # Symmetric KL between the two sigmoid outputs, applied BEFORE gradient stop
    # so both heads are pushed toward agreement.  Only active when CONSIST_W > 0
    # and the loader produced a second view (train mode only).
    cw = getattr(cfg, "CONSIST_W", 0.0)
    if cw > 0 and xf2 is not None:
        out2 = model(xf2, xr)
        if not isinstance(out2, dict): out2 = {"logit": out2}
        p1  = torch.sigmoid(out["logit"].float()).clamp(1e-6, 1 - 1e-6)
        p2  = torch.sigmoid(out2["logit"].float()).clamp(1e-6, 1 - 1e-6)
        # Symmetric KL  (Jensen–Shannon style, but without the midpoint average)
        kl  = (p1 * (p1 / p2).log() + (1 - p1) * ((1 - p1) / (1 - p2)).log())
        kl  = kl + (p2 * (p2 / p1).log() + (1 - p2) * ((1 - p2) / (1 - p1)).log())
        lc  = kl.mean() * 0.5
        loss = loss + cw * lc; parts["consist"] = float(lc.detach())

    if out.get("aux") is not None and aux_w > 0:
        a = out["aux"].float()
        la = sum(mixed_bce(a[:, j], ya, yb, lam, pos_weight=pos_weight)
                 for j in range(a.size(1))) / a.size(1)
        loss = loss + aux_w * la; parts["aux"] = float(la.detach())

    # The seg and multi-task targets are NOT mixed. Under mixup the pixels of two images
    # are blended, so a single mask is no longer the right target for them; the honest
    # move is to apply these losses only on unmixed batches rather than to invent a
    # blended target for a spatial map.
    if out.get("seg") is not None and seg_t is not None and cfg.SEG_W > 0 and lam >= 1.0:
        ls = dice_bce(out["seg"], seg_t, seg_v)
        loss = loss + cfg.SEG_W * ls; parts["seg"] = float(ls.detach())
    if out.get("mt") is not None and mt_t is not None and cfg.MT_W > 0 and lam >= 1.0:
        lm = masked_bce(out["mt"], mt_t, mt_v)
        loss = loss + cfg.MT_W * lm; parts["mt"] = float(lm.detach())
    return loss, parts


def run_train_epoch(model, loader, opt, sched, scaler, ema, aux_w, pos_weight=None,
                    mix=True, accum=None):
    accum = accum or cfg.ACCUM
    model.train()
    tot, n, skipped = 0.0, 0, 0
    agg = defaultdict(float); agg_n = defaultdict(int)
    opt.zero_grad(set_to_none=True)
    for step, b in enumerate(loader):
        xf  = b["x_full"].to(DEVICE, non_blocking=True)
        xr  = b["x_roi"].to(DEVICE, non_blocking=True)
        xf2 = b["x_full2"].to(DEVICE, non_blocking=True) if "x_full2" in b else None
        y   = b["y"].to(DEVICE, non_blocking=True)
        st  = b["seg"].to(DEVICE, non_blocking=True)
        sv  = b["seg_valid"].to(DEVICE, non_blocking=True)
        mt  = b["mt"].to(DEVICE, non_blocking=True)
        mv  = b["mt_valid"].to(DEVICE, non_blocking=True)
        if mix:
            tensors = [xf, xr] + ([xf2] if xf2 is not None else [])
            mixed, ya, yb, lam = mix_batch(tensors, y)
            xf, xr = mixed[0], mixed[1]
            xf2 = mixed[2] if xf2 is not None else None
        else:
            ya, yb, lam = y, y, 1.0
        try:
            with autocast_ctx():
                loss, parts = compute_loss(model, xf, xr, ya, yb, lam, aux_w,
                                           st, sv, mt, mv, pos_weight, xf2=xf2)
            scaler.scale(loss / accum).backward()
        except Exception as e:
            if not _is_oom(e):
                raise
            skipped += 1
            opt.zero_grad(set_to_none=True)
            del xf, xr, y; free()
            if skipped > max(8, 0.05 * len(loader)):
                raise RuntimeError(
                    f"CUDA OOM on {skipped} batches - lower cfg.BATCH (currently {cfg.BATCH}) "
                    f"or cfg.IMG_SIZE, or set cfg.GRAD_CKPT = True.") from e
            continue

        if (step + 1) % accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(unwrap(model).parameters(), cfg.GRAD_CLIP)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            if sched is not None: sched.step()
            if ema  is not None:  ema.update(model)
        bs = xf.size(0)
        tot += float(loss.detach()) * bs; n += bs
        for k, v in parts.items():
            agg[k] += v * bs; agg_n[k] += bs
        del loss
    if skipped:
        print(f"        note: {skipped} batch(es) skipped on OOM")
    return tot / max(n, 1), {k: agg[k] / max(agg_n[k], 1) for k in agg}


def probe_memory(build_fn, batch=None, img_size=None, aux_w=None):
    """
    One full forward+backward with the backbone UNFROZEN, before training starts.
    Activations, gradients and AdamW state for the backbone all appear at the unfreeze
    epoch; probing that configuration up front turns a mid-run kernel restart into a
    startup message and shrinks cfg.BATCH automatically until the heaviest step fits.
    """
    if DEVICE.type != "cuda" or not cfg.AUTO_BATCH:
        return cfg.BATCH
    b   = batch or cfg.BATCH
    s   = img_size or cfg.IMG_SIZE
    aux = cfg.AUX_W if aux_w is None else aux_w
    while b >= cfg.MIN_BATCH:
        m = opt = None
        try:
            free()
            m = build_fn().to(DEVICE)
            if hasattr(m, "set_frozen"): m.set_frozen(False)
            m.train()
            opt = torch.optim.AdamW(param_groups(m, cfg.LR, cfg.WD, cfg.BACKBONE_LR_MULT))
            sc  = make_scaler()
            x = torch.randn(b, 3, s, s, device=DEVICE)
            y = torch.randint(0, 2, (b,), device=DEVICE).float()
            g = max(s // 8, 8)
            st = torch.zeros(b, 1, g, g, device=DEVICE); sv = torch.ones(b, device=DEVICE)
            mt = torch.zeros(b, MT_DIM, device=DEVICE); mv = torch.ones(b, MT_DIM, device=DEVICE)
            for _ in range(2):        # 2 steps: AdamW state only exists after the first
                with autocast_ctx():
                    loss, _ = compute_loss(m, x, x, y, y, 1.0, aux, st, sv, mt, mv)
                sc.scale(loss).backward(); sc.step(opt); sc.update()
                opt.zero_grad(set_to_none=True)
            peak  = torch.cuda.max_memory_allocated() / 1e9
            total = torch.cuda.get_device_properties(0).total_memory / 1e9
            del m, opt, sc, x, y, loss, st, sv, mt, mv; free()
            print(f"  memory probe: batch {b} @ {s}px, "
                  f"{'2 views, ' if cfg.USE_ROI_VIEW else ''}backbone UNFROZEN -> "
                  f"peak {peak:.2f} GB of {total:.1f} GB")
            if b != cfg.BATCH:
                print(f"  cfg.BATCH lowered {cfg.BATCH} -> {b}; ACCUM raised {cfg.ACCUM} -> "
                      f"{cfg.ACCUM * (cfg.BATCH // b)} to hold the effective batch at "
                      f"{cfg.BATCH * cfg.ACCUM}")
                cfg.ACCUM = cfg.ACCUM * max(cfg.BATCH // b, 1); cfg.BATCH = b
            return b
        except Exception as e:
            try:    del m, opt
            except Exception: pass
            free()
            if not _is_oom(e):
                print(f"  memory probe skipped ({type(e).__name__}: {e})")
                return cfg.BATCH
            print(f"  memory probe: batch {b} does not fit, trying {b // 2}")
            b //= 2
    raise RuntimeError(f"even batch {cfg.MIN_BATCH} does not fit at {s}px. "
                       f"Lower cfg.IMG_SIZE or set cfg.GRAD_CKPT = True.")


def clean_train_loader(frame, n=None, seed=0):
    """Deterministic, un-augmented sample of the training split, for the honest gap."""
    n = n or cfg.CLEAN_N
    sub = frame.sample(min(n, len(frame)), random_state=seed).reset_index(drop=True)
    return make_loader(sub, train=False, bs=max(cfg.BATCH * 2, 16))


def train_model(model, dl_train, dl_val, key, epochs=None, lr=None, wd=None,
                freeze_epochs=None, patience=None, aux_w=None, dl_clean=None,
                use_ema=True, mix=True, verbose=True, min_epoch_budget=1.25,
                min_epochs=None):
    """
    Trains `model`, checkpointing after every epoch under STORE/ckpt/<key>_last.pt.
    completed=False means the budget ran out mid-training; call again with the same
    `key` next session and it resumes from the saved epoch.
    """
    epochs = epochs if epochs is not None else cfg.EPOCHS
    lr     = lr     if lr     is not None else cfg.LR
    wd     = wd     if wd     is not None else cfg.WD
    fe     = freeze_epochs if freeze_epochs is not None else cfg.FREEZE_EPOCHS
    pat    = patience if patience is not None else cfg.PATIENCE
    minep  = cfg.MIN_EPOCHS if min_epochs is None else min_epochs
    minep  = min(minep, max(epochs - 3, 0))
    aux_w  = cfg.AUX_W if aux_w is None else aux_w

    last_p = STORE.path(f"ckpt/{key}_last.pt")
    best_p = STORE.path(f"ckpt/{key}_best.pt")

    model = model.to(DEVICE)
    if USE_DP and not isinstance(model, nn.DataParallel):
        model = nn.DataParallel(model)
    core = unwrap(model)
    has_freeze = hasattr(core, "set_frozen")

    opt    = torch.optim.AdamW(param_groups(model, lr, wd, cfg.BACKBONE_LR_MULT))
    sched  = make_scheduler(opt, epochs, max(len(dl_train) // cfg.ACCUM, 1))
    scaler = make_scaler()
    ema    = ModelEMA(model) if use_ema else None

    hist, cands, start_ep = [], [], 0
    best_metric, best_epoch, bad = -1e9, -1, 0

    if last_p.exists():
        try:
            ck = torch.load(last_p, map_location="cpu", weights_only=False)
            core.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
            sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
            if ema is not None and ck.get("ema") is not None:
                ema.module.load_state_dict(ck["ema"]); ema.step = ck.get("ema_step", 0)
            hist, cands = ck["hist"], ck.get("cands", [])
            start_ep    = ck["epoch"] + 1
            best_metric = ck["best_metric"]; best_epoch = ck["best_epoch"]; bad = ck["bad"]
            print(f"  [resume] '{key}' continues from epoch {start_ep+1}/{epochs}")
        except Exception as e:
            print(f"  [resume] checkpoint for '{key}' unreadable ({e}); starting fresh")

    ep_times = [h["secs"] for h in hist if h.get("secs")]
    completed = True

    for ep in range(start_ep, epochs):
        need = (np.median(ep_times) if ep_times else 0.0) * min_epoch_budget
        if ep_times and not have_time(need, f"{key} (epoch {ep+1}/{epochs})"):
            completed = False
            break

        if has_freeze:
            core.set_frozen(ep < fe)
            if ep == fe and fe > 0:
                for g, ng in zip(opt.param_groups,
                                 param_groups(model, lr, wd, cfg.BACKBONE_LR_MULT)):
                    g["params"] = ng["params"]
                if verbose:
                    print(f"        backbone unfrozen at epoch {ep+1} "
                          f"(LR x{cfg.BACKBONE_LR_MULT} = {lr*cfg.BACKBONE_LR_MULT:g})")

        t0 = time.time()
        tr_loss, parts = run_train_epoch(model, dl_train, opt, sched, scaler, ema,
                                         aux_w, mix=mix)
        eval_model = ema.module.to(DEVICE) if ema is not None else model
        vm, _, _ = eval_loader(eval_model, dl_val)
        cm_ = eval_loader(eval_model, dl_clean)[0] if dl_clean is not None else None
        tr_acc = cm_["acc"] if cm_ else float("nan")
        gap = (tr_acc - vm["acc"]) if cm_ else 0.0

        # ------------------------------------------------------------------
        # F1. The stopping metric is RAW val AUROC. The gap-penalised score is
        # computed and printed for comparison, but it has no authority here: in
        # v8 it flattened while AUROC climbed and killed the run at epoch 22,
        # which was the epoch with the highest AUROC of the entire run.
        # Gap control happens ONCE, at the end, in the SELECT_BAND rule below.
        # ------------------------------------------------------------------
        score  = vm["auroc"] - cfg.GAP_LAMBDA * max(0.0, gap - cfg.GAP_TOL)
        metric = vm["auroc"] if cfg.EARLY_METRIC == "auroc" else vm["acc"]

        secs = time.time() - t0; ep_times.append(secs)
        hist.append({"epoch": ep + 1, "train_loss": tr_loss, "train_acc": tr_acc,
                     "val_loss": vm["loss"], "val_acc": vm["acc"], "val_auroc": vm["auroc"],
                     "val_f1": vm["f1"], "gap": gap, "score": score,
                     "loss_main": parts.get("main", float("nan")),
                     "loss_seg": parts.get("seg", float("nan")),
                     "loss_mt": parts.get("mt", float("nan")),
                     "lr": opt.param_groups[0]["lr"], "secs": secs})

        improved = metric > best_metric + 1e-6
        if improved:
            best_metric, best_epoch, bad = metric, ep + 1, 0
        elif ep + 1 > minep:
            bad += 1

        cands.append({"epoch": ep + 1, "auroc": float(vm["auroc"]), "gap": float(gap),
                      "acc": float(vm["acc"])})
        cands.sort(key=lambda c: -c["auroc"])
        if len(cands) > cfg.N_CAND:
            dp = STORE.path(f"ckpt/{key}_cand{cands.pop()['epoch']}.pt")
            if dp.exists(): dp.unlink()
        if any(c["epoch"] == ep + 1 for c in cands):
            torch.save({"model": _sd(eval_model), "epoch": ep + 1, "val": vm},
                       STORE.path(f"ckpt/{key}_cand{ep+1}.pt"))

        if verbose:
            _al, _pk, _tt = gpu_mem()
            _mem = f" | vram {_pk:.1f}/{_tt:.0f}G" if DEVICE.type == "cuda" else ""
            _sg  = f" | seg {parts['seg']:.3f}" if "seg" in parts else ""
            _mtl = f" | mt {parts['mt']:.3f}"   if "mt"  in parts else ""
            _arm = "" if ep + 1 > minep else " [warmup]"
            print(f"  ep {ep+1:3d}/{epochs} | loss {tr_loss:.4f}{_sg}{_mtl} | "
                  f"train {tr_acc:.4f} | val {vm['acc']:.4f} | gap {gap:+.4f} | "
                  f"AUROC {vm['auroc']:.4f} | score {score:.4f}{_mem}"
                  f"{'  <-- best' if improved else ''}{_arm}")

        ck = {"model": _sd(model), "opt": _cpu_state(opt.state_dict()),
              "sched": sched.state_dict(), "scaler": scaler.state_dict(),
              "ema": (_sd(ema.module) if ema is not None else None),
              "ema_step": (ema.step if ema is not None else 0),
              "hist": hist, "cands": cands, "epoch": ep,
              "best_metric": best_metric, "best_epoch": best_epoch, "bad": bad}
        torch.save(ck, last_p); del ck
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        if bad >= pat and ep + 1 > minep:
            if verbose:
                print(f"  early stop at epoch {ep+1}: no val AUROC gain for {pat} epochs "
                      f"(best was epoch {best_epoch}, AUROC {best_metric:.4f})")
            break

    chosen = None
    if completed and cands:
        top    = max(c["auroc"] for c in cands)
        band   = [c for c in cands if c["auroc"] >= cfg.SELECT_BAND * top]
        chosen = min(band, key=lambda c: c["gap"])
        cp = STORE.path(f"ckpt/{key}_cand{chosen['epoch']}.pt")
        if cp.exists():
            core.load_state_dict(torch.load(cp, map_location="cpu",
                                            weights_only=False)["model"])
            core.to(DEVICE)
            torch.save({"model": _sd(core), "epoch": chosen["epoch"], "val": chosen}, best_p)
        best_epoch = chosen["epoch"]
        if verbose:
            print(f"  selection: peak val AUROC {top:.4f}; {len(band)} epoch(s) within "
                  f"{cfg.SELECT_BAND:.0%} of it; chose epoch {chosen['epoch']} "
                  f"(AUROC {chosen['auroc']:.4f}, gap {chosen['gap']:+.4f})")
        for c in cands:
            cp = STORE.path(f"ckpt/{key}_cand{c['epoch']}.pt")
            if c["epoch"] != chosen["epoch"] and cp.exists():
                cp.unlink()
    if completed and last_p.exists():
        last_p.unlink()          # optimiser state no longer needed; keeps the output small

    free()
    return {"history": pd.DataFrame(hist), "best_epoch": best_epoch,
            "best_metric": best_metric, "completed": completed, "model": model,
            "chosen": chosen}

print("TRAINING ENGINE READY  (v9)")
print("-" * 78)
print("  run_train_epoch : AMP, grad accumulation, clipping, EMA, per-batch OOM recovery")
print("  compute_loss    : BCE(tumour) + AUX_W*branch + SEG_W*Dice-BCE + MT_W*multi-task")
print(f"                  + CONSIST_W={getattr(cfg,'CONSIST_W',0.0)} * sym-KL(view1, view2)  [v9.3]")
print("                    seg and multi-task losses are skipped on MIXED batches - a")
print("                    blended image has no single correct spatial mask")
print(f"  early stopping  : raw val {cfg.EARLY_METRIC}, patience {cfg.PATIENCE}, "
      f"armed after epoch {cfg.MIN_EPOCHS}   [F1]")
print(f"  selection       : smallest gap among epochs within {cfg.SELECT_BAND:.0%} of peak AUROC")
print("  probe_memory    : one UNFROZEN fwd+bwd before training, auto-shrinks cfg.BATCH")
print("-" * 78)


TRAINING ENGINE READY  (v9)
------------------------------------------------------------------------------
  run_train_epoch : AMP, grad accumulation, clipping, EMA, per-batch OOM recovery
  compute_loss    : BCE(tumour) + AUX_W*branch + SEG_W*Dice-BCE + MT_W*multi-task
                  + CONSIST_W=0.0 * sym-KL(view1, view2)  [v9.3]
                    seg and multi-task losses are skipped on MIXED batches - a
                    blended image has no single correct spatial mask
  early stopping  : raw val auroc, patience 10, armed after epoch 15   [F1]
  selection       : smallest gap among epochs within 98% of peak AUROC
  probe_memory    : one UNFROZEN fwd+bwd before training, auto-shrinks cfg.BATCH
------------------------------------------------------------------------------


In [20]:
# ============================================================================
# CELL 20 - TRAIN BT-CAGTNet v9
# ============================================================================
PROP_KEY = "proposed_v9"
dl_clean = clean_train_loader(tr_df)

# v9.4 FORCED RETRAIN: architecture changed (TRANS_DIM/EMB_DIM 96->128, TRANS_DEPTH 1->2)
# so any checkpoint from a v9.3 run is incompatible. Unmark the stage and remove
# the stale checkpoint files so stage_status returns "run" and training starts fresh.
# Cache, split, and all other stages are left untouched (no wasted rebuild time).
if STORE.done(PROP_KEY):
    STORE.unmark(PROP_KEY)
    print("  [v9.4] unmarked 'proposed_v9' stage — will retrain with new architecture")
for _stale in ["proposed_v9_best.pt", "proposed_v9_resume.pt",
               "proposed_v9_ema.pt",  "proposed_v9_last.pt"]:
    _sp = STORE.root / "ckpt" / _stale
    if _sp.exists():
        _sp.unlink()
        print(f"  [v9.4] removed incompatible checkpoint: {_stale}")

pos_w = None
_pr = float(tr_df[cfg.TARGET].mean())
if not (0.35 < _pr < 0.65):
    pos_w = torch.tensor([(1 - _pr) / max(_pr, 1e-6)], device=DEVICE)
    print(f"  class imbalance (pos rate {_pr:.3f}) -> pos_weight {float(pos_w):.3f}")

_st = stage_status(PROP_KEY, 600, "proposed-model training")
prop_hist, model = None, None

if _st == "done":
    model = build_proposed().to(DEVICE)
    model.load_state_dict(torch.load(STORE.path(f"ckpt/{PROP_KEY}_best.pt"),
                                     map_location="cpu", weights_only=False)["model"])
    model.eval()
    prop_hist = STORE.get_df("obj/proposed_history.csv")
    print(f"  loaded trained weights (best epoch "
          f"{int(STORE.get_json('obj/proposed_meta.json', {}).get('best_epoch', -1))})")
elif _st == "run":
    probe_memory(build_proposed)
    if cfg.BATCH != dl_tr.batch_size:
        dl_tr, dl_va, dl_te = (make_loader(tr_df, train=True),
                               make_loader(va_df), make_loader(te_df))
        dl_clean = clean_train_loader(tr_df)
        print(f"  loaders rebuilt at batch {cfg.BATCH} ({len(dl_tr)} steps/epoch, "
              f"effective batch {cfg.BATCH*cfg.ACCUM})")
    set_seed()
    model = build_proposed()
    print(f"BT-CAGTNet v9 | {cfg.BACKBONE} | branches {model.names} | "
          f"seg {'on' if model.seg is not None else 'off'} | "
          f"multi-task {'on' if model.mt is not None else 'off'} | "
          f"{n_params(model)/1e6:.2f} M params")
    print("-" * 100)
    res = train_model(model, dl_tr, dl_va, PROP_KEY, dl_clean=dl_clean)
    model, prop_hist = res["model"], res["history"]
    if res["completed"]:
        STORE.put_df("obj/proposed_history.csv", prop_hist)
        STORE.put_json("obj/proposed_meta.json",
                       {"best_epoch": res["best_epoch"], "best_metric": res["best_metric"],
                        "params": n_params(model), "branches": unwrap(model).names})
        STORE.mark(PROP_KEY)
        torch.save({"model": _sd(model)}, f"{cfg.OUT_DIR}/models/btcagtnet_v9.pth")
        print(f"  weights saved -> {cfg.OUT_DIR}/models/btcagtnet_v9.pth")
    else:
        print("\n" + "=" * 100)
        print("  TIME BUDGET REACHED DURING TRAINING - progress is checkpointed.")
        print("  Save this version, attach its output to a new session, Run All to continue.")
        print("=" * 100)
    model = unwrap(model)

budget_banner()


  memory probe: batch 8 @ 384px, backbone UNFROZEN -> peak 1.02 GB of 15.6 GB
BT-CAGTNet v9 | effnetb0 | branches ['cnn_full', 'trans_full'] | seg off | multi-task on | 4.99 M params
----------------------------------------------------------------------------------------------------
  ep   1/60 | loss 0.9285 | mt 0.680 | train 0.6642 | val 0.6414 | gap +0.0228 | AUROC 0.7427 | score 0.7427 | vram 0.6/16G  <-- best [warmup]
  ep   2/60 | loss 0.8611 | mt 0.556 | train 0.7125 | val 0.7227 | gap -0.0102 | AUROC 0.8106 | score 0.8106 | vram 0.4/16G  <-- best [warmup]
        backbone unfrozen at epoch 3 (LR x0.1 = 2e-05)
  ep   3/60 | loss 0.8348 | mt 0.406 | train 0.7167 | val 0.7449 | gap -0.0283 | AUROC 0.8170 | score 0.8170 | vram 0.4/16G  <-- best [warmup]
  ep   4/60 | loss 0.8191 | mt 0.378 | train 0.7350 | val 0.7560 | gap -0.0210 | AUROC 0.8278 | score 0.8278 | vram 0.4/16G  <-- best [warmup]
  ep   5/60 | loss 0.7889 | mt 0.374 | train 0.7558 | val 0.7652 | gap -0.0094 | AUROC 0.

In [21]:
# ============================================================================
# CELL 21 - TRAINING CURVES AND THE F1 DIAGNOSTIC
# ============================================================================
if prop_hist is not None and len(prop_hist):
    h = prop_hist
    fig, ax = plt.subplots(1, 5, figsize=(24, 4.2))
    ax[0].plot(h.epoch, h.train_loss, label="train (total)")
    ax[0].plot(h.epoch, h.val_loss,   label="val (BCE)")
    if h.loss_seg.notna().any():
        ax[0].plot(h.epoch, h.loss_seg, label="seg (Dice+BCE)", ls=":")
    ax[0].set_title("loss"); ax[0].legend(fontsize=7)

    ax[1].plot(h.epoch, h.train_acc, label="clean-train")
    ax[1].plot(h.epoch, h.val_acc,   label="val")
    ax[1].axhline(0.93, ls="--", color="grey", lw=.8)
    ax[1].set_title("accuracy (dashed = 0.93 target)"); ax[1].legend(fontsize=7)

    ax[2].plot(h.epoch, h.gap, color="crimson")
    ax[2].axhline(cfg.GAP_TOL, ls="--", color="k")
    ax[2].set_title(f"generalisation gap (tolerance {cfg.GAP_TOL})")

    ax[3].plot(h.epoch, h.val_auroc, label="val AUROC (the stopping metric)")
    ax[3].plot(h.epoch, h.score, label="gap-penalised score (v8's metric)", ls="--")
    ax[3].axvline(int(h.loc[h.val_auroc.idxmax(), "epoch"]), color="green", ls=":",
                  label="peak AUROC")
    ax[3].axvline(int(h.loc[h.score.idxmax(), "epoch"]), color="crimson", ls=":",
                  label="where v8 would have stopped")
    ax[3].set_title("F1: what each metric would have chosen"); ax[3].legend(fontsize=7)

    ax[4].plot(h.epoch, h.lr, color="slateblue"); ax[4].set_yscale("log")
    ax[4].set_title("head learning rate")
    for a in ax: a.set_xlabel("epoch")
    fig.suptitle("BT-CAGTNet v9 training dynamics", fontweight="bold")
    fig.tight_layout(); savefig(fig, "06_training")

    _pk_a = int(h.loc[h.val_auroc.idxmax(), "epoch"])
    _pk_s = int(h.loc[h.score.idxmax(), "epoch"])
    print("\nGENERALISATION GAP OVER TRAINING")
    print("-" * 84)
    print(f"  epochs run            : {int(h.epoch.max())}")
    print(f"  final clean-train acc : {h.train_acc.iloc[-1]:.4f}")
    print(f"  final val acc         : {h.val_acc.iloc[-1]:.4f}")
    print(f"  final gap             : {h.gap.iloc[-1]:+.4f}")
    print(f"  max gap during run    : {h.gap.max():+.4f} "
          f"(epoch {int(h.loc[h.gap.idxmax(),'epoch'])})")
    print("-" * 84)
    print(f"  peak val AUROC at epoch {_pk_a} ({h.val_auroc.max():.4f})")
    print(f"  v8's gap-penalised score peaked at epoch {_pk_s} "
          f"({h.score.max():.4f}, AUROC there {h.loc[h.score.idxmax(),'val_auroc']:.4f})")
    if _pk_a > _pk_s:
        _lost = h.val_auroc.max() - h.loc[h.score.idxmax(), "val_auroc"]
        print(f"  -> F1 CONFIRMED on this run too: the old rule would have stopped "
              f"{_pk_a - _pk_s} epoch(s) early")
        print(f"     and given up {_lost:.4f} AUROC. v9 kept training.")
    else:
        print("  -> on this run the two rules agree, so F1 cost nothing here.")
    print("-" * 84)
    print("  For comparison: v5 ended at gap +0.1204 (train 0.9958 / val 0.8755);")
    print("  v8 ended at gap +0.1003 (train 0.8567 / val 0.7563) while still improving.")
else:
    print("  (training not complete in this session - curves appear after the next run)")


  figure saved -> /kaggle/working/outputs/figures/06_training.png

GENERALISATION GAP OVER TRAINING
------------------------------------------------------------------------------------
  epochs run            : 60
  final clean-train acc : 0.8633
  final val acc         : 0.8299
  final gap             : +0.0334
  max gap during run    : +0.0398 (epoch 54)
------------------------------------------------------------------------------------
  peak val AUROC at epoch 51 (0.8892)
  v8's gap-penalised score peaked at epoch 51 (0.8892, AUROC there 0.8892)
  -> on this run the two rules agree, so F1 cost nothing here.
------------------------------------------------------------------------------------
  For comparison: v5 ended at gap +0.1204 (train 0.9958 / val 0.8755);
  v8 ended at gap +0.1003 (train 0.8567 / val 0.7563) while still improving.


## Section 8 — Headline result, protocol A (case level)

The threshold is fitted on **validation** and applied unchanged to all three splits. The test
split is evaluated once, here, and never used to make a decision anywhere in this notebook.


In [22]:
# ============================================================================
# CELL 22 - CALIBRATE THE THRESHOLD, EVALUATE ALL THREE SPLITS
# ============================================================================
def evaluate_all_splits(mdl, frames, thr_objective="acc", calibrate=None, tta=None):
    calibrate = cfg.CALIBRATE_THRESHOLD if calibrate is None else calibrate
    preds = {k: predict_frame(mdl, fr, tta=tta) for k, fr in frames.items()}
    thr, _ = (best_threshold(preds["val"][1], preds["val"][0], thr_objective)
              if calibrate else (0.5, None))
    return {k: compute_metrics(y, p, thr) for k, (p, y) in preds.items()}, preds, thr

MAIN_READY = model is not None and STORE.done(PROP_KEY)
res_A = preds_A = None; THR = 0.5

if MAIN_READY:
    frames = {"train": tr_df, "val": va_df, "test": te_df}
    with Timer("full evaluation"):
        res_A, preds_A, THR = evaluate_all_splits(model.to(DEVICE), frames)

    print("\nDECISION THRESHOLD")
    print("-" * 84)
    print(f"  fitted on VALIDATION only : {THR:.4f} "
          f"({'calibrated' if cfg.CALIBRATE_THRESHOLD else 'fixed 0.5'})")
    print(f"  test-time augmentation    : "
          f"{'hflip x ' + str(cfg.TTA_SCALES) if cfg.USE_TTA else 'off'}")
    print("-" * 84)

    print("\nBT-CAGTNet v9 - PROTOCOL A (case-level split, leakage-safe)   << HEADLINE >>")
    print("=" * 104)
    rows = []
    for k in ["train", "val", "test"]:
        m = res_A[k]
        print(f"  {k:<6} {fmt_metrics(m)}")
        rows.append({"split": k, **{kk: round(float(m[kk]), 4) for kk in
                     ["acc","bacc","prec","rec","spec","f1","kappa","mcc","auroc","auprc"]}})
    print("=" * 104)
    main_tab = pd.DataFrame(rows)
    savetab(main_tab, "proposed_protocolA")
    STORE.put_df("obj/proposed_protocolA.csv", main_tab)
    for k, (p, y) in preds_A.items():
        STORE.put_np(f"obj/pred_A_{k}.npy", np.stack([p, y]))
    STORE.put_json("obj/threshold_A.json", {"thr": THR})

    accs = {k: res_A[k]["acc"] for k in ["train", "val", "test"]}
    print(f"\n  TARGET CHECK - protocol A")
    for lo in (0.90, 0.93):
        hit = all(v >= lo for v in accs.values())
        print(f"    >= {lo:.2f} on all three splits : {'MET' if hit else 'not met'}")
    print(f"    train {accs['train']:.4f} | val {accs['val']:.4f} | test {accs['test']:.4f}")
    print(f"    train-val gap {accs['train']-accs['val']:+.4f} | "
          f"train-test gap {accs['train']-accs['test']:+.4f}")
    print(f"    val AUROC {res_A['val']['auroc']:.4f} | test AUROC {res_A['test']['auroc']:.4f}")
    print("    Reference: accuracy ~0.93 on a balanced binary task needs AUROC ~0.975.")
    print("    v5 measured 0.9285 val / 0.9384 test; 5-fold CV put the ceiling at 0.927 +/- 0.008.")
else:
    print("  (skipped - the proposed model has not finished training yet)")


  [full evaluation] took 55.1s

DECISION THRESHOLD
------------------------------------------------------------------------------------
  fitted on VALIDATION only : 0.5238 (calibrated)
  test-time augmentation    : hflip x (1.0, 0.9)
------------------------------------------------------------------------------------

BT-CAGTNet v9 - PROTOCOL A (case-level split, leakage-safe)   << HEADLINE >>
  train  acc 0.8745 | bacc 0.8745 | f1 0.8740 | sens 0.8783 | spec 0.8708 | AUROC 0.9493 | AUPRC 0.9521
  val    acc 0.8318 | bacc 0.8314 | f1 0.8378 | sens 0.8545 | spec 0.8083 | AUROC 0.8934 | AUPRC 0.8879
  test   acc 0.8115 | bacc 0.8114 | f1 0.8192 | sens 0.8506 | spec 0.7722 | AUROC 0.9064 | AUPRC 0.9254
  table saved  -> /kaggle/working/outputs/tables/proposed_protocolA.csv

  TARGET CHECK - protocol A
    >= 0.90 on all three splits : not met
    >= 0.93 on all three splits : not met
    train 0.8745 | val 0.8318 | test 0.8115
    train-val gap +0.0427 | train-test gap +0.0629
    val AU

In [23]:
# ============================================================================
# CELL 23 - ROC, PR, CONFUSION MATRIX, GATE ANALYSIS, SEGMENTATION SANITY
# ============================================================================
if MAIN_READY:
    fig, ax = plt.subplots(1, 4, figsize=(20, 4.3))
    for k, c in [("train", "grey"), ("val", "tab:orange"), ("test", "tab:blue")]:
        p, y = preds_A[k]
        fpr, tpr, _ = roc_curve(y, p)
        ax[0].plot(fpr, tpr, color=c, label=f"{k} AUC {res_A[k]['auroc']:.3f}")
        pr, rc, _ = precision_recall_curve(y, p)
        ax[1].plot(rc, pr, color=c, label=f"{k} AP {res_A[k]['auprc']:.3f}")
    ax[0].plot([0,1],[0,1],"k--",lw=.8); ax[0].set_title("ROC"); ax[0].legend(fontsize=8)
    ax[0].set_xlabel("1 - specificity"); ax[0].set_ylabel("sensitivity")
    ax[1].set_title("Precision-Recall"); ax[1].legend(fontsize=8)
    ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision")

    cm = np.array(res_A["test"]["cm"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax[2],
                xticklabels=["no tumor","tumor"], yticklabels=["no tumor","tumor"])
    ax[2].set_title(f"test confusion matrix (thr {THR:.3f})")
    ax[2].set_xlabel("predicted"); ax[2].set_ylabel("true")

    p, y = preds_A["test"]
    safe_hist(ax[3], p[y == 0], 40, alpha=.6, label="no tumor")
    safe_hist(ax[3], p[y == 1], 40, alpha=.6, label="tumor")
    ax[3].axvline(THR, color="k", ls="--"); ax[3].set_title("test score distribution")
    ax[3].legend(fontsize=8)
    fig.suptitle("BT-CAGTNet v9 - protocol A (case level)", fontweight="bold")
    fig.tight_layout(); savefig(fig, "07_curves")

    # ---- what the adaptive gate learned -----------------------------------
    core = unwrap(model).to(DEVICE).eval()
    gate_tab = None
    if getattr(core, "gate", None) is not None:
        gates = []
        with torch.no_grad():
            for b in make_loader(te_df, bs=max(cfg.BATCH*2, 16)):
                core(b["x_full"].to(DEVICE), b["x_roi"].to(DEVICE))
                gates.append(core.last_gate.float().cpu().numpy())
        G = np.concatenate(gates)
        gate_tab = pd.DataFrame({"branch": core.names,
                                 "mean_weight": G.mean(0).round(4),
                                 "std_weight":  G.std(0).round(4),
                                 "argmax_share": [round(float((G.argmax(1) == i).mean()), 4)
                                                  for i in range(G.shape[1])]})
        print("\nADAPTIVE GATE ON THE TEST SPLIT")
        print("-" * 84)
        print(gate_tab.to_string(index=False))
        print("-" * 84)
        print("  A near-uniform mean with non-trivial std is the healthy pattern: the gate")
        print("  re-weights per image rather than collapsing onto one branch. A branch with")
        print("  argmax_share 0.00 never decided an image and should be cut, as the patch-")
        print("  graph branch was after v5.")
        savetab(gate_tab, "gate_weights")

        fig, ax = plt.subplots(1, 2, figsize=(13, 3.9))
        for i, nm in enumerate(core.names):
            safe_hist(ax[0], G[:, i], 40, alpha=.55, label=nm)
        ax[0].set_title("per-image gate weight"); ax[0].legend(fontsize=7)
        ax[1].bar(core.names, G.mean(0)); ax[1].set_title("mean gate weight")
        ax[1].tick_params(axis="x", rotation=20)
        fig.tight_layout(); savefig(fig, "08_gate")

    # ---- did the segmentation head actually learn to localise? -------------
    if core.seg is not None:
        _sub = te_df[te_df.has_annot].head(6)
        if len(_sub) >= 2:
            _dl = make_loader(_sub, bs=len(_sub))
            b = next(iter(_dl))
            with torch.no_grad():
                out = core(b["x_full"].to(DEVICE), b["x_roi"].to(DEVICE))
            pm = torch.sigmoid(out["seg"].float()).cpu().numpy()[:, 0]
            # the target is cached at IMG_SIZE//8, the decoder outputs the backbone's
            # feature grid - pool the target down so the two are comparable
            gt = F.adaptive_avg_pool2d(b["seg"].float(),
                                       out["seg"].shape[-2:]).numpy()[:, 0]
            inter = (pm > 0.5).astype(float) * (gt > 0.5).astype(float)
            union = np.clip((pm > 0.5).astype(float) + (gt > 0.5).astype(float), 0, 1)
            iou = float(inter.sum() / max(union.sum(), 1))
            fig, ax = plt.subplots(3, len(_sub), figsize=(3*len(_sub), 9))
            for j in range(len(_sub)):
                im = b["x_full"][j][0].numpy() * current_norm()[1][0] + current_norm()[0][0]
                ax[0,j].imshow(np.clip(im, 0, 1), cmap="gray"); ax[0,j].set_title("input", fontsize=8)
                ax[1,j].imshow(gt[j], cmap="magma"); ax[1,j].set_title("mask target", fontsize=8)
                ax[2,j].imshow(pm[j], cmap="magma", vmin=0, vmax=1)
                ax[2,j].set_title("predicted", fontsize=8)
                for r in range(3): ax[r,j].axis("off")
            fig.suptitle(f"Segmentation deep supervision - IoU on this sample {iou:.3f}",
                         fontweight="bold")
            fig.tight_layout(); savefig(fig, "09_seg")
            print(f"\n  segmentation head IoU on {len(_sub)} annotated test images : {iou:.3f}")
            print("  This is a diagnostic, not a result: the head exists to force the feature")
            print("  map to localise, and a modest IoU is enough for that. A near-zero IoU")
            print("  would mean the auxiliary loss contributed nothing.")
    free()


  figure saved -> /kaggle/working/outputs/figures/07_curves.png

ADAPTIVE GATE ON THE TEST SPLIT
------------------------------------------------------------------------------------
    branch  mean_weight  std_weight  argmax_share
  cnn_full       0.0834      0.0753           0.0
trans_full       0.9166      0.0753           1.0
------------------------------------------------------------------------------------
  A near-uniform mean with non-trivial std is the healthy pattern: the gate
  re-weights per image rather than collapsing onto one branch. A branch with
  argmax_share 0.00 never decided an image and should be cut, as the patch-
  graph branch was after v5.
  table saved  -> /kaggle/working/outputs/tables/gate_weights.csv
  figure saved -> /kaggle/working/outputs/figures/08_gate.png


In [24]:
# ============================================================================
# CELL 24 - PROVENANCE ASSERTIONS: NO METADATA, NO LABEL, REACHES ANY BRANCH
# ============================================================================
_meta_cols = set(df.columns) - {"path", "stem", "cache_idx", "split", "split_img",
                                "group_id", "group_meta", "group_phash", "md5", "has_annot", "fold"}
_probe = BTDataset(tr_df.iloc[[0]], train=False)
_d     = _probe[0]

print("INPUT PROVENANCE ASSERTIONS")
print("-" * 84)
print(f"  metadata columns in the Excel file : {len(_meta_cols)}")
print(f"  model input tensors : x_full {tuple(_d['x_full'].shape)}  "
      f"x_roi {tuple(_d['x_roi'].shape)}   (pixels only)")
print("  1. BTDataset reads IMGS[cache_idx], MASKS[cache_idx], ROIBOX[cache_idx] and labels.")
assert set(BTDataset.__init__.__code__.co_names) & {"cache_idx"}, "dataset provenance changed"
print("     PASS - no metadata column can reach the CNN, transformer or gate.")

print("  2. forward() signature takes only tensors:")
_sig = BTCAGTNet.forward.__code__.co_varnames[:BTCAGTNet.forward.__code__.co_argcount]
print(f"     {list(_sig)}")
assert set(_sig) <= {"self", "x_full", "x_roi"}, "forward() gained a non-pixel argument"
print("     PASS - the network is never handed a label, a mask or a metadata row.")

print("  3. the multi-task and segmentation labels are TARGETS, not inputs:")
print(f"     mt {tuple(_d['mt'].shape)} and seg {tuple(_d['seg'].shape)} appear only inside")
print("     compute_loss(), never inside model(...). Verified by the signature above.")

print("  4. the ROI crop is label-free (asserted in full in Cell 10).")
print("-" * 84)
del _probe, _d


INPUT PROVENANCE ASSERTIONS
------------------------------------------------------------------------------------
  metadata columns in the Excel file : 37
  model input tensors : x_full (3, 384, 384)  x_roi (3, 384, 384)   (pixels only)
  1. BTDataset reads IMGS[cache_idx], MASKS[cache_idx], ROIBOX[cache_idx] and labels.
     PASS - no metadata column can reach the CNN, transformer or gate.
  2. forward() signature takes only tensors:
     ['self', 'x_full', 'x_roi']
     PASS - the network is never handed a label, a mask or a metadata row.
  3. the multi-task and segmentation labels are TARGETS, not inputs:
     mt (23,) and seg (1, 1, 1) appear only inside
     compute_loss(), never inside model(...). Verified by the signature above.
  4. the ROI crop is label-free (asserted in full in Cell 10).
------------------------------------------------------------------------------------


## Section 9 — Architecture ensemble

A second member with a genuinely different inductive bias (ConvNeXt-Tiny: large-kernel depthwise
convolutions and LayerNorm, against EfficientNetV2-S's inverted-residual MBConv stack) trained on
the same split with the same recipe, then averaged in probability space with **one** threshold
re-fitted on validation.

This is not the fold ensemble that v5 tried. That averaged five models each trained on 80% of the
training split, and it *lost* to the single model (0.8595 against 0.8759) because every member was
weaker than the whole-data model. Here both members see all the training data, and they differ in
architecture rather than in which rows they were shown.


In [25]:
# ============================================================================
# CELL 25 - ARCHITECTURE ENSEMBLE
# ============================================================================
ens_members, ens_res, ens_tab = [], None, None

if cfg.RUN_ENSEMBLE and MAIN_READY:
    for bb in cfg.ENSEMBLE_BACKBONES:
        key = f"ens_{bb}"
        st  = stage_status(key, 900, f"ensemble member {bb}")
        if st == "skip":
            break
        m = build_proposed(backbone=bb)
        if st == "done":
            m.load_state_dict(torch.load(STORE.path(f"ckpt/{key}_best.pt"),
                                         map_location="cpu", weights_only=False)["model"])
            ens_members.append((bb, m.to(DEVICE).eval()))
            print(f"  [resume] ensemble member {bb} loaded")
            continue
        print(f"\n>>> ensemble member: {bb}  ({n_params(m)/1e6:.2f} M params)")
        r = train_model(m, dl_tr, dl_va, key, epochs=cfg.ENS_EPOCHS, dl_clean=dl_clean,
                        min_epochs=min(cfg.MIN_EPOCHS, cfg.ENS_EPOCHS - 3))
        if r["completed"]:
            STORE.mark(key)
            ens_members.append((bb, unwrap(r["model"]).to(DEVICE).eval()))
        else:
            print("  budget reached - checkpointed, resume next session")
        del m, r; free()

if ens_members and MAIN_READY:
    all_m = [(cfg.BACKBONE, model.to(DEVICE).eval())] + ens_members
    P = {}
    for split, fr in [("train", tr_df), ("val", va_df), ("test", te_df)]:
        ps = []
        for nm, m in all_m:
            p, y = predict_frame(m, fr)
            ps.append(p)
        P[split] = (np.mean(ps, axis=0), y, np.stack(ps))
    thr_e, _ = best_threshold(P["val"][1], P["val"][0], "acc")
    ens_res  = {k: compute_metrics(P[k][1], P[k][0], thr_e) for k in P}

    print("\nENSEMBLE  (" + " + ".join(nm for nm, _ in all_m) + ")")
    print("=" * 104)
    rows = []
    for k in ["train", "val", "test"]:
        print(f"  {k:<6} {fmt_metrics(ens_res[k])}")
        rows.append({"split": k, **{kk: round(float(ens_res[k][kk]), 4) for kk in
                     ["acc","bacc","prec","rec","spec","f1","kappa","mcc","auroc","auprc"]}})
    print("=" * 104)
    print(f"  threshold {thr_e:.4f} (validation only) | members {len(all_m)}")
    print(f"  single model   : test acc {res_A['test']['acc']:.4f} | "
          f"AUROC {res_A['test']['auroc']:.4f}")
    print(f"  ensemble       : test acc {ens_res['test']['acc']:.4f} | "
          f"AUROC {ens_res['test']['auroc']:.4f}   "
          f"({ens_res['test']['acc']-res_A['test']['acc']:+.4f} acc)")
    _corr = np.corrcoef(P["test"][2])[0, 1] if P["test"][2].shape[0] > 1 else float("nan")
    print(f"  member score correlation on test : {_corr:.4f}")
    print("  Low correlation is what makes an ensemble worth its compute; near 1.0 means")
    print("  the second architecture learned the same function and the gain will be noise.")
    ens_tab = pd.DataFrame(rows)
    savetab(ens_tab, "ensemble_protocolA")
    STORE.put_df("obj/ensemble_protocolA.csv", ens_tab)
    for k in P:
        STORE.put_np(f"obj/pred_ens_{k}.npy", np.stack([P[k][0], P[k][1]]))
    STORE.put_json("obj/threshold_ens.json", {"thr": thr_e})
    free()
elif cfg.RUN_ENSEMBLE:
    print("  (ensemble skipped - no additional member finished)")

budget_banner()


Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 202MB/s] 



>>> ensemble member: convnext_t  (28.54 M params)
  ep   1/45 | loss 0.9403 | mt 0.690 | train 0.5750 | val 0.5675 | gap +0.0075 | AUROC 0.6997 | score 0.6997 | vram 0.7/16G  <-- best [warmup]
  ep   2/45 | loss 0.8929 | mt 0.561 | train 0.6992 | val 0.7061 | gap -0.0069 | AUROC 0.7792 | score 0.7792 | vram 0.7/16G  <-- best [warmup]
        backbone unfrozen at epoch 3 (LR x0.1 = 2e-05)
  ep   3/45 | loss 0.8290 | mt 0.400 | train 0.7308 | val 0.7338 | gap -0.0030 | AUROC 0.8284 | score 0.8284 | vram 1.0/16G  <-- best [warmup]
  ep   4/45 | loss 0.8068 | mt 0.383 | train 0.7492 | val 0.7505 | gap -0.0013 | AUROC 0.8438 | score 0.8438 | vram 1.0/16G  <-- best [warmup]
  ep   5/45 | loss 0.7818 | mt 0.377 | train 0.7850 | val 0.7837 | gap +0.0013 | AUROC 0.8584 | score 0.8584 | vram 1.0/16G  <-- best [warmup]
  ep   6/45 | loss 0.7667 | mt 0.374 | train 0.7992 | val 0.7911 | gap +0.0080 | AUROC 0.8690 | score 0.8690 | vram 1.0/16G  <-- best [warmup]
  ep   7/45 | loss 0.7488 | mt 0.370

In [26]:
# ============================================================================
# CELL 26 - PROTOCOL B: THE OFFICIAL BTXRD IMAGE-LEVEL SPLIT
# ============================================================================
# The BTXRD dataset paper divides images randomly, image level, and reports on that.
# Training the SAME architecture with the SAME recipe on that split is the only way to
# state the leakage premium as a measured quantity rather than an assertion - and it is
# the number that is comparable with published BTXRD results.
res_B, model_B = None, None

if cfg.RUN_RQ1:
    key = "rq1_imagelevel"
    st  = stage_status(key, 900, "protocol B (image level)")
    if st == "done":
        res_B = STORE.get_json(f"obj/{key}.json")
    elif st == "run":
        tr_i = df[df.split_img == "train"].reset_index(drop=True)
        va_i = df[df.split_img == "val"].reset_index(drop=True)
        te_i = df[df.split_img == "test"].reset_index(drop=True)
        if cfg.QUICK_MODE:
            tr_i = tr_i.sample(min(128, len(tr_i)), random_state=0).reset_index(drop=True)
            va_i = va_i.sample(min(64,  len(va_i)), random_state=0).reset_index(drop=True)
            te_i = te_i.sample(min(64,  len(te_i)), random_state=0).reset_index(drop=True)
        print(f">>> protocol B: train {len(tr_i)} / val {len(va_i)} / test {len(te_i)}")
        set_seed()
        m = build_proposed()
        r = train_model(m, make_loader(tr_i, train=True), make_loader(va_i), key,
                        epochs=cfg.RQ1_EPOCHS, dl_clean=clean_train_loader(tr_i),
                        min_epochs=min(cfg.MIN_EPOCHS, cfg.RQ1_EPOCHS - 3))
        if r["completed"]:
            mdl = unwrap(r["model"]).to(DEVICE)
            rb, pb, thr_b = evaluate_all_splits(mdl, {"train": tr_i, "val": va_i, "test": te_i})
            res_B = {k: {kk: float(v[kk]) for kk in
                         ["acc","bacc","prec","rec","spec","f1","kappa","mcc","auroc","auprc"]}
                     for k, v in rb.items()}
            res_B["thr"] = thr_b
            STORE.put_json(f"obj/{key}.json", res_B); STORE.mark(key)
            for k in pb:
                STORE.put_np(f"obj/pred_B_{k}.npy", np.stack([pb[k][0], pb[k][1]]))
            print("\nBT-CAGTNet v9 - PROTOCOL B (image-level split, official BTXRD protocol)")
            print("=" * 104)
            for k in ["train", "val", "test"]:
                print(f"  {k:<6} {fmt_metrics(rb[k])}")
            print("=" * 104)
            del mdl
        else:
            print("  budget reached - checkpointed, resume next session")
        del m, r; free()
else:
    print("  protocol B disabled (cfg.RUN_RQ1 = False)")

# ---- the comparison table ---------------------------------------------------
proto_tab = None
if MAIN_READY and res_B is not None:
    rows = []
    for k in ["train", "val", "test"]:
        rows.append({"split": k, "protocol": "A - case level (leakage-safe)",
                     "acc": round(res_A[k]["acc"], 4), "f1": round(res_A[k]["f1"], 4),
                     "auroc": round(res_A[k]["auroc"], 4)})
        rows.append({"split": k, "protocol": "B - image level (official BTXRD)",
                     "acc": round(res_B[k]["acc"], 4), "f1": round(res_B[k]["f1"], 4),
                     "auroc": round(res_B[k]["auroc"], 4)})
    proto_tab = pd.DataFrame(rows)
    print("\nPROTOCOL COMPARISON - SAME ARCHITECTURE, SAME RECIPE, SAME SEED")
    print("=" * 104)
    print(proto_tab.pivot(index="split", columns="protocol",
                          values=["acc", "f1", "auroc"]).to_string())
    print("=" * 104)
    prem = res_B["test"]["acc"] - res_A["test"]["acc"]
    print(f"\n  LEAKAGE PREMIUM on test accuracy : {prem:+.4f} ({prem*100:+.2f} points)")
    print(f"  {img_span} of {len(df)} images ({100*img_span/len(df):.1f}%) sit in case groups")
    print("  that span splits under protocol B. That is the mechanism behind the premium.")
    print("  Protocol B is comparable with published BTXRD numbers. It is NOT comparable")
    print("  with protocol A, and quoting it as leakage-safe would sink the study.")
    savetab(proto_tab, "protocol_comparison")
    STORE.put_df("obj/protocol_comparison.csv", proto_tab)

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
    pv = proto_tab.pivot(index="split", columns="protocol", values="acc").loc[["train","val","test"]]
    pv.plot(kind="bar", ax=ax[0]); ax[0].set_ylim(0.5, 1.02)
    ax[0].axhline(0.93, ls="--", color="k"); ax[0].set_title("accuracy by protocol (dashed 0.93)")
    pv2 = proto_tab.pivot(index="split", columns="protocol", values="auroc").loc[["train","val","test"]]
    pv2.plot(kind="bar", ax=ax[1]); ax[1].set_ylim(0.5, 1.02); ax[1].set_title("AUROC by protocol")
    for a in ax: a.tick_params(axis="x", rotation=0); a.legend(fontsize=7)
    fig.tight_layout(); savefig(fig, "10_protocol_comparison")

budget_banner()


>>> protocol B: train 2622 / val 562 / test 562
  ep   1/40 | loss 0.9232 | mt 0.680 | train 0.6883 | val 0.6762 | gap +0.0122 | AUROC 0.7572 | score 0.7572 | vram 0.5/16G  <-- best [warmup]
  ep   2/40 | loss 0.8531 | mt 0.543 | train 0.7408 | val 0.7135 | gap +0.0273 | AUROC 0.8116 | score 0.8116 | vram 0.5/16G  <-- best [warmup]
        backbone unfrozen at epoch 3 (LR x0.1 = 2e-05)
  ep   3/40 | loss 0.8430 | mt 0.400 | train 0.7250 | val 0.7278 | gap -0.0028 | AUROC 0.7852 | score 0.7852 | vram 0.5/16G [warmup]
  ep   4/40 | loss 0.8130 | mt 0.377 | train 0.7392 | val 0.7278 | gap +0.0114 | AUROC 0.8091 | score 0.8091 | vram 0.5/16G [warmup]
  ep   5/40 | loss 0.7997 | mt 0.371 | train 0.7458 | val 0.7367 | gap +0.0092 | AUROC 0.8273 | score 0.8273 | vram 0.5/16G  <-- best [warmup]
  ep   6/40 | loss 0.7830 | mt 0.371 | train 0.7575 | val 0.7598 | gap -0.0023 | AUROC 0.8394 | score 0.8394 | vram 0.5/16G  <-- best [warmup]
  ep   7/40 | loss 0.7750 | mt 0.371 | train 0.7708 | val 0

In [27]:
# ============================================================================
# CELL 27 - 5-FOLD GROUP CROSS-VALIDATION
# ============================================================================
# Folds are over CASE GROUPS, not images, and stratified by the group-level label, so
# every fold obeys protocol A. This is what gives the honest variance of the estimate -
# the single-split number is one draw from this distribution.
cv_rows, cv_preds = [], {}

if cfg.RUN_CV:
    g = df.groupby("group_id")[cfg.TARGET].first().reset_index()
    rng = np.random.default_rng(cfg.SEED)
    fold_of = {}
    for lab in sorted(g[cfg.TARGET].unique()):
        ids = g.loc[g[cfg.TARGET] == lab, "group_id"].to_numpy().copy()
        rng.shuffle(ids)
        for i, gid in enumerate(ids):
            fold_of[gid] = i % cfg.N_FOLDS
    df["fold"] = df.group_id.map(fold_of)

    for f in range(cfg.N_FOLDS):
        key = f"cv{f}"
        st  = stage_status(key, 700, f"CV fold {f+1}/{cfg.N_FOLDS}")
        if st == "skip":
            break
        if st == "done":
            cv_rows.append(STORE.get_json(f"obj/{key}.json")); continue
        te_f = df[df.fold == f].reset_index(drop=True)
        rest = df[df.fold != f].reset_index(drop=True)
        vg   = rest.groupby("group_id")[cfg.TARGET].first().reset_index()
        rg   = np.random.default_rng(cfg.SEED + f)
        vids = set()
        for lab in sorted(vg[cfg.TARGET].unique()):
            ids = vg.loc[vg[cfg.TARGET] == lab, "group_id"].to_numpy().copy()
            rg.shuffle(ids); vids |= set(ids[:max(1, int(0.15 * len(ids)))])
        va_f = rest[rest.group_id.isin(vids)].reset_index(drop=True)
        tr_f = rest[~rest.group_id.isin(vids)].reset_index(drop=True)
        if cfg.QUICK_MODE:
            tr_f = tr_f.sample(min(96, len(tr_f)), random_state=0).reset_index(drop=True)
            va_f = va_f.sample(min(48, len(va_f)), random_state=0).reset_index(drop=True)
            te_f = te_f.sample(min(48, len(te_f)), random_state=0).reset_index(drop=True)
        print(f"\n>>> fold {f+1}/{cfg.N_FOLDS}: train {len(tr_f)} / val {len(va_f)} / test {len(te_f)}")
        set_seed(cfg.SEED + f)
        m = build_proposed()
        r = train_model(m, make_loader(tr_f, train=True), make_loader(va_f), key,
                        epochs=cfg.CV_EPOCHS, dl_clean=clean_train_loader(tr_f, 500),
                        min_epochs=min(cfg.MIN_EPOCHS, cfg.CV_EPOCHS - 3))
        if not r["completed"]:
            print("  budget reached - checkpointed, resume next session")
            del m, r; free(); break
        mdl = unwrap(r["model"]).to(DEVICE)
        pv, yv = predict_frame(mdl, va_f); pt, yt = predict_frame(mdl, te_f)
        ptr, ytr = predict_frame(mdl, tr_f.sample(min(500, len(tr_f)), random_state=0))
        thr, _ = best_threshold(yv, pv, "acc")
        mv, mt_, mtr = (compute_metrics(yv, pv, thr), compute_metrics(yt, pt, thr),
                        compute_metrics(ytr, ptr, thr))
        row = {"fold": f + 1, "thr": round(thr, 4),
               "train_acc": round(mtr["acc"], 4),
               "val_acc": round(mv["acc"], 4), "val_f1": round(mv["f1"], 4),
               "val_auroc": round(mv["auroc"], 4),
               "test_acc": round(mt_["acc"], 4), "test_f1": round(mt_["f1"], 4),
               "test_auroc": round(mt_["auroc"], 4),
               "gap": round(mtr["acc"] - mv["acc"], 4)}
        cv_rows.append(row)
        cv_preds[f] = np.stack([pt, yt])
        STORE.put_json(f"obj/{key}.json", row)
        STORE.put_np(f"obj/{key}_pred.npy", cv_preds[f])
        STORE.mark(key)
        print(f"  fold {f+1} -> val acc {mv['acc']:.4f} | test acc {mt_['acc']:.4f} | "
              f"test AUROC {mt_['auroc']:.4f} | gap {row['gap']:+.4f}")
        del m, r, mdl; free()
else:
    print("  cross-validation disabled (cfg.RUN_CV = False)")

cv_tab = pd.DataFrame(cv_rows).round(4) if cv_rows else pd.DataFrame()
if len(cv_tab):
    print("\nCROSS-VALIDATION SUMMARY")
    print("=" * 104)
    print(cv_tab.to_string(index=False))
    print("-" * 104)
    for c in ["train_acc", "val_acc", "val_auroc", "test_acc", "test_auroc"]:
        print(f"  {c:<11}: {cv_tab[c].mean():.4f} +/- {cv_tab[c].std():.4f}")
    print(f"  {'gap':<11}: {cv_tab['gap'].mean():+.4f} +/- {cv_tab['gap'].std():.4f}"
          f"   (tolerance {cfg.GAP_TOL})")
    print("=" * 104)
    print("  v5 for comparison: test acc 0.846 +/- 0.013, test AUROC 0.927 +/- 0.008,")
    print("  mean gap +0.124. The AUROC column is the one that has to move.")
    savetab(cv_tab, "cv_folds"); STORE.put_df("obj/cv_folds.csv", cv_tab)

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    cv_tab.set_index("fold")[["train_acc","val_acc","test_acc"]].plot(kind="bar", ax=ax[0])
    ax[0].set_ylim(0.5, 1.02); ax[0].axhline(0.93, ls="--", color="k")
    ax[0].set_title("accuracy per fold")
    cv_tab.set_index("fold")[["val_auroc","test_auroc"]].plot(kind="bar", ax=ax[1])
    ax[1].set_ylim(0.5, 1.02); ax[1].set_title("AUROC per fold")
    cv_tab.set_index("fold")["gap"].plot(kind="bar", ax=ax[2], color="crimson")
    ax[2].axhline(cfg.GAP_TOL, ls="--", color="k"); ax[2].set_title("train-val gap per fold")
    for a in ax: a.tick_params(axis="x", rotation=0)
    fig.tight_layout(); savefig(fig, "11_cv")

    worst = cv_tab["gap"].max()
    print(f"\n  largest train-val gap across folds : {worst:+.4f}")
    print("  " + ("PASS - every fold is inside tolerance." if worst <= cfg.GAP_TOL + 0.02
                  else f"at least one fold exceeds the {cfg.GAP_TOL:.0%} tolerance."))
budget_banner()



>>> fold 1/5: train 2535 / val 423 / test 788
  ep   1/30 | loss 0.9351 | mt 0.679 | train 0.6440 | val 0.6194 | gap +0.0246 | AUROC 0.7026 | score 0.7026 | vram 0.5/16G  <-- best [warmup]
  ep   2/30 | loss 0.8647 | mt 0.552 | train 0.7160 | val 0.6832 | gap +0.0328 | AUROC 0.7505 | score 0.7505 | vram 0.5/16G  <-- best [warmup]
        backbone unfrozen at epoch 3 (LR x0.1 = 2e-05)
  ep   3/30 | loss 0.8601 | mt 0.421 | train 0.7040 | val 0.6785 | gap +0.0255 | AUROC 0.7397 | score 0.7397 | vram 0.5/16G [warmup]
  ep   4/30 | loss 0.8133 | mt 0.389 | train 0.7240 | val 0.7021 | gap +0.0219 | AUROC 0.7556 | score 0.7556 | vram 0.5/16G  <-- best [warmup]
  ep   5/30 | loss 0.8084 | mt 0.383 | train 0.7320 | val 0.7092 | gap +0.0228 | AUROC 0.7753 | score 0.7753 | vram 0.5/16G  <-- best [warmup]
  ep   6/30 | loss 0.7932 | mt 0.379 | train 0.7420 | val 0.7045 | gap +0.0375 | AUROC 0.7885 | score 0.7885 | vram 0.5/16G  <-- best [warmup]
  ep   7/30 | loss 0.7825 | mt 0.378 | train 0.754

In [28]:
# ============================================================================
# CELL 28 - BENCHMARKS  (same recipe, same schedule, same threshold procedure)
# ============================================================================
# If the proposed model got mixup, EMA and a calibrated threshold and the baselines did
# not, the table would be measuring the recipe rather than the architecture. Every row
# below runs through the identical train_model() call.
class SimpleNet(nn.Module):
    """Backbone + optional CBAM + linear head, with the same freeze/unfreeze API."""
    def __init__(self, fn, weights_enum, use_cbam=False, dropout=None, img_size=None):
        super().__init__()
        drop = cfg.DROPOUT if dropout is None else dropout
        w = None
        if cfg.PRETRAINED and weights_enum:
            try:    w = getattr(torchvision.models, weights_enum).IMAGENET1K_V1
            except Exception: w = None
        m = getattr(torchvision.models, fn)(weights=w)
        self.backbone = m.features if hasattr(m, "features") else nn.Sequential(
            *list(m.children())[:-2])
        self.frozen = False
        with torch.no_grad():
            self.backbone.eval()
            s = img_size or cfg.IMG_SIZE
            C = self.backbone(torch.zeros(1, 3, s, s)).shape[1]
        self.cbam = CBAM(C) if use_cbam else nn.Identity()
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(C, 1))
        self.last_fmap = None
    def set_frozen(self, flag):
        self.frozen = bool(flag)
        for p in self.backbone.parameters():
            p.requires_grad_(not self.frozen)
        if self.frozen: self.backbone.eval()
        return self
    def train(self, mode=True):
        super().train(mode)
        if self.frozen: self.backbone.eval()
        return self
    def forward(self, x_full, x_roi=None):
        f = self.cbam(self.backbone(x_full))
        self.last_fmap = f.detach() if not self.training else None
        return {"logit": self.head(f.mean((2, 3))).squeeze(-1),
                "aux": None, "gate": None, "seg": None, "mt": None}

BENCH = {
    "ResNet18 + CBAM": lambda: SimpleNet("resnet18",    "ResNet18_Weights",    True),
    "DenseNet121":     lambda: SimpleNet("densenet121", "DenseNet121_Weights", False),
    "EffNetV2-S":      lambda: SimpleNet("efficientnet_v2_s", "EfficientNet_V2_S_Weights", False),
}

bench_rows = []
if cfg.RUN_BENCH and MAIN_READY:
    for name, fn in BENCH.items():
        key = "bench_" + re.sub(r"[^a-z0-9]+", "_", name.lower())
        st  = stage_status(key, 500, f"benchmark {name}")
        if st == "skip":
            break
        if st == "done":
            bench_rows.append(STORE.get_json(f"obj/{key}.json")); continue
        print(f"\n>>> benchmark: {name}")
        set_seed()
        m = fn()
        print(f"    {sum(p.numel() for p in m.parameters())/1e6:.2f} M params")
        r = train_model(m, dl_tr, dl_va, key, epochs=cfg.BENCH_EPOCHS, dl_clean=dl_clean,
                        min_epochs=min(cfg.MIN_EPOCHS, cfg.BENCH_EPOCHS - 3))
        if not r["completed"]:
            print("  budget reached - checkpointed, resume next session")
            del m, r; free(); break
        mdl = unwrap(r["model"]).to(DEVICE)
        rr, pp, thr = evaluate_all_splits(mdl, {"train": tr_df, "val": va_df, "test": te_df})
        row = {"model": name, "params_M": round(sum(p.numel() for p in mdl.parameters())/1e6, 2),
               "train_acc": round(rr["train"]["acc"], 4), "val_acc": round(rr["val"]["acc"], 4),
               "test_acc": round(rr["test"]["acc"], 4), "test_f1": round(rr["test"]["f1"], 4),
               "test_auroc": round(rr["test"]["auroc"], 4),
               "test_auprc": round(rr["test"]["auprc"], 4),
               "gap": round(rr["train"]["acc"] - rr["val"]["acc"], 4)}
        bench_rows.append(row); STORE.put_json(f"obj/{key}.json", row); STORE.mark(key)
        STORE.put_np(f"obj/{key}_pred.npy", np.stack([pp["test"][0], pp["test"][1]]))
        print(f"  {name} -> test acc {row['test_acc']:.4f} | AUROC {row['test_auroc']:.4f}")
        del m, r, mdl; free()

bench_tab = pd.DataFrame(bench_rows) if bench_rows else pd.DataFrame()
if MAIN_READY:
    prop_row = {"model": "BT-CAGTNet v9 (proposed)",
                "params_M": round(n_params(model)/1e6, 2),
                "train_acc": round(res_A["train"]["acc"], 4),
                "val_acc": round(res_A["val"]["acc"], 4),
                "test_acc": round(res_A["test"]["acc"], 4),
                "test_f1": round(res_A["test"]["f1"], 4),
                "test_auroc": round(res_A["test"]["auroc"], 4),
                "test_auprc": round(res_A["test"]["auprc"], 4),
                "gap": round(res_A["train"]["acc"] - res_A["val"]["acc"], 4)}
    bench_tab = pd.concat([bench_tab, pd.DataFrame([prop_row])], ignore_index=True)
    if ens_res is not None:
        bench_tab = pd.concat([bench_tab, pd.DataFrame([{
            "model": "BT-CAGTNet v9 + ensemble", "params_M": float("nan"),
            "train_acc": round(ens_res["train"]["acc"], 4),
            "val_acc": round(ens_res["val"]["acc"], 4),
            "test_acc": round(ens_res["test"]["acc"], 4),
            "test_f1": round(ens_res["test"]["f1"], 4),
            "test_auroc": round(ens_res["test"]["auroc"], 4),
            "test_auprc": round(ens_res["test"]["auprc"], 4),
            "gap": round(ens_res["train"]["acc"] - ens_res["val"]["acc"], 4)}])],
            ignore_index=True)

if len(bench_tab):
    bench_tab = bench_tab.sort_values("test_auroc", ascending=False).reset_index(drop=True)
    print("\nBENCHMARK TABLE  (protocol A, identical recipe for every row)")
    print("=" * 116)
    print(bench_tab.to_string(index=False))
    print("=" * 116)
    savetab(bench_tab, "benchmarks"); STORE.put_df("obj/benchmarks.csv", bench_tab)
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.6))
    bench_tab.set_index("model")[["val_acc","test_acc"]].plot(kind="barh", ax=ax[0])
    ax[0].set_xlim(0.5, 1.0); ax[0].axvline(0.93, ls="--", color="k")
    ax[0].set_title("accuracy (dashed 0.93)")
    bench_tab.set_index("model")["test_auroc"].plot(kind="barh", ax=ax[1], color="teal")
    ax[1].set_xlim(0.5, 1.0); ax[1].set_title("test AUROC")
    fig.tight_layout(); savefig(fig, "12_benchmarks")
budget_banner()



>>> benchmark: ResNet18 + CBAM
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 194MB/s]


    11.21 M params
  ep   1/22 | loss 0.6978 | train 0.5258 | val 0.5434 | gap -0.0176 | AUROC 0.6238 | score 0.6238 | vram 1.1/16G  <-- best [warmup]
  ep   2/22 | loss 0.6908 | train 0.5267 | val 0.5471 | gap -0.0205 | AUROC 0.6267 | score 0.6267 | vram 0.5/16G  <-- best [warmup]
        backbone unfrozen at epoch 3 (LR x0.1 = 2e-05)
  ep   3/22 | loss 0.6714 | train 0.6958 | val 0.7061 | gap -0.0103 | AUROC 0.7721 | score 0.7721 | vram 1.2/16G  <-- best [warmup]
  ep   4/22 | loss 0.6348 | train 0.6825 | val 0.7061 | gap -0.0236 | AUROC 0.7903 | score 0.7903 | vram 0.8/16G  <-- best [warmup]
  ep   5/22 | loss 0.6265 | train 0.6950 | val 0.7153 | gap -0.0203 | AUROC 0.8071 | score 0.8071 | vram 0.8/16G  <-- best [warmup]
  ep   6/22 | loss 0.6186 | train 0.7142 | val 0.7135 | gap +0.0007 | AUROC 0.8199 | score 0.8199 | vram 0.8/16G  <-- best [warmup]
  ep   7/22 | loss 0.5950 | train 0.7225 | val 0.7246 | gap -0.0021 | AUROC 0.8241 | score 0.8241 | vram 0.8/16G  <-- best [warmup]
  

100%|██████████| 30.8M/30.8M [00:00<00:00, 202MB/s]


    6.95 M params
  ep   1/22 | loss 0.7108 | train 0.5067 | val 0.4917 | gap +0.0150 | AUROC 0.5486 | score 0.5486 | vram 0.9/16G  <-- best [warmup]
  ep   2/22 | loss 0.6980 | train 0.5667 | val 0.5305 | gap +0.0362 | AUROC 0.6140 | score 0.6140 | vram 0.5/16G  <-- best [warmup]
        backbone unfrozen at epoch 3 (LR x0.1 = 2e-05)
  ep   3/22 | loss 0.6741 | train 0.6825 | val 0.6784 | gap +0.0041 | AUROC 0.7403 | score 0.7403 | vram 2.2/16G  <-- best [warmup]
  ep   4/22 | loss 0.6473 | train 0.7125 | val 0.6969 | gap +0.0156 | AUROC 0.7680 | score 0.7680 | vram 1.9/16G  <-- best [warmup]
  ep   5/22 | loss 0.6339 | train 0.7075 | val 0.7098 | gap -0.0023 | AUROC 0.7902 | score 0.7902 | vram 1.9/16G  <-- best [warmup]
  ep   6/22 | loss 0.6127 | train 0.7267 | val 0.7209 | gap +0.0058 | AUROC 0.8074 | score 0.8074 | vram 1.9/16G  <-- best [warmup]
  ep   7/22 | loss 0.6127 | train 0.7308 | val 0.7338 | gap -0.0030 | AUROC 0.8132 | score 0.8132 | vram 1.9/16G  <-- best [warmup]
  e

100%|██████████| 82.7M/82.7M [00:00<00:00, 229MB/s]


    20.18 M params
  ep   1/22 | loss 0.6973 | train 0.5275 | val 0.4972 | gap +0.0303 | AUROC 0.4956 | score 0.4956 | vram 0.8/16G  <-- best [warmup]
  ep   2/22 | loss 0.6903 | train 0.5825 | val 0.5601 | gap +0.0224 | AUROC 0.5958 | score 0.5958 | vram 0.6/16G  <-- best [warmup]
        backbone unfrozen at epoch 3 (LR x0.1 = 2e-05)
  ep   3/22 | loss 0.6816 | train 0.6375 | val 0.6211 | gap +0.0164 | AUROC 0.7166 | score 0.7166 | vram 3.1/16G  <-- best [warmup]
  ep   4/22 | loss 0.6465 | train 0.6725 | val 0.6987 | gap -0.0262 | AUROC 0.7469 | score 0.7469 | vram 2.5/16G  <-- best [warmup]
  ep   5/22 | loss 0.6304 | train 0.6825 | val 0.7135 | gap -0.0310 | AUROC 0.7741 | score 0.7741 | vram 2.5/16G  <-- best [warmup]
  ep   6/22 | loss 0.6206 | train 0.7000 | val 0.7209 | gap -0.0209 | AUROC 0.7943 | score 0.7943 | vram 2.5/16G  <-- best [warmup]
  ep   7/22 | loss 0.6158 | train 0.7117 | val 0.7135 | gap -0.0018 | AUROC 0.8066 | score 0.8066 | vram 2.5/16G  <-- best [warmup]
  

In [29]:
# ============================================================================
# CELL 29 - ABLATIONS  (each row is one change from the v9.2 proposed default)
# ============================================================================
# Trimmed to the six rows that carry the argument, because the budget is one session.
# v9.2's default is SINGLE view (see Cell 1 - the dual-view head was what memorised
# the training set), so the ROI-view row now tests ADDING it back, not removing it -
# the interesting question is whether the capacity cut left AUROC on the table.
ABL = {
    "full v9 (v9.2 default)":    dict(),
    "- segmentation head":       dict(use_seg=False),
    "- multi-task heads":        dict(use_mt=False),
    "+ ROI view (dual scale)":   dict(use_roi=True),
    "- CBAM":                    dict(use_cbam=False),
    "- transformer":             dict(use_trans=False),
    "+ patch-graph branch":      dict(use_graph=True),
}
if not cfg.USE_SEG_HEAD:
    ABL.pop("- segmentation head", None)
if not (cfg.USE_MULTITASK and MT_DIM > 0):
    ABL.pop("- multi-task heads", None)

abl_rows = []
if cfg.RUN_ABLATION and MAIN_READY:
    for name, kw in ABL.items():
        key = "abl_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
        st  = stage_status(key, 500, f"ablation {name}")
        if st == "skip":
            break
        if st == "done":
            abl_rows.append(STORE.get_json(f"obj/{key}.json")); continue
        print(f"\n>>> ablation: {name}")
        set_seed()
        m = build_proposed(**kw)
        # An ablation that removes a view must also stop the loader producing it, or the
        # comparison silently measures a different data pipeline as well.
        _tr = make_loader(tr_df, train=True, roi=kw.get("use_roi", cfg.USE_ROI_VIEW),
                          seg=kw.get("use_seg", cfg.USE_SEG_HEAD))
        _va = make_loader(va_df, roi=kw.get("use_roi", cfg.USE_ROI_VIEW))
        r = train_model(m, _tr, _va, key, epochs=cfg.ABL_EPOCHS,
                        dl_clean=clean_train_loader(tr_df, 500),
                        min_epochs=min(cfg.MIN_EPOCHS, cfg.ABL_EPOCHS - 3))
        if not r["completed"]:
            print("  budget reached - checkpointed, resume next session")
            del m, r; free(); break
        mdl = unwrap(r["model"]).to(DEVICE)
        _fr = {"train": tr_df, "val": va_df, "test": te_df}
        preds = {k: predict_frame(mdl, fr, roi=kw.get("use_roi", cfg.USE_ROI_VIEW)) for k, fr in _fr.items()}
        thr, _ = best_threshold(preds["val"][1], preds["val"][0], "acc")
        rr = {k: compute_metrics(y, p, thr) for k, (p, y) in preds.items()}
        row = {"variant": name, "params_M": round(n_params(mdl)/1e6, 2),
               "val_acc": round(rr["val"]["acc"], 4),
               "test_acc": round(rr["test"]["acc"], 4),
               "test_f1": round(rr["test"]["f1"], 4),
               "test_auroc": round(rr["test"]["auroc"], 4),
               "gap": round(rr["train"]["acc"] - rr["val"]["acc"], 4)}
        abl_rows.append(row); STORE.put_json(f"obj/{key}.json", row); STORE.mark(key)
        print(f"  {name} -> test acc {row['test_acc']:.4f} | AUROC {row['test_auroc']:.4f}")
        del m, r, mdl, _tr, _va; free()
else:
    print("  ablation disabled or main model unavailable")

abl_tab = pd.DataFrame(abl_rows) if abl_rows else pd.DataFrame()
_BASE = "full v9 (v9.2 default)"
if len(abl_tab):
    base = abl_tab[abl_tab.variant == _BASE]
    if len(base):
        b = float(base.test_auroc.iloc[0])
        abl_tab["d_auroc"] = (abl_tab.test_auroc - b).round(4)
    print("\nABLATION TABLE  (protocol A, identical recipe, "
          f"{cfg.ABL_EPOCHS} epochs each)")
    print("=" * 110)
    print(abl_tab.to_string(index=False))
    print("=" * 110)
    print("  d_auroc is the change from the v9.2 default (single view). A NEGATIVE value on")
    print("  '- segmentation head' means mask supervision was doing work. '+ ROI view (dual")
    print("  scale)' tests whether the capacity cut in Cell 1 left AUROC on the table - if it")
    print("  is not clearly positive, the single-view default is the right call to keep.")
    savetab(abl_tab, "ablations"); STORE.put_df("obj/ablations.csv", abl_tab)
    fig, ax = plt.subplots(figsize=(9, 4.4))
    d = abl_tab.set_index("variant")["test_auroc"].sort_values()
    d.plot(kind="barh", ax=ax, color=["crimson" if v < d.get(_BASE, 0) else "teal"
                                      for v in d.values])
    ax.axvline(d.get(_BASE, np.nan), ls="--", color="k", label=_BASE)
    ax.set_xlim(max(0.5, d.min() - 0.05), min(1.0, d.max() + 0.02))
    ax.set_title("ablation: test AUROC"); ax.legend(fontsize=8)
    fig.tight_layout(); savefig(fig, "13_ablations")
budget_banner()



>>> ablation: full v9 (v9.2 default)
  ep   1/22 | loss 0.9285 | mt 0.680 | train 0.6680 | val 0.6414 | gap +0.0266 | AUROC 0.7427 | score 0.7427 | vram 0.5/16G  <-- best [warmup]
  ep   2/22 | loss 0.8611 | mt 0.556 | train 0.7240 | val 0.7227 | gap +0.0013 | AUROC 0.8106 | score 0.8106 | vram 0.5/16G  <-- best [warmup]
        backbone unfrozen at epoch 3 (LR x0.1 = 2e-05)
  ep   3/22 | loss 0.8349 | mt 0.406 | train 0.7260 | val 0.7486 | gap -0.0226 | AUROC 0.8166 | score 0.8166 | vram 0.5/16G  <-- best [warmup]
  ep   4/22 | loss 0.8191 | mt 0.378 | train 0.7380 | val 0.7560 | gap -0.0180 | AUROC 0.8284 | score 0.8284 | vram 0.5/16G  <-- best [warmup]
  ep   5/22 | loss 0.7882 | mt 0.374 | train 0.7760 | val 0.7671 | gap +0.0089 | AUROC 0.8290 | score 0.8290 | vram 0.5/16G  <-- best [warmup]
  ep   6/22 | loss 0.7902 | mt 0.377 | train 0.7880 | val 0.7652 | gap +0.0228 | AUROC 0.8284 | score 0.8284 | vram 0.5/16G [warmup]
  ep   7/22 | loss 0.7735 | mt 0.371 | train 0.7960 | val 0

In [30]:
# ============================================================================
# CELL 30 - EXPLAINABILITY: Grad-CAM, CBAM attention, and the seg map
# ============================================================================
# With the mask supervision on, XAI is no longer only qualitative: the Grad-CAM peak can
# be checked AGAINST the annotated lesion, so "the model looks at the right place" becomes
# a number (pointing game hit rate) rather than a picture the reader has to take on trust.
def grad_cam(mdl, xf, xr):
    mdl = mdl.to(DEVICE).eval()
    feats, grads = {}, {}
    target = mdl.cbam if getattr(mdl, "use_cbam", False) else mdl.backbone
    def fh(_, __, o): feats["v"] = o
    def bh(_, gi, go): grads["v"] = go[0]
    h1 = target.register_forward_hook(fh)
    h2 = target.register_full_backward_hook(bh)
    try:
        with torch.enable_grad():
            out = mdl(xf.to(DEVICE), xr.to(DEVICE))
            lg  = out["logit"] if isinstance(out, dict) else out
            mdl.zero_grad(set_to_none=True)
            lg.sum().backward()
        if "v" not in feats or "v" not in grads:
            # can happen if the hooked module is an nn.Identity (CBAM ablated off)
            return (np.zeros((xf.shape[0], 8, 8), np.float32),
                    torch.sigmoid(lg.float()).detach().cpu().numpy())
        f, g = feats["v"].detach().float(), grads["v"].detach().float()
        w = g.mean((2, 3), keepdim=True)
        cam = F.relu((w * f).sum(1, keepdim=True))
        cam = cam / cam.amax((2, 3), keepdim=True).clamp(min=1e-6)
        return cam[:, 0].cpu().numpy(), torch.sigmoid(lg.float()).detach().cpu().numpy()
    finally:
        h1.remove(); h2.remove()

if cfg.RUN_XAI and MAIN_READY and have_time(300, "XAI"):
    core = unwrap(model).to(DEVICE).eval()
    pool = te_df[te_df.has_annot] if te_df.has_annot.any() else te_df
    sub  = pool.sample(min(cfg.XAI_N, len(pool)), random_state=cfg.SEED).reset_index(drop=True)
    dl_x = make_loader(sub, bs=len(sub))
    b    = next(iter(dl_x))
    cam, prob = grad_cam(core, b["x_full"], b["x_roi"])
    with torch.no_grad():
        o = core(b["x_full"].to(DEVICE), b["x_roi"].to(DEVICE))
    segm = (torch.sigmoid(o["seg"].float()).cpu().numpy()[:, 0]
            if o.get("seg") is not None else None)
    cba  = (core.cbam.last_spatial.float().cpu().numpy()[:, 0]
            if getattr(core, "use_cbam", False) and core.cbam.last_spatial is not None else None)

    m_, sd_ = current_norm()
    nrow = 2 + (segm is not None) + (cba is not None)
    fig, ax = plt.subplots(nrow, len(sub), figsize=(2.7*len(sub), 2.9*nrow), squeeze=False)
    hits, tried = 0, 0
    for j in range(len(sub)):
        im = np.clip(b["x_full"][j][0].numpy() * sd_[0] + m_[0], 0, 1)
        gt = b["seg"][j][0].numpy() if float(b["seg_valid"][j]) > 0 else None
        r = 0
        ax[r,j].imshow(im, cmap="gray")
        ax[r,j].set_title(f"y={int(b['y'][j])}  p={prob[j]:.2f}", fontsize=8); r += 1
        ax[r,j].imshow(im, cmap="gray")
        ax[r,j].imshow(cv2.resize(cam[j], im.shape[::-1]), cmap="jet", alpha=.42)
        ax[r,j].set_title("Grad-CAM", fontsize=8); r += 1
        if cba is not None:
            ax[r,j].imshow(cv2.resize(cba[j], im.shape[::-1]), cmap="viridis")
            ax[r,j].set_title("CBAM spatial", fontsize=8); r += 1
        if segm is not None:
            ax[r,j].imshow(cv2.resize(segm[j], im.shape[::-1]), cmap="magma", vmin=0, vmax=1)
            ax[r,j].set_title("seg head", fontsize=8); r += 1
        # pointing game: does the CAM peak land inside the annotated lesion?
        if gt is not None and gt.max() > 0.5 and int(b["y"][j]) == 1:
            cg = cv2.resize(cam[j], gt.shape[::-1])
            yy, xx = np.unravel_index(int(np.argmax(cg)), cg.shape)
            tried += 1; hits += int(gt[yy, xx] > 0.5)
            for rr in range(nrow):
                ax[rr,j].contour(cv2.resize(gt, im.shape[::-1]), levels=[0.5],
                                 colors="lime", linewidths=1.1)
        for rr in range(nrow):
            ax[rr,j].axis("off")
    fig.suptitle("Explainability - green contour is the annotated lesion", fontweight="bold")
    fig.tight_layout(); savefig(fig, "14_xai")

    print("\nEXPLAINABILITY")
    print("-" * 84)
    print(f"  images shown           : {len(sub)}  (annotated positives preferred)")
    if tried:
        print(f"  pointing game          : Grad-CAM peak inside the lesion in {hits}/{tried} "
              f"({100*hits/max(tried,1):.0f}%)")
        print("  This is a small sample and is reported as a diagnostic, not a claim. It is")
        print("  only measurable at all because the annotations are loaded - without them")
        print("  the XAI section can show pictures but cannot check them against anything.")
    else:
        print("  pointing game          : no annotated positive in the sample - not measurable")
    print("-" * 84)
    free()
else:
    print("  XAI skipped")
budget_banner()


  figure saved -> /kaggle/working/outputs/figures/14_xai.png

EXPLAINABILITY
------------------------------------------------------------------------------------
  images shown           : 8  (annotated positives preferred)
  pointing game          : no annotated positive in the sample - not measurable
------------------------------------------------------------------------------------
  [budget] elapsed 4.94 h / 10.6 h  | remaining 5.66 h


In [31]:
# ============================================================================
# CELL 31 - STATISTICS: bootstrap CIs, DeLong, McNemar
# ============================================================================
def boot_ci(y, p, thr, fn, n=None, seed=0):
    n = n or cfg.BOOT_N
    rng = np.random.default_rng(seed)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) < 2:
            continue
        try:    vals.append(fn(y[i], p[i], thr))
        except Exception: pass
    if not vals:
        return (float("nan"),) * 3
    v = np.array(vals)
    return float(v.mean()), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def _auc(y, p, thr=None): return roc_auc_score(y, p)
def _acc(y, p, thr):      return accuracy_score(y, (p >= thr).astype(int))
def _f1(y, p, thr):       return f1_score(y, (p >= thr).astype(int), zero_division=0)

def delong_p(y, p1, p2):
    """Two-sided DeLong test for two correlated ROC curves on the same sample."""
    y = np.asarray(y).astype(int)
    pos, neg = y == 1, y == 0
    m, n = int(pos.sum()), int(neg.sum())
    if m < 2 or n < 2:
        return float("nan"), float("nan")
    def _struct(p):
        X, Y = p[pos], p[neg]
        v01 = np.array([( (Y < x).sum() + 0.5*(Y == x).sum() ) / n for x in X])
        v10 = np.array([( (X > y_).sum() + 0.5*(X == y_).sum() ) / m for y_ in Y])
        return v01, v10, float(v01.mean())
    a01, a10, A = _struct(np.asarray(p1))
    b01, b10, B = _struct(np.asarray(p2))
    S01 = np.cov(np.vstack([a01, b01])); S10 = np.cov(np.vstack([a10, b10]))
    S = S01 / m + S10 / n
    var = S[0,0] + S[1,1] - 2*S[0,1]
    if var <= 0:
        return A - B, float("nan")
    z = (A - B) / math.sqrt(var)
    return A - B, float(2 * (1 - sps.norm.cdf(abs(z))))

def mcnemar_p(y, p1, p2, thr1, thr2):
    a = (p1 >= thr1).astype(int) == y
    b = (p2 >= thr2).astype(int) == y
    n01 = int((~a & b).sum()); n10 = int((a & ~b).sum())
    if n01 + n10 == 0:
        return n01, n10, 1.0
    return n01, n10, float(sps.binomtest(n10, n01 + n10, 0.5).pvalue)

stat_tab = None
if cfg.RUN_STATS and MAIN_READY:
    y_t, p_t = preds_A["test"][1], preds_A["test"][0]
    rows = []
    for nm, fn in [("accuracy", _acc), ("F1", _f1), ("AUROC", _auc)]:
        mu, lo, hi = boot_ci(y_t, p_t, THR, fn)
        rows.append({"metric": nm, "estimate": round(float(
            fn(y_t, p_t, THR)), 4), "boot_mean": round(mu, 4),
            "ci_low": round(lo, 4), "ci_high": round(hi, 4)})
    stat_tab = pd.DataFrame(rows)
    print(f"BOOTSTRAP 95% CONFIDENCE INTERVALS  (test split, {cfg.BOOT_N} resamples)")
    print("=" * 84)
    print(stat_tab.to_string(index=False))
    print("=" * 84)
    savetab(stat_tab, "bootstrap_ci"); STORE.put_df("obj/bootstrap_ci.csv", stat_tab)

    comp = []
    for key, label in [(f"bench_{re.sub(r'[^a-z0-9]+','_',k.lower())}", k) for k in BENCH]:
        f = STORE.root / f"obj/{key}_pred.npy"
        if not f.exists():
            continue
        pb, yb = np.load(f)
        if len(yb) != len(y_t) or not np.allclose(yb, y_t):
            continue
        d, pv = delong_p(y_t, p_t, pb)
        n01, n10, pm = mcnemar_p(y_t.astype(int), p_t, pb, THR, 0.5)
        comp.append({"comparator": label, "delta_auroc": round(d, 4),
                     "delong_p": (round(pv, 5) if pv == pv else float("nan")),
                     "mcnemar_b": n01, "mcnemar_c": n10, "mcnemar_p": round(pm, 5)})
    if ens_res is not None and (STORE.root / "obj/pred_ens_test.npy").exists():
        pe, ye = STORE.get_np("obj/pred_ens_test.npy")
        d, pv = delong_p(y_t, pe, p_t)
        comp.append({"comparator": "single model (vs ensemble)", "delta_auroc": round(d, 4),
                     "delong_p": (round(pv, 5) if pv == pv else float("nan")),
                     "mcnemar_b": np.nan, "mcnemar_c": np.nan, "mcnemar_p": np.nan})
    if comp:
        comp_tab = pd.DataFrame(comp)
        print("\nPAIRED SIGNIFICANCE TESTS vs the proposed model (test split)")
        print("=" * 100)
        print(comp_tab.to_string(index=False))
        print("=" * 100)
        print("  delta_auroc > 0 favours the proposed model. DeLong compares the two ROC")
        print("  curves on the same sample; McNemar compares the two decision vectors at")
        print("  each model's own calibrated threshold.")
        savetab(comp_tab, "significance"); STORE.put_df("obj/significance.csv", comp_tab)
else:
    print("  statistics skipped")
budget_banner()


BOOTSTRAP 95% CONFIDENCE INTERVALS  (test split, 1000 resamples)
  metric  estimate  boot_mean  ci_low  ci_high
accuracy    0.8115     0.8114  0.7769   0.8442
      F1    0.8192     0.8187  0.7821   0.8522
   AUROC    0.9064     0.9066  0.8803   0.9307
  table saved  -> /kaggle/working/outputs/tables/bootstrap_ci.csv

PAIRED SIGNIFICANCE TESTS vs the proposed model (test split)
                comparator  delta_auroc  delong_p  mcnemar_b  mcnemar_c  mcnemar_p
           ResNet18 + CBAM       0.0202   0.06731       36.0       48.0    0.22986
               DenseNet121       0.0278   0.01089       26.0       61.0    0.00022
                EffNetV2-S       0.0172   0.13830       38.0       71.0    0.00203
single model (vs ensemble)       0.0235   0.00010        NaN        NaN        NaN
  delta_auroc > 0 favours the proposed model. DeLong compares the two ROC
  curves on the same sample; McNemar compares the two decision vectors at
  each model's own calibrated threshold.
  table saved  

In [32]:
# ============================================================================
# CELL 32 - FINAL REPORT AND ARTIFACT BUNDLE
# ============================================================================
R = []; A = R.append
A("=" * 100)
A("  BT-CAGTNet v9 - FINAL REPORT")
A("  Dual-scale CBAM-CNN + Transformer, adaptive gated fusion, lesion-mask supervision")
A("=" * 100)
A("")
A(f"  dataset          : {len(df)} radiographs, {df.group_id.nunique()} case groups "
  f"({'dHash subdivision' if cfg.REGROUP_NEG else 'metadata grouping'})")
A(f"  task             : binary '{cfg.TARGET}' (tumour vs no-tumour)")
A(f"  backbone         : {cfg.BACKBONE} | input {cfg.IMG_SIZE} px | cache {cfg.CACHE_SIZE} px")
A(f"  views            : {'full + label-free bone ROI (shared backbone)' if cfg.USE_ROI_VIEW else 'full only'}")
A(f"  annotations      : {ANNOT_KIND} ({len(ANNOT)} images) | "
  f"seg head {'ON' if cfg.USE_SEG_HEAD else 'off'} | "
  f"multi-task {'ON' if (cfg.USE_MULTITASK and MT_DIM) else 'off'}")
if MAIN_READY:
    A(f"  branches         : {unwrap(model).names} | fusion {cfg.FUSION}")
    A(f"  parameters       : {n_params(model)/1e6:.2f} M")
A(f"  session elapsed  : {elapsed_h():.2f} h of a {cfg.TIME_BUDGET_H:g} h budget")
A("")

if MAIN_READY:
    A("-" * 100)
    A("  RESULT 1 - PROTOCOL A (case-level split, leakage-safe)   << HEADLINE >>")
    A("-" * 100)
    for k in ["train", "val", "test"]:
        m = res_A[k]
        A(f"    {k:<6} acc {m['acc']:.4f} | bacc {m['bacc']:.4f} | f1 {m['f1']:.4f} | "
          f"sens {m['rec']:.4f} | spec {m['spec']:.4f} | AUROC {m['auroc']:.4f}")
    A(f"    threshold {THR:.4f}, fitted on validation only")
    A(f"    train-val gap {res_A['train']['acc']-res_A['val']['acc']:+.4f}")
    for lo in (0.90, 0.93, 0.95):
        A(f"    >= {lo:.2f} on all three splits : "
          f"{'YES' if all(res_A[k]['acc'] >= lo for k in ['train','val','test']) else 'no'}")
    A("")

if ens_res is not None:
    A("-" * 100)
    A("  RESULT 2 - ENSEMBLE (protocol A)")
    A("-" * 100)
    for k in ["train", "val", "test"]:
        m = ens_res[k]
        A(f"    {k:<6} acc {m['acc']:.4f} | f1 {m['f1']:.4f} | AUROC {m['auroc']:.4f}")
    A("")

if res_B is not None:
    A("-" * 100)
    A("  RESULT 3 - PROTOCOL B (image-level split, the official BTXRD protocol)")
    A("-" * 100)
    for k in ["train", "val", "test"]:
        m = res_B[k]
        A(f"    {k:<6} acc {m['acc']:.4f} | f1 {m['f1']:.4f} | AUROC {m['auroc']:.4f}")
    if MAIN_READY:
        A(f"    leakage premium on test accuracy : "
          f"{res_B['test']['acc'] - res_A['test']['acc']:+.4f}")
        A(f"    {img_span} of {len(df)} images ({100*img_span/len(df):.1f}%) sit in case groups")
        A("    that span splits under this protocol. Comparable with published BTXRD numbers,")
        A("    NOT comparable with protocol A, and never to be quoted as leakage-safe.")
    A("")

if len(cv_tab):
    A("-" * 100)
    A(f"  RESULT 4 - {len(cv_tab)}-FOLD GROUP CROSS-VALIDATION (protocol A)")
    A("-" * 100)
    A(f"    val  acc   {cv_tab.val_acc.mean():.4f} +/- {cv_tab.val_acc.std():.4f}")
    A(f"    test acc   {cv_tab.test_acc.mean():.4f} +/- {cv_tab.test_acc.std():.4f}")
    A(f"    test AUROC {cv_tab.test_auroc.mean():.4f} +/- {cv_tab.test_auroc.std():.4f}")
    A(f"    mean gap   {cv_tab.gap.mean():+.4f}")
    A(f"    v5 baseline: test acc 0.846 +/- 0.013, test AUROC 0.927 +/- 0.008, gap +0.124")
    A("")

if len(bench_tab):
    A("-" * 100); A("  RESULT 5 - BENCHMARKS (protocol A, identical recipe)"); A("-" * 100)
    for _, r_ in bench_tab.iterrows():
        A(f"    {r_['model']:<28} test acc {r_['test_acc']:.4f} | AUROC {r_['test_auroc']:.4f} "
          f"| gap {r_['gap']:+.4f}")
    A("")

if len(abl_tab):
    A("-" * 100); A("  RESULT 6 - ABLATIONS"); A("-" * 100)
    for _, r_ in abl_tab.iterrows():
        d = r_.get("d_auroc", float("nan"))
        A(f"    {r_['variant']:<28} test AUROC {r_['test_auroc']:.4f}"
          + (f"  ({d:+.4f})" if d == d else ""))
    A("")

if stat_tab is not None:
    A("-" * 100); A("  RESULT 7 - BOOTSTRAP 95% CI (test split)"); A("-" * 100)
    for _, r_ in stat_tab.iterrows():
        A(f"    {r_['metric']:<10} {r_['estimate']:.4f}  "
          f"[{r_['ci_low']:.4f}, {r_['ci_high']:.4f}]")
    A("")

A("-" * 100); A("  THE FIVE DEFECTS FIXED, AND THE ONE REPORTED"); A("-" * 100)
A("    F1  early stopping fired on a gap-penalised score that flattens while AUROC climbs.")
A("        v8 stopped at ep 22 and v6 at ep 26 - each on its own AUROC MAXIMUM.")
A(f"        v9 stops on raw val AUROC, patience {cfg.PATIENCE}, armed after epoch {cfg.MIN_EPOCHS}.")
A(f"    F2  LR 1e-4 x 0.05 = 5e-6 on the backbone. v9 runs {cfg.LR:g} x {cfg.BACKBONE_LR_MULT:g} "
  f"= {cfg.LR*cfg.BACKBONE_LR_MULT:.2g}.")
A("    F3  the gap was being closed by pushing train DOWN, which took val with it.")
A("        v9 closes it by lifting val: mask supervision, multi-task heads, more signal.")
A("    F4  CutMix and constant-fill erasing injected label noise on a localised lesion.")
A("    F5  the EMA warmup reached d=0.81 after 40 epochs - a 5-step window, i.e. no EMA.")
A(f"    F6  negative case groups over-merge: group-level positive rate {am['posrate']:.2f}")
A(f"        against an image-level {df[cfg.TARGET].mean():.2f}. MEASURED AND REPORTED, not")
A("        silently corrected. Subdividing them by perceptual hash would split real")
A("        multi-view cases apart and CAN leak; over-merging only costs sample size.")
A("")
A("-" * 100); A("  v9.2 - THE HEAD WAS OVERFITTING, NOT (ONLY) THE BACKBONE"); A("-" * 100)
A("    The first v9 run hit train acc 1.0000 by epoch 2 WHILE THE BACKBONE WAS STILL")
A("    FROZEN - proof the 4-branch dual-view head, not backbone fine-tuning, was doing")
A("    the memorising. Fixed structurally, not just with more dropout:")
A(f"      - single view by default (2 branches, not 4) - USE_ROI_VIEW = {cfg.USE_ROI_VIEW}")
A(f"      - embedding/transformer dim cut {160} -> {cfg.EMB_DIM}, heads {4} -> {cfg.TRANS_HEADS}")
A(f"      - dedicated {cfg.HEAD_DROPOUT:g} dropout on the final classifier only")
A(f"      - WD {2e-2:g} -> {5e-2:g}, dropout/droppath/aug raised")
A("    The dual-view ROI branch is kept as an ablation row ('+ ROI view (dual scale)')")
A("    rather than deleted, so the report can show whether it is worth its capacity cost.")
A("")
A("-" * 100); A("  v9.3 - HARD AUGMENTATION (data-level fix for head memorisation)"); A("-" * 100)
A("    Even after the v9.2 capacity cut, the head memorised training images in a single")
A("    epoch. Root cause: the augmentation pipeline was not diverse enough to prevent")
A(f"    the ~400 K-parameter head from converging to {len(tr_df)} training images in one pass.")
A("    v9.3 fixes this at the DATA level with a fundamentally stronger augmentation:")
A(f"      backbone       : effnetv2s (20.2 M) -> effnetb0 (5.3 M), same 1280-ch output")
A(f"      AUG_STRENGTH   : 1.35 -> {cfg.AUG_STRENGTH}")
A("      CLAHE          : random clip 1.5-4.0, 8x8 grid — X-ray standard preprocessing")
A("      elastic deform : random Gaussian displacement field (40% per image per epoch)")
A("      perspective    : random corner displacement, simulates off-axis acquisition (35%)")
A(f"      crop scale     : floor 0.55 -> 0.30 (3x more spatial variety at full strength)")
A("      solarize       : random threshold pixel inversion — breaks intensity memorisation")
A("      motion blur    : horizontal / vertical — simulates acquisition motion")
A("      noise          : Gaussian + Poisson (quantum noise) + salt-and-pepper")
A(f"      grid shuffle   : 4x4 patch permutation (25% prob) — destroys texture layouts")
A(f"      erasing        : up to {cfg.ERASE_MAX_N} noise-filled patches <= {cfg.ERASE_MAX_FRAC:.0%}")
A(f"      consistency    : KL(view1, view2) dual-view regularisation (weight {getattr(cfg,'CONSIST_W',0.0)})")
A(f"      WD {5e-2:g} -> {cfg.WD:g}, HEAD_DROPOUT 0.50 -> {cfg.HEAD_DROPOUT:g}, FREEZE_EPOCHS {4} -> {cfg.FREEZE_EPOCHS}")
A("")
A("-" * 100); A("  LIMITATIONS, STATED"); A("-" * 100)
A("    - Case groups are RECONSTRUCTED. BTXRD ships no patient identifier, so protocol A")
A("      is a best-effort reconstruction, not ground truth. It errs toward over-merging,")
A("      which costs data efficiency and can never create leakage.")
A("    - Normal radiographs carry no bone-site metadata, so their group key is only")
A("      (centre, age, sex) and over-merges unrelated patients. See F6.")
A("    - The segmentation head is supervised only where an annotation exists; its IoU is")
A("      a diagnostic that the auxiliary loss did something, not a segmentation result.")
A("    - Protocol B is reported because it is the official BTXRD protocol and makes the")
A("      numbers comparable. It is not leakage-safe and is never presented as such.")
A("")
if PENDING:
    A("-" * 100); A("  PENDING - the budget ran out before these stages"); A("-" * 100)
    for s in PENDING:
        A(f"    - {s}")
    A("    Save this version, attach its output to a new session, Run All to continue.")
    A("")
A("=" * 100)

report = "\n".join(R)
print(report)
Path(f"{cfg.OUT_DIR}/REPORT.txt").write_text(report)

bundle = f"{cfg.OUT_DIR}/btcagtnet_v9_artifacts.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for p in Path(cfg.OUT_DIR).rglob("*"):
        if p.is_file() and p.name != Path(bundle).name:
            z.write(p, p.relative_to(cfg.OUT_DIR))
print(f"\n  report saved  -> {cfg.OUT_DIR}/REPORT.txt")
print(f"  bundle saved  -> {bundle}")
STORE.mark("report")
budget_banner()


  BT-CAGTNet v9 - FINAL REPORT
  Dual-scale CBAM-CNN + Transformer, adaptive gated fusion, lesion-mask supervision

  dataset          : 3746 radiographs, 1318 case groups (metadata grouping)
  task             : binary 'tumor' (tumour vs no-tumour)
  backbone         : effnetb0 | input 384 px | cache 448 px
  views            : full only
  annotations      : none (0 images) | seg head off | multi-task ON
  branches         : ['cnn_full', 'trans_full'] | fusion gated
  parameters       : 4.99 M
  session elapsed  : 4.94 h of a 10.6 h budget

----------------------------------------------------------------------------------------------------
  RESULT 1 - PROTOCOL A (case-level split, leakage-safe)   << HEADLINE >>
----------------------------------------------------------------------------------------------------
    train  acc 0.8745 | bacc 0.8745 | f1 0.8740 | sens 0.8783 | spec 0.8708 | AUROC 0.9493
    val    acc 0.8318 | bacc 0.8314 | f1 0.8378 | sens 0.8545 | spec 0.8083 | AUROC 0

In [33]:
# ============================================================================
# FINAL CELL - BUNDLE EVERYTHING INTO ONE DOWNLOADABLE ZIP
# ============================================================================
import zipfile, os, shutil
from pathlib import Path
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
bundle_path = f"/kaggle/working/btcagtnet_v9_DOWNLOAD_{timestamp}.zip"

OUT   = Path(cfg.OUT_DIR)   # /kaggle/working/outputs
STATE = Path(cfg.STATE_DIR) # /kaggle/working/state

print("=" * 70)
print("  BUILDING DOWNLOAD BUNDLE")
print("=" * 70)

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:

    # ── 1. REPORT ────────────────────────────────────────────────────────────
    rpt = OUT / "REPORT.txt"
    if rpt.exists():
        z.write(rpt, "REPORT.txt")
        print(f"  [report]   REPORT.txt")

    # ── 2. FIGURES ───────────────────────────────────────────────────────────
    fig_dir = OUT / "figures"
    if fig_dir.exists():
        figs = sorted(fig_dir.rglob("*"))
        for p in figs:
            if p.is_file():
                arcname = f"figures/{p.relative_to(fig_dir)}"
                z.write(p, arcname)
                print(f"  [figure]   {arcname}")

    # ── 3. TABLES (CSV) ──────────────────────────────────────────────────────
    tbl_dir = OUT / "tables"
    if tbl_dir.exists():
        for p in sorted(tbl_dir.rglob("*.csv")):
            arcname = f"tables/{p.relative_to(tbl_dir)}"
            z.write(p, arcname)
            print(f"  [table]    {arcname}")

    # ── 4. MODEL WEIGHTS (.pth / .pt in outputs/models) ─────────────────────
    mdl_dir = OUT / "models"
    if mdl_dir.exists():
        for p in sorted(mdl_dir.rglob("*.pt")) + sorted(mdl_dir.rglob("*.pth")):
            arcname = f"models/{p.name}"
            z.write(p, arcname)
            sz = p.stat().st_size / 1e6
            print(f"  [model]    {arcname}  ({sz:.1f} MB)")

    # ── 5. CHECKPOINTS from STATE_DIR (best + ema + resume) ─────────────────
    ckpt_dir = STATE / "ckpt"
    if ckpt_dir.exists():
        for p in sorted(ckpt_dir.glob("*.pt")):
            arcname = f"checkpoints/{p.name}"
            z.write(p, arcname)
            sz = p.stat().st_size / 1e6
            print(f"  [ckpt]     {arcname}  ({sz:.1f} MB)")

    # ── 6. DONE.json (keeps state resumable in next session) ─────────────────
    done_f = STATE / "DONE.json"
    if done_f.exists():
        z.write(done_f, "checkpoints/DONE.json")
        print(f"  [state]    checkpoints/DONE.json")

    # ── 7. TRAINING HISTORY CSV (from STORE obj/) ────────────────────────────
    obj_dir = STATE / "obj"
    if obj_dir.exists():
        for p in sorted(obj_dir.rglob("*.csv")):
            arcname = f"history/{p.name}"
            z.write(p, arcname)
            print(f"  [history]  {arcname}")
        for p in sorted(obj_dir.rglob("*.json")):
            arcname = f"history/{p.name}"
            z.write(p, arcname)
            print(f"  [history]  {arcname}")

    # ── 8. XAI / GradCAM images ──────────────────────────────────────────────
    xai_dir = OUT / "xai"
    if xai_dir.exists():
        for p in sorted(xai_dir.rglob("*")):
            if p.is_file():
                arcname = f"xai/{p.relative_to(xai_dir)}"
                z.write(p, arcname)
                print(f"  [xai]      {arcname}")

# ── summary ──────────────────────────────────────────────────────────────────
bundle_mb = Path(bundle_path).stat().st_size / 1e6
print("=" * 70)
print(f"  ZIP SIZE  : {bundle_mb:.1f} MB")
print(f"  SAVED TO  : {bundle_path}")
print()
print("  HOW TO DOWNLOAD:")
print("  -> Kaggle sidebar  : Output tab (right panel) -> click the zip file")
print("  -> Kaggle menu     : Data -> Output -> Download")
print("=" * 70)

  BUILDING DOWNLOAD BUNDLE
  [report]   REPORT.txt
  [figure]   figures/01_eda.png
  [figure]   figures/02_preprocessing_roi.png
  [figure]   figures/03_grouping.png
  [figure]   figures/04_split.png
  [figure]   figures/05_augmentation.png
  [figure]   figures/06_training.png
  [figure]   figures/07_curves.png
  [figure]   figures/08_gate.png
  [figure]   figures/10_protocol_comparison.png
  [figure]   figures/11_cv.png
  [figure]   figures/12_benchmarks.png
  [figure]   figures/13_ablations.png
  [figure]   figures/14_xai.png
  [table]    tables/ablations.csv
  [table]    tables/benchmarks.csv
  [table]    tables/bootstrap_ci.csv
  [table]    tables/cv_folds.csv
  [table]    tables/ensemble_protocolA.csv
  [table]    tables/gate_weights.csv
  [table]    tables/grouping_summary.csv
  [table]    tables/proposed_protocolA.csv
  [table]    tables/protocol_comparison.csv
  [table]    tables/significance.csv
  [table]    tables/split_summary.csv
  [model]    models/btcagtnet_v9.pth  (20.3 